# Forecast Unify & Publish

Consolidates the per-product V5 forecasting outputs into **one unified results table**
and **one AI-insights table** that the executive app reads directly — with all
information every Forecasting-panel tab needs.

---

## Upstream Source Tables (`datagroup_mdl.mdl_sales_analytics`)

| Table | Rows | Date Range | Models | Schema Highlights |
|-------|------|-----------|--------|-------------------|
| `ucc_forecast_v5` | see run log | 2023 history to 2027-10-04 | 6 | Canonical P05/P25/P50/P75/P95, weekly grain read here, rolling ~1-year forward |
| `itsg_forecast_v5` | see run log | 2023 history to 2027-10-04 | 6 | Real P05/P25/P50/P75/P95, total + market grain, rolling ~1-year forward |
| `ucc_forecast_cbe` / `itsg_forecast_cbe` | see run log | 2025-12-29 to 2027-10-04 | 6 | Source of **actuals** (`source = 'actual'`, weekly Salesforce Growth ARR) and current-best-estimate forecast |
| `ucc_forecast_monitoring_baseline` | 439 | — | 6 | Per-model holdout WAPE/RMSE, alert thresholds |
| `itsg_forecast_monitoring_baseline` | ~50 | — | 6 | Per-model WAPE/MAPE by metric_scope |
| `ucc_forecast_boq` | see run log | rolling ~1 year to 2027-10-04 | 7 | Frozen BOQ plan, P05/P25/P50/P75/P95; UCC forward weeks (current fiscal year) of `forecast_results` and the app snapshot are taken from its `forecast` column |
| `ucc_forecast_vintages` | grows each run | retained since 2026-07 | 6 | Append-only forecast history for vintage tracking |
| `ucc_forecast_v4` | 792 | 2026-08-03 to 2026-12-28 | 6 | Legacy mirror for backward compatibility |

**2026-08-30: Added Cell 6b — Hierarchical Reconciliation.** Top-down proportional
scaling ensures market-level forecasts sum exactly to the total-level forecast
within each product (UCC, ITSG). The "Total" product rollup is then recomputed
from reconciled values. Method: Athanasopoulos et al. (2009).
| `itsg_forecast_boq` | see run log | rolling ~1 year to 2027-10-04 | 6 | Rolling 1-year BOQ plan, P05/P25/P50/P75/P95, forecast only |
| `itsg_forecast_v4` | 972 | 2026-08-10 to 2027-02-08 | 6 | Legacy mirror for backward compatibility |

**Notes:**
- `ucc_forecast_v5` uses DATE for `ds`; `itsg_forecast_v5` uses TIMESTAMP — both cast to DATE here.
- **UCC quantiles**: the canonical **P05/P25/P50/P75/P95** (the legacy p10..p90-by-10s grid is no longer produced upstream; removed 2026-09-28 from the UCC notebook and every downstream consumer — PowerBI export, app snapshot, forecast_all_models, scenario grid). UCC data now runs a rolling year forward (to 2027-10-04), not just the current quarter.
- **ITSG quantiles**: real, native P05/P25/P50/P75/P95 (no longer p10/p50/p90 — converted 2026-09-28; series is noisier at ~$350K avg weekly, ~30% WAPE). MC-based for Ensemble, holdout-residual-based for individual models.
- UCC models: `Adaptive_Ensemble`, `Prophet_trend`, `MSTL_v2`, `DHR_ARIMA`, `Global_LGB_Q50_UCC`, `ETS`.
- ITSG models: `Adaptive_Ensemble`, `ETS`, `MSTL_v2`, `Global_LGB_Q50_ITSG`, `Prophet_trend`, `DHR_ARIMA`.
- Both products' BOQ/CBE tables share the same schema and consumption patterns (forecast only vs forecast+actuals).

---

## Outputs Written by This Notebook

| Table / View | Mode | Purpose |
|-------------|------|--------|
| `forecast_results` | Overwrite (replaceWhere = today) | Unified long-format: all models, products, forecast_types, both views |
| `forecast_results_latest` (view) | Recreated | Latest run_date slice for app queries |
| `forecast_insights` | Overwrite | AI-insights tab: momentum, risk, confidence per product |
| `forecast_graph_explanations` | Overwrite | Per-graph "explain this chart" text (3 products x 5 regions x 2 periods x 10 graphs) |
| `forecast_publish_log` | Append | Observability: row counts, schema version, timestamps |
| `forecast_results_v2compat` (view) | Recreated | Wide-format compat (mirrors `arr_forecast_v2` schema for app endpoints) |
| `forecast_leaderboard_v2compat` (view) | Recreated | Model accuracy leaderboard for the Accuracy tab |
| `forecast_insights_v2compat` (view) | Recreated | Insights reshaped for legacy app endpoint |
| `forecast_app_snapshot` | Overwrite | App table (federated copy too): weekly actuals + forecast, canonical quantiles, UCC forward weeks from the BOQ plan |
| `forecast_app_snapshot_incl_miradore` | Overwrite | Same table with the Miradore estimate added to ITSG and Total (`miradore_amount`, `miradore_basis` columns); the app's "Include Miradore" toggle reads it |

All tables/views GRANTed SELECT to app service principal `324a6ec7-...`.

---

## Design Decisions

1. **Actuals** pulled from the CBE tables (`ucc_forecast_cbe`, `itsg_forecast_cbe`, `source = 'actual'`; weekly Salesforce Growth ARR, UCC + ITSG).
2. **Accuracy** = per-model holdout WAPE from monitoring baselines.
3. **Three-way disjoint tagging**: `rolling` (next 13 wks), `roy` (rest of FY), `forward_1yr` (2027+).
4. **Two forecast views**: `adaptive` (default, app blends actuals in) + `boq` (pure model, stable plan target).
5. **Total rollup**: UCC + ITSG summed per (ds, market, model, forecast_type, view).
6. **Insights**: deterministic (no LLM) — momentum from trajectory slope, confidence from band spread + WAPE.

---

## How the App Reads the Output

The executive app reads from `forecast_results_latest` (the latest `run_date` slice).
Two primary consumption patterns:

### Pattern 1 — Forecast Only (BOQ / Pure Model)
```sql
SELECT * FROM forecast_results_latest
WHERE forecast_view = 'boq'
  AND forecast_type IN ('rolling', 'roy', 'forward_1yr')
```
- **Use case**: Beginning-of-Quarter plan target, stable reference that doesn't shift as actuals arrive.
- **Rows**: One per (ds, model, product, sales_market, grain_level) — forward weeks only.
- **Columns used**: `p05`, `p50` (= `most_likely`), `p95` for scenario bands.
- **No actuals blended** — pure model output locked at forecast time.

### Pattern 2 — Forecast + Actuals (Adaptive / CBE)
```sql
SELECT * FROM forecast_results_latest
WHERE forecast_view = 'adaptive'
```
- **Use case**: Overview KPI, YTD cumulative, weekly chart, monthly comparisons.
- **Rows include**:
  - `forecast_type = 'actuals'` — closed weeks with `actual` populated, model = `ACTUAL`.
  - `forecast_type = 'rolling'` — next 13 open weeks (current quarter forward).
  - `forecast_type = 'roy'` — rest of FY after rolling window.
  - `forecast_type = 'forward_1yr'` — 2027+ planning horizon.
- **KPI formula**: `SUM(actual) WHERE actuals + SUM(p50) WHERE rolling/roy` = current-quarter landing.
- Actuals update daily as weeks close; forecast portion shrinks over the quarter.

### Pattern 3 — V2 Compat Wide Format (Legacy Endpoints)
```sql
SELECT * FROM forecast_results_v2compat
```
- **Use case**: Drop-in replacement for `arr_forecast_v2` in existing dashboard SQL.
- **Columns**: `Actuals`, `Most_Likely`, `Worst_Case`, `Best_Case`, `p05`, `p25`, `p50`, `p75`, `p95` (pivoted from long). Worst_Case = real P25, Best_Case = real P75 (2026-09-28: no longer an interpolated P10/P90 average).
- One row per (ds, product, sales_market, forecast_type, run_date).

### Pattern 4 — Model Accuracy (Leaderboard Tab)
```sql
SELECT * FROM forecast_leaderboard_v2compat
```
- Per-model WAPE from monitoring baselines, by product.

### Upstream Planning Tables (Not Published Here, But App-Accessible)

| Table | Product | Pattern | Description |
|-------|---------|---------|-------------|
| `ucc_forecast_boq` | UCC | Forecast only | Rolling 1-year pure model plan, P05/P25/P50/P75/P95 |
| `ucc_forecast_cbe` | UCC | Forecast + Actuals | Actuals for closed weeks + forecast for open, `pace_vs_boq_pct`, cumulative QTD |
| `ucc_forecast_v5` | UCC | Forecast only | Rolling ~1-year forward (to 2027-10-04), 6 models, total grain (market-grain rows removed 2026-09-28), P05/P25/P50/P75/P95 |
| `itsg_forecast_boq` | ITSG | Forecast only | Rolling 1-year pure model plan, real P05/P25/P50/P75/P95 quantiles |
| `itsg_forecast_cbe` | ITSG | Forecast + Actuals | Actuals for closed weeks + forecast for open, `pace_vs_boq_pct`, cumulative QTD |
| `itsg_forecast_v5` | ITSG | Forecast only | Rolling ~1-year forward (to 2027-10-04), 6 models, total + market grain, real P05/P25/P50/P75/P95 |

The `cbe` table is the closest analog to Pattern 2 at weekly grain with QTD tracking;
`boq` is the closest to Pattern 1 with a full rolling 1-year window.

**Schedule:** task 3 of the weekly ARR forecasting job (Mondays), after the UCC and ITSG notebooks and before the Power BI export. See `notebooks/README.md`.

## Cell 1 — Config

In [ ]:
from datetime import timedelta, datetime, timezone

CATALOG = "datagroup_mdl"
SCHEMA  = "mdl_sales_analytics"

UCC_SRC       = f"{CATALOG}.{SCHEMA}.ucc_forecast_v5"
ITSG_SRC      = f"{CATALOG}.{SCHEMA}.itsg_forecast_v5"
# Actuals from CBE tables (TRUE weekly SFDC Growth ARR — replaces arr_forecast_v2
# which had inflated ITSG numbers from non-Growth revenue streams)
UCC_CBE_SRC   = f"{CATALOG}.{SCHEMA}.ucc_forecast_cbe"
ITSG_CBE_SRC  = f"{CATALOG}.{SCHEMA}.itsg_forecast_cbe"
UCC_BASELINE  = f"{CATALOG}.{SCHEMA}.ucc_forecast_monitoring_baseline"
ITSG_BASELINE = f"{CATALOG}.{SCHEMA}.itsg_forecast_monitoring_baseline"

RESULTS_TABLE  = f"{CATALOG}.{SCHEMA}.forecast_results"
RESULTS_VIEW   = f"{CATALOG}.{SCHEMA}.forecast_results_latest"
INSIGHTS_TABLE = f"{CATALOG}.{SCHEMA}.forecast_insights"
PUBLISH_LOG    = f"{CATALOG}.{SCHEMA}.forecast_publish_log"

# App-compatibility views: expose the unified data in the exact shapes the
# existing backend endpoints read (wide arr_forecast_v2 schema, leaderboard,
# insights), so repointing the app is a table-name change only — no endpoint
# SQL rewrite. forecast_results stays the single source of truth.
RESULTS_V2_VIEW  = f"{CATALOG}.{SCHEMA}.forecast_results_v2compat"
LEADERBOARD_VIEW = f"{CATALOG}.{SCHEMA}.forecast_leaderboard_v2compat"
INSIGHTS_V2_VIEW = f"{CATALOG}.{SCHEMA}.forecast_insights_v2compat"

# Per-graph "explain this chart in detail" text, precomputed per (graph, product).
# The app shows it behind an Explain icon on every chart. Precomputed baseline;
# a Genie/on-demand path can supersede this later using the same table as fallback.
GRAPH_EXPLAIN_TABLE = f"{CATALOG}.{SCHEMA}.forecast_graph_explanations"
SCENARIO_GRID_TABLE  = f"{CATALOG}.{SCHEMA}.forecast_scenario_grid"
SENSITIVITY_TABLE    = f"{CATALOG}.{SCHEMA}.forecast_sensitivity"
MODEL_SCENARIO_TABLE = f"{CATALOG}.{SCHEMA}.forecast_model_scenarios"
HIST_SCENARIO_TABLE  = f"{CATALOG}.{SCHEMA}.forecast_historical_scenarios"

APP_SERVICE_PRINCIPAL = "324a6ec7-e988-42c7-8a7f-55465f5bea37"
REC_MODEL       = "Adaptive_Ensemble"   # the model flagged recommended_for_exec
ROLLING_WEEKS   = 13
SCHEMA_VERSION  = "2.1"

# Fiscal year boundary: last Monday of the current fiscal year. Weeks beyond
# this = forward_1yr. GoTo fiscal year = calendar year.
# CURRENT_FY and FY_END_DATE are NOT hardcoded here — they are derived from
# MAX(ds) across the source forecast tables in Cell 2 (right after the
# _safe_read calls), so this notebook needs no annual manual edit. See that
# cell for the computation; CURRENT_FY / FY_END_DATE become available there.

# Canonical unified schema (long format — one row per model per week per scenario).
# 2026-09-28: legacy p10/p20/p30/p40/p60/p70/p80/p90 grid REMOVED from BASE.
# Both upstream V5 notebooks (UCC and ITSG) now compute genuine quantiles at
# P05/P25/P50/P75/P95 only, and every downstream consumer (PowerBI export,
# app snapshot, forecast_all_models, scenario grid) has been updated in this
# same pass to read P05/P25/P50/P75/P95 instead of the old P10/P50/P90 /
# P20-P80 columns. No backfill/interpolation needed any more since both
# products now publish real values at every published quantile.
BASE = ["run_date", "ds", "product", "sales_market", "grain_level", "model",
        "forecast_type", "forecast_view", "actual",
        "p05", "p25", "p50", "p75", "p95",
        "most_likely",
        "recommended_for_exec", "source_run_ts", "published_at_utc"]

COLS = BASE + ["model_wape", "model_mape"]   # accuracy joined on at the end

# ── Z-scores for normal-approximation quantile interpolation ────────────────
# Shared across cells that still need to derive a band from sigma (e.g.
# Prophet-vs-other-model interval math). ppf(1-alpha) of the standard normal
# at each quantile's distance from the median.
Z_05 = 1.6449   # P05 / P95
Z_25 = 0.6745   # P25 / P75

print(f"Publishing → {RESULTS_TABLE}, {INSIGHTS_TABLE}")

## Cell 2 — Extract the V5 forecast tables

In [ ]:
def _safe_read(name):
    try:
        df = spark.read.table(name)
        print(f"  read {name}: {df.count()} rows")
        return df
    except Exception as exc:
        print(f"  WARN could not read {name}: {exc}")
        return None

ucc  = _safe_read(UCC_SRC)
itsg = _safe_read(ITSG_SRC)
assert ucc is not None or itsg is not None, "No source forecast tables available."

# ── Current fiscal year & FY end, derived from the source data ──────────────
# Replaces the old hardcoded "current year" literal (FY_END_DATE was pinned to
# a specific calendar year and required an annual manual edit). GoTo fiscal
# year = calendar year; FY end = last Monday on/before Dec 31 of that year
# (mirrors the CURRENT_FY-from-max(ds) pattern used in the upstream per-product
# forecast notebooks). Computed here, right after the source tables are read,
# since MAX(ds) needs the raw ucc/itsg frames.
from datetime import date
from pyspark.sql import functions as F

# 2026-09-28 CRITICAL FIX: ucc_forecast_v5 (Step 14E-V5 in the UCC notebook)
# writes MULTIPLE grains into the SAME table -- weekly rows AND monthly/
# quarterly/yearly rollup rows, distinguished only by a `grain` column
# ('weekly' / 'monthly' / 'quarterly' / 'yearly'). This notebook's BASE
# schema only carries `grain_level` (total vs market), not `grain`, so
# without an explicit filter here every read of ucc_forecast_v5 silently
# pulled ALL grains at once. Any quarterly/FY sum downstream then double-
# (or more) counted: e.g. Q4 2026 summed to ~$28.5M instead of the correct
# ~$14M, because the 3 month-start rollup rows in that quarter (~$14.5M)
# were being added on top of the 13 real weekly rows (~$14.0M) rather than
# being excluded. Filter to weekly grain only -- the monthly/quarterly/
# yearly rollup rows in ucc_forecast_v5 are redundant with sums over the
# weekly rows and were never meant to be read by this notebook.
# ITSG's V5 table has no `grain` column (it only ever writes weekly rows),
# so this filter is applied defensively, only when the column exists.
if ucc is not None and "grain" in ucc.columns:
    _ucc_rows_before = ucc.count()
    ucc = ucc.where(F.col("grain") == "weekly")
    print(f"  UCC: filtered ucc_forecast_v5 to grain='weekly' only "
          f"({_ucc_rows_before} -> {ucc.count()} rows; dropped monthly/quarterly/yearly rollup rows)")
if itsg is not None and "grain" in itsg.columns:
    _itsg_rows_before = itsg.count()
    itsg = itsg.where(F.col("grain") == "weekly")
    print(f"  ITSG: filtered itsg_forecast_v5 to grain='weekly' only "
          f"({_itsg_rows_before} -> {itsg.count()} rows)")

# 2026-09-30 CRITICAL FIX: CURRENT_FY was computed from MAX(ds) across `ucc`/
# `itsg` -- but those are the FULL forward-looking forecast tables (extending
# ~1 year ahead), not the actuals. A real Databricks run confirmed the bug:
# with data extending to 2027-09-27, this produced CURRENT_FY=2027 (should
# have been 2026), which made FY_END_DATE=2027-12-27 -- a full year too late.
# That cascaded into two separate, serious downstream breaks:
#   1. Cell 10's 3-way rolling/roy/forward_1yr split uses FY_END_DATE as the
#      roy/forward_1yr boundary, so EVERY 2027 week got mistagged 'roy'
#      instead of 'forward_1yr' -- meaning the app's "current FY landing" KPI
#      (SUM of forecast_type IN ('rolling','roy')) silently summed in an
#      entire extra year of forecast on top of the real FY2026 total.
#   2. Cell 38's scenario grid groups by fiscal_quarter alone (no year), so
#      once 2027 weeks leaked into 'roy', 2026-Q1 and 2027-Q1 (etc.) got
#      summed into the SAME "Q1" bucket -- confirmed in the real run:
#      weeks_in_quarter=26 (13+13) instead of 13, and actual_to_date/
#      actual_weeks came back NULL for every row (act_q filtered on
#      YEAR(ds) = CURRENT_FY = 2027, a year with zero actuals).
# UCC's and ITSG's own notebooks never had this bug: they derive CURRENT_FY
# from MAX(ds) of their ACTUALS frame (weekly_ucc / weekly_itsg), which is
# naturally "now". This notebook has no equivalent actuals-only frame yet at
# this point (that's built in Cell 7), so -- matching this notebook's own
# stated design intent ("GoTo fiscal year = calendar year") -- CURRENT_FY is
# simply today's calendar year. This needs no annual manual edit (same
# guarantee the old approach was trying for) and cannot drift into a future
# forecast horizon the way MAX(ds) over forward-looking data can.
_max_ds_candidates = [d.agg(F.max(F.to_date("ds"))).collect()[0][0]
                       for d in (ucc, itsg) if d is not None]
_max_ds_candidates = [d for d in _max_ds_candidates if d is not None]
assert _max_ds_candidates, "Cannot determine max(ds): no ds values in source tables."
_max_ds = max(_max_ds_candidates)
CURRENT_FY = date.today().year

def _last_monday_on_or_before(d):
    return d - timedelta(days=d.weekday())  # Monday == 0

FY_END_DATE = _last_monday_on_or_before(date(CURRENT_FY, 12, 31)).strftime("%Y-%m-%d")
print(f"CURRENT_FY={CURRENT_FY} (today's calendar year), FY_END_DATE={FY_END_DATE} "
      f"(source data max(ds)={_max_ds}, for reference only -- no longer used to derive CURRENT_FY)")


## Cell 2b — Extend forecast with BOQ 2027 data

V5 forecast tables may stop before the full 2027 planning window. Pull the
forecast-only BOQ rows after each V5 table's max `ds` so Cell 4's existing
`forward_1yr` tagging has complete 2027 input data.

In [ ]:
from pyspark.sql import functions as F

UCC_BOQ_SRC  = f"{CATALOG}.{SCHEMA}.ucc_forecast_boq"
ITSG_BOQ_SRC = f"{CATALOG}.{SCHEMA}.itsg_forecast_boq"

# 2026-09-28: trimmed to the canonical P05/P25/P50/P75/P95 grid (legacy
# p10/p20/p30/p40/p60/p70/p80/p90 removed everywhere). We cannot query these
# tables' schemas ahead of time (no direct access here), so selection stays
# fully defensive: read boq.columns at runtime and only pull quantile columns
# that are actually present, filling the rest with NULL.
ALL_QUANTILE_COLS = ["p05", "p25", "p50", "p75", "p95"]

def _boq_extension_rows(boq_table, v5_df, product):
    """Map forecast-only BOQ rows beyond the V5 horizon into V5-compatible rows."""
    if v5_df is None:
        return None
    max_ds = v5_df.agg(F.max(F.to_date("ds")).alias("max_ds")).collect()[0]["max_ds"]
    if max_ds is None:
        return None
    try:
        boq = spark.table(boq_table)
    except Exception as exc:
        print(f"  WARN could not read {boq_table}: {exc}")
        return None

    # Defensive quantile-column selection: only cast/alias columns that exist
    # on this particular BOQ table; missing ones become NULL so downstream
    # unionByName(allowMissingColumns=True) stays compatible either way.
    quantile_selects = [
        (F.col(c).cast("double").alias(c) if c in boq.columns
         else F.lit(None).cast("double").alias(c))
        for c in ALL_QUANTILE_COLS
    ]

    out = (boq
        .where((F.to_date("ds") > F.lit(max_ds)) & (F.lower(F.col("source")) == F.lit("forecast")))
        .select(
            F.col("model"),
            F.to_date("ds").alias("ds"),
            F.col("forecast").cast("double").alias("forecast"),
            *quantile_selects,
            F.col("grain_level"),
            F.col("sales_market"),
            F.when(F.col("model") == F.lit(REC_MODEL), F.lit(1)).otherwise(F.lit(0)).cast("bigint").alias("recommended_for_exec"),
            F.col("run_date_utc").cast("timestamp").alias("run_timestamp_utc"),
            F.col("pipeline_version"),
        ))
    missing = [c for c in ALL_QUANTILE_COLS if c not in boq.columns]
    if missing:
        print(f"  {product} BOQ ({boq_table}) missing quantile cols (filled NULL): {missing}")
    print(f"  {product} BOQ extension rows after {max_ds}: {out.count()}")
    return out

ucc_ext = _boq_extension_rows(UCC_BOQ_SRC, ucc, "UCC")
itsg_ext = _boq_extension_rows(ITSG_BOQ_SRC, itsg, "ITSG")

if ucc_ext is not None:
    ucc = ucc.unionByName(ucc_ext, allowMissingColumns=True)
if itsg_ext is not None:
    itsg = itsg.unionByName(itsg_ext, allowMissingColumns=True)

print(f"Rows after BOQ extension — UCC: {ucc.count() if ucc is not None else 0}, ITSG: {itsg.count() if itsg is not None else 0}")


## Cell 3 — Normalize forecasts to the canonical schema

`run_date` = publish date (this run), so the daily `replaceWhere` overwrite is
idempotent; source lineage is kept in `source_run_ts`. `ds` cast to DATE to
reconcile the UCC(date)/ITSG(timestamp) mismatch.

In [ ]:
from pyspark.sql import functions as F

# ══════════════════════════════════════════════════════════════════════════════
# EXACT-VALUE PHILOSOPHY
# ══════════════════════════════════════════════════════════════════════════════
# This notebook publishes EXACT values from the upstream forecast pipelines.
#
# Source of p05 / p25 / p50 / p75 / p95:
#   - UCC: Monte Carlo simulation quantiles from ucc_forecast_v5
#   - ITSG: Holdout-residual-based quantiles from itsg_forecast_v5
#   These are the EXACT quantiles written by the per-product forecasting
#   notebooks. No reconstruction, no band-math, no sqrt(sum-of-squares).
#
# Source of most_likely:
#   - Always equals p50 (the median point estimate). Never an inferred or
#     approximate value.
#
# Source of actuals:
#   - CBE tables (ucc_forecast_cbe, itsg_forecast_cbe) for 2026+ weeks: TRUE
#     Growth ARR from SFDC, closed weekly.
#   - arr_forecast_v2 for 2022-2025 historical (UCC only; ITSG there is inflated).
#
# The app MUST consume these exact published values. It should NOT reconstruct
# Most Likely / Stretch Case / Risk Floor from approximate band math.
# ══════════════════════════════════════════════════════════════════════════════

def normalize_forecast(df, product):
    """Normalize a V5 forecast table into the canonical schema.

    All numeric values (p05, p25, p50, p75, p95, most_likely) are passed through EXACTLY
    as they appear in the source table. No transformation, no approximation.
    recommended_for_exec is preserved from the upstream pipeline.
    """
    if df is None:
        return None
    cols = set(df.columns)
    out = (df
        .withColumn("product", F.lit(product))
        .withColumn("run_date", F.current_date())
        .withColumn("published_at_utc", F.current_timestamp())
        .withColumn("ds", F.to_date(F.col("ds")))                 # unify DATE type
        .withColumn("most_likely", F.col("p50"))                   # EXACT: most_likely := p50
        .withColumn("actual", F.lit(None).cast("double"))          # forecast rows have no actual
        .withColumn("forecast_type", F.lit(None).cast("string"))  # set in Cell 4
        .withColumn("forecast_view", F.lit("adaptive"))            # default; BOQ rows added in Cell 4b
        .withColumn("source_run_ts",
                    F.col("run_timestamp_utc") if "run_timestamp_utc" in cols
                    else F.lit(None).cast("timestamp")))
    # recommended_for_exec is preserved from the source V5 table (bigint).
    # If not present in source, it defaults to NULL (filled by the loop below).
    for c in BASE:
        if c not in out.columns:
            out = out.withColumn(c, F.lit(None))
    return out.select(*BASE)

fc_parts = [p for p in (normalize_forecast(ucc, "UCC"), normalize_forecast(itsg, "ITSG")) if p is not None]
fc = fc_parts[0]
for p in fc_parts[1:]:
    fc = fc.unionByName(p, allowMissingColumns=True)

# 2026-09-28: ITSG backward-compat backfill REMOVED. Both UCC and ITSG now
# publish real (non-approximated) p05/p25/p50/p75/p95 values natively, and
# every downstream reader has been updated to use only that 5-column grid, so
# there is no longer a legacy p10/p20/.../p90 column for either product to
# backfill.

# central market-label normalization
fc = fc.withColumn(
    "sales_market",
    F.when(F.upper(F.trim(F.col("sales_market"))).isin("ALL", "TOTAL"), F.lit("Total"))
     .when(F.upper(F.trim(F.col("sales_market"))) == "UNKNOWN", F.lit("Unknown"))
     .otherwise(F.trim(F.col("sales_market"))),
)
print(f"Normalized forecast rows: {fc.count()}")

# 2026-10-06: UCC Adaptive_Ensemble weekly p50 / most_likely come from the frozen BOQ plan
# (ucc_forecast_boq.forecast) so forecast_results -- and everything built on it
# (AI insights, graph explanations, scenario grid, sensitivity, model scenarios,
# leaderboard compat views) -- matches forecast_app_snapshot. ITSG is unchanged.
_RESULTS_BOQ_PRODUCTS = {"UCC"}
# Current fiscal year only: the raw BOQ "forecast" overstates 2027+ vs p50 (see UCC notebook, CBE step).
_RESULTS_BOQ_FY_END = f"{date.today().year}-12-31"

def load_boq_weekly(product, fy_end=_RESULTS_BOQ_FY_END):
    """Single source for the BOQ-plan weekly override used by forecast_results AND forecast_app_snapshot.
    Returns a Spark DF (_boq_ds DATE, _boq_p50 DOUBLE): latest-run Adaptive_Ensemble BOQ plan, total grain, <= fy_end."""
    _tbl = f"{CATALOG}.{SCHEMA}.{product.lower()}_forecast_boq"
    return spark.sql(f"""
        SELECT CAST(ds AS DATE) AS _boq_ds, CAST(forecast AS DOUBLE) AS _boq_p50
        FROM {_tbl}
        WHERE model = 'Adaptive_Ensemble' AND source = 'forecast'
          AND grain_level = 'total' AND ds <= '{fy_end}'
          AND run_date_utc = (SELECT MAX(run_date_utc) FROM {_tbl})
    """).dropDuplicates(["_boq_ds"])
for _p in sorted(_RESULTS_BOQ_PRODUCTS):
    _boq_tbl = f"{CATALOG}.{SCHEMA}.{_p.lower()}_forecast_boq"
    try:
        _boq_w = load_boq_weekly(_p)
        _is_target = ((F.col("product") == _p) & (F.col("model") == "Adaptive_Ensemble")
                      & (F.col("sales_market") == "Total") & F.col("_boq_p50").isNotNull())
        fc = (fc.join(_boq_w, fc["ds"] == _boq_w["_boq_ds"], "left")
                .withColumn("p50", F.when(_is_target, F.col("_boq_p50")).otherwise(F.col("p50")))
                .withColumn("most_likely", F.when(_is_target, F.col("_boq_p50")).otherwise(F.col("most_likely")))
                .drop("_boq_ds", "_boq_p50"))
        print(f"  {_p}: forecast_results p50/most_likely overridden from BOQ plan ({_boq_tbl}), {_boq_w.count()} weeks")
    except Exception as exc:
        print(f"  WARN {_p} BOQ override for forecast_results skipped: {exc}")


## Cell 4 — Tag forecast_type (rolling / roy / forward_1yr)

Three-way DISJOINT tagging:
- `rolling` = next 13 weeks (current quarter)
- `roy` = rest of fiscal year (after rolling, through FY_END_DATE)
- `forward_1yr` = weeks beyond the current fiscal year (2027+ extension from Step 15C/8C)

All three are disjoint — the app sums `rolling + roy` for current-FY totals,
and uses `forward_1yr` for the next-year planning view.

In [ ]:
# 2026-09-30 FIX: anchor the "current 13-week rolling window" to TODAY, not
# to fc's own minimum date. fc's date range extends backward for backtest/
# vintage continuity (a real run showed min_ds in early January while "today"
# was Sep 29) -- anchoring to min_ds silently tagged long-past weeks as
# "rolling" while the real current quarter landed in "roy" instead. Same
# species of bug as the CURRENT_FY-from-max(ds) issue (Cell 2): deriving a
# "now" concept from data extent instead of the wall clock.
#
# Also: any week that has ALREADY CLOSED (ds <= today) is now excluded from
# all three forward-looking buckets entirely, rather than being tagged
# rolling/roy/forward_1yr using the model's own stale self-estimate for that
# week. Closed weeks' real values are covered by the separate actuals
# pipeline (build_actuals_from_cbe, Cell 6) -- letting a closed week's raw,
# unblended model prediction leak into forecast_type='rolling'/'roy' let
# partial-quarter totals understate true pace (the model's own guess for an
# already-closed week can differ meaningfully from the real actual booked
# for it, which is the entire reason BOQ/CBE overrides and period-end uplift
# exist elsewhere in this pipeline).
_today_anchor = date.today()
rolling_cutoff = _today_anchor + timedelta(weeks=ROLLING_WEEKS - 1)
fy_end = datetime.strptime(FY_END_DATE, "%Y-%m-%d").date()

# 2026-09-30 FOLLOW-UP FIX: don't cut off "closed" weeks strictly by calendar
# date. Data ingestion for real actuals lags -- a week's ds can already be
# <= today while its real actual hasn't landed in the CBE tables yet (seen
# in a real run: Q3 showed actual_weeks=12 of 13 on the exact last day of
# Q3). Cutting off by pure calendar date silently dropped that week from
# BOTH sides: too "closed" (by date) to get a model forecast row, but no
# real actual yet either -- understating partial-quarter totals by a full
# week (worse, likely the quarter-end week, the one most prone to a real
# sales-acceleration bump). Instead, cut off at whichever is EARLIER of
# (a) today, or (b) the latest date real actuals actually exist for in the
# CBE tables -- so any week without a landed actual keeps a model forecast
# row until the real value supersedes it in a later run.
def _max_cbe_actual_ds(tbl):
    try:
        row = spark.sql(f"SELECT MAX(CAST(ds AS DATE)) AS m FROM {tbl} WHERE source = 'actual'").collect()[0]
        return row["m"]
    except Exception as exc:
        print(f"  WARN could not read latest actual ds from {tbl}: {exc}")
        return None

_cbe_max_candidates = [d for d in (
    _max_cbe_actual_ds(f"{CATALOG}.{SCHEMA}.ucc_forecast_cbe"),
    _max_cbe_actual_ds(f"{CATALOG}.{SCHEMA}.itsg_forecast_cbe"),
) if d is not None]
_actuals_boundary = min(min(_cbe_max_candidates), _today_anchor) if _cbe_max_candidates else _today_anchor
if _actuals_boundary < _today_anchor:
    print(f"  Actuals lag detected: latest landed actual is {_actuals_boundary}, "
          f"{(_today_anchor - _actuals_boundary).days} day(s) behind today ({_today_anchor}) "
          f"-- weeks after {_actuals_boundary} keep a model forecast row until actuals catch up.")

# Three-way DISJOINT tagging (weeks without a landed actual yet, ds > boundary):
#   rolling     = next 13 weeks (current quarter forward)
#   roy         = rest of current fiscal year AFTER the rolling window
#   forward_1yr = all weeks BEYOND the current fiscal year end (2027+)
# `rolling_cutoff` stays anchored to TODAY (the window length is a calendar
# concept), only the lower bound uses the actuals-lag-aware boundary.
# They MUST be disjoint: the app's KPI query sums forecast_type IN
# ('rolling','roy'), so overlapping rows double-count the quarter total.
rolling     = fc.where((F.col("ds") > F.lit(_actuals_boundary)) & (F.col("ds") <= F.lit(rolling_cutoff))).withColumn("forecast_type", F.lit("rolling"))
roy         = fc.where((F.col("ds") > F.lit(rolling_cutoff)) & (F.col("ds") <= F.lit(fy_end))).withColumn("forecast_type", F.lit("roy"))
forward_1yr = fc.where(F.col("ds") > F.lit(fy_end)).withColumn("forecast_type", F.lit("forward_1yr"))
forecasts = rolling.unionByName(roy).unionByName(forward_1yr)
print(f"Forecast rows after 3-way split (ds > {_actuals_boundary}): {forecasts.count()}")
print(f"  rolling: {rolling.count()} (through {rolling_cutoff})")
print(f"  roy: {roy.count()} (through {fy_end})")
print(f"  forward_1yr: {forward_1yr.count()} (beyond FY into 2027+)")

In [ ]:
# BOQ rows = duplicate of ALL forecast rows but tagged forecast_view='boq'.
# BOQ = "Beginning of Quarter" pure model prediction — no actuals blended.
# The adaptive view (default) will blend actuals in the app layer;
# the BOQ view gives a stable plan target that doesn't change mid-quarter.
#
# Both views share the same forecast_type tagging (rolling/roy/forward_1yr)
# so the app can filter by horizon AND by view independently.

boq_forecasts = forecasts.withColumn("forecast_view", F.lit("boq"))
# Original forecasts already have forecast_view='adaptive' from normalize step
forecasts_with_views = forecasts.unionByName(boq_forecasts)
print(f"Forecast rows with both views: {forecasts_with_views.count()} "
      f"(adaptive: {forecasts.count()}, boq: {boq_forecasts.count()})")

# Replace 'forecasts' variable for downstream cells
forecasts = forecasts_with_views

## Cell 5 — Pull actuals (CBE 2026+ and SFDC 2024-2025 Growth ARR)

Current actuals (2026+) from CBE tables (TRUE Growth ARR from SFDC).
Historical actuals (2024-2025) from `cds_sfdc_opp_products_latest` using the
SAME filters as the V5 forecast pipeline (Growth purchase_type, exclude
Care/Sales Other). Uses `total_price2_converted` (USD line-item value) to
ensure valid YoY comparisons (arr_forecast_v2 included non-Growth revenue).

In [ ]:
def build_actuals_from_cbe(cbe_table, product):
    """Pull actual rows from a CBE table (source='actual').
    CBE tables have one row per (model, ds, grain_level, sales_market) for closed weeks.
    We pick model='Adaptive_Ensemble' to deduplicate (all models carry the same actual value)."""
    try:
        cbe = spark.table(cbe_table)
    except Exception as exc:
        print(f"  WARN {cbe_table} unavailable: {exc}")
        return None

    mkt = (F.when(F.upper(F.trim(F.col("sales_market"))).isin("ALL", "TOTAL"), F.lit("Total"))
            .when(F.upper(F.trim(F.col("sales_market"))) == "UNKNOWN", F.lit("Unknown"))
            .otherwise(F.trim(F.col("sales_market"))))

    out = (cbe
        .where("source = 'actual' AND model = 'Adaptive_Ensemble'")
        .select(
            F.current_date().alias("run_date"),
            F.to_date("ds").alias("ds"),
            F.lit(product).alias("product"),
            mkt.alias("sales_market"),
            F.col("grain_level"),
            F.lit("ACTUAL").alias("model"),
            F.lit("actuals").alias("forecast_type"),
            F.lit("adaptive").alias("forecast_view"),
            F.col("forecast").cast("double").alias("actual"),
            F.lit(None).cast("double").alias("p05"),
            F.lit(None).cast("double").alias("p25"),
            F.lit(None).cast("double").alias("p50"),
            F.lit(None).cast("double").alias("p75"),
            F.lit(None).cast("double").alias("p95"),
            F.col("forecast").cast("double").alias("most_likely"),
            F.lit(0).cast("bigint").alias("recommended_for_exec"),
            F.lit(None).cast("timestamp").alias("source_run_ts"),
            F.current_timestamp().alias("published_at_utc"),
        ).select(*BASE))
    n = out.count()
    print(f"  {product} actuals from {cbe_table}: {n} rows")
    return out

ucc_actuals  = build_actuals_from_cbe(UCC_CBE_SRC, "UCC")
itsg_actuals = build_actuals_from_cbe(ITSG_CBE_SRC, "ITSG")

# ── Historical actuals from SFDC (Growth-only, pre-CBE window) ───────────────
# The CBE tables start ~2025-12-29. For the Multi-Year seasonality overlay and
# correct YoY comparisons, we need 2024-2025 Growth ARR from the SAME source
# as the V5 forecast (SFDC opp products). This replaces arr_forecast_v2 which
# included non-Growth revenue (Miradore, Self-serve) and inflated the baseline.

def build_historical_actuals_from_sfdc(cbe_earliest_ds):
    """Pull 2024-2025 Growth ARR actuals from SFDC at weekly grain.
    Uses total_price2_converted (USD line-item value) with the same filters
    as the V5 forecast pipeline: Growth purchase_type, exclude Care/Sales Other.
    Returns total grain rows (market rollup) for YoY overlay."""
    SFDC_TABLE = "datagroup.datalake_transform.cds_sfdc_opp_products_latest"
    try:
        sfdc = spark.table(SFDC_TABLE)
    except Exception as exc:
        print(f"  WARN {SFDC_TABLE} unavailable: {exc}")
        return None

    # Weekly Growth ARR per product_group (total grain for YoY overlay)
    out = spark.sql(f"""
        SELECT
            CURRENT_DATE() as run_date,
            CAST(DATE_TRUNC('week', close_date) AS DATE) as ds,
            CASE product_group WHEN 'UCC' THEN 'UCC' WHEN 'ITSG' THEN 'ITSG' END as product,
            'Total' as sales_market,
            'total' as grain_level,
            'ACTUAL' as model,
            'actuals' as forecast_type,
            'adaptive' as forecast_view,
            SUM(total_price2_converted) as actual,
            CAST(NULL AS DOUBLE) as p05,
            CAST(NULL AS DOUBLE) as p25,
            CAST(NULL AS DOUBLE) as p50,
            CAST(NULL AS DOUBLE) as p75,
            CAST(NULL AS DOUBLE) as p95,
            SUM(total_price2_converted) as most_likely,
            CAST(0 AS BIGINT) as recommended_for_exec,
            CAST(NULL AS TIMESTAMP) as source_run_ts,
            CURRENT_TIMESTAMP() as published_at_utc
        FROM {SFDC_TABLE}
        WHERE is_won = TRUE
          AND is_closed = TRUE
          AND UPPER(COALESCE(purchase_type_rollup, '')) = 'GROWTH'
          AND sales_channel NOT IN ('Care', 'Sales Other')
          AND product_group IN ('UCC', 'ITSG')
          AND close_date >= '2024-01-01'
          AND close_date < '{cbe_earliest_ds}'
        GROUP BY DATE_TRUNC('week', close_date), product_group
        HAVING SUM(total_price2_converted) IS NOT NULL
    """).select(*BASE)
    n = out.count()
    yr_totals = out.groupBy("product").agg(F.round(F.sum("actual") / 1e6, 1).alias("M"))
    print(f"  Historical actuals from SFDC Growth ARR (pre-CBE): {n} rows")
    yr_totals.show(truncate=False)
    return out

# Determine the earliest date in CBE actuals to avoid overlap
cbe_min_ds = "2025-12-29"  # CBE window start (aligned Monday before Jan 1 2026)
if ucc_actuals is not None:
    _cbe_min = ucc_actuals.agg(F.min("ds")).collect()[0][0]
    if _cbe_min:
        cbe_min_ds = str(_cbe_min)

hist_actuals = build_historical_actuals_from_sfdc(cbe_min_ds)

# ── Combine all actuals ──────────────────────────────────────────────────────
actual_parts = [a for a in (ucc_actuals, itsg_actuals, hist_actuals) if a is not None]
if actual_parts:
    actuals = actual_parts[0]
    for a in actual_parts[1:]:
        actuals = actuals.unionByName(a)
else:
    actuals = None

combined = forecasts if actuals is None else forecasts.unionByName(actuals)
print(f"Rows incl. actuals: {combined.count()}")

In [ ]:
from pyspark.sql import functions as F

def _table_available(name):
    try:
        if not spark.catalog.tableExists(name):
            print(f"  ⚠️  table {name} not found yet — skipping this diagnostic, expected on first run")
            return False
        return True
    except Exception as e:
        print(f"  ⚠️  could not check/read table {name} ({e!r}) — skipping this diagnostic, expected on first run")
        return False

# 2026-09-28: rewritten for the P05/P25/P50/P75/P95 convention. The old
# legacy p10/p20/.../p90 "graduated fan" grid has been removed everywhere
# (forecast_results, v2compat views, forecast_pure_predictions/app_snapshot,
# and the UCC/ITSG app_forecast_export write into arr_forecast_app_latest).
# Both UCC and ITSG now publish real, native P05/P25/P75/P95 -- these checks
# confirm that 5-column grid actually flows end-to-end to the app tables,
# rather than checking the old 9-column by-10s grid.

print("=" * 80)
print("CHECK 1: forecast_results_v2compat — P05/P25/P75/P95 for UCC forecast rows")
print("=" * 80)

has_band = False
_v2compat_name = "datagroup_mdl.mdl_sales_analytics.forecast_results_v2compat"
if _table_available(_v2compat_name):
    try:
        compat = spark.table(_v2compat_name) \
            .where(
                (F.col("product") == "UCC") &
                (F.col("sales_market") == "Total") &
                (F.col("forecast_type").isin("rolling", "roy"))
            ) \
            .select("ds", "Most_Likely", "Worst_Case", "Best_Case",
                    "p05", "p25", "p75", "p95") \
            .orderBy(F.desc("ds")).limit(5).toPandas()

        p05_nulls = compat['p05'].isna().sum()
        print(f"  Sample rows: {len(compat)}")
        print(f"  p05 NULLs: {p05_nulls} / {len(compat)}  →  {'❌ MISSING' if p05_nulls == len(compat) else '✅ POPULATED'}")
        print(compat[['ds', 'Most_Likely', 'p05', 'p25', 'p75', 'p95']].to_string(index=False))
    except Exception as e:
        print(f"  ⚠️  read of {_v2compat_name} failed ({e!r}) — skipping this diagnostic, expected on first run")

print("\n" + "=" * 80)
print("CHECK 2: forecast_results_v2compat — P05/P25/P75/P95 for ITSG forecast rows")
print("=" * 80)

itsg_p05 = None
itsg_compat = None
if _table_available(_v2compat_name):
    try:
        itsg_compat = spark.table(_v2compat_name) \
            .where(
                (F.col("product") == "ITSG") &
                (F.col("sales_market") == "Total") &
                (F.col("forecast_type").isin("rolling", "roy"))
            ) \
            .select("ds", "Most_Likely", "p05", "p25", "p75", "p95") \
            .orderBy(F.desc("ds")).limit(3).toPandas()

        itsg_p05 = itsg_compat['p05'].isna().sum()
        print(f"  p05 NULLs: {itsg_p05} / {len(itsg_compat)}  →  {'❌ MISSING' if itsg_p05 == len(itsg_compat) else '✅ POPULATED (ITSG publishes real P05/P25/P75/P95 natively)'}")
    except Exception as e:
        print(f"  ⚠️  read of {_v2compat_name} failed ({e!r}) — skipping this diagnostic, expected on first run")

print("\n" + "=" * 80)
print("CHECK 3: arr_forecast_app_latest — schema + UCC P05/P25/P75/P95")
print("=" * 80)

missing = None
ucc_app = None
ucc_p05 = None
_app_tbl_name = "datagroup_mdl.mdl_sales_analytics.arr_forecast_app_latest"
if _table_available(_app_tbl_name):
    try:
        app_tbl = spark.table(_app_tbl_name)
        app_cols = {f.name for f in app_tbl.schema.fields}
        band_cols = ['p05', 'p25', 'p75', 'p95']
        missing = [c for c in band_cols if c not in app_cols]
        print(f"  Table has p05/p25/p75/p95 columns: {'✅ YES' if not missing else '❌ MISSING: ' + str(missing)}")
        legacy_cols = [c for c in ['p10', 'p20', 'p30', 'p40', 'p60', 'p70', 'p80', 'p90'] if c in app_cols]
        if legacy_cols:
            print(f"  ⚠️  legacy columns still present on this table (not removed by this notebook): {legacy_cols}")

        ucc_app = app_tbl.where(
            (F.col("product_group") == "UCC") &
            (F.col("model") == "Adaptive_Ensemble") &
            (F.col("grain_level") == "total")
        ).select("ds", "forecast", "p05", "p25", "p50", "p75", "p95") \
         .orderBy(F.desc("ds")).limit(3).toPandas()

        ucc_p05 = ucc_app['p05'].isna().sum() if 'p05' in ucc_app.columns else len(ucc_app)
        print(f"  UCC Ensemble p05: {'⚠️ NULL (populates on next pipeline run with updated UCC notebook)' if ucc_p05 == len(ucc_app) else '✅ POPULATED'}")
        print(ucc_app.to_string(index=False))
    except Exception as e:
        print(f"  ⚠️  read of {_app_tbl_name} failed ({e!r}) — skipping this diagnostic, expected on first run")

print("\n" + "=" * 80)
print("CHECK 4: Simulate backend /weekly SQL — does P05/P25/P75/P95 flow to the app?")
print("=" * 80)

sim = None
if _table_available(_v2compat_name):
    try:
        sim = spark.sql("""
            SELECT
                CAST(ds AS STRING) AS ds,
                COALESCE(CAST(Most_Likely AS DOUBLE), 0) AS value,
                COALESCE(CAST(Worst_Case AS DOUBLE), 0) AS p25,
                COALESCE(CAST(Best_Case AS DOUBLE), 0) AS p75,
                CAST(p05 AS DOUBLE) AS p05,
                CAST(p95 AS DOUBLE) AS p95
            FROM `datagroup_mdl`.`mdl_sales_analytics`.`forecast_results_v2compat`
            WHERE run_date = (SELECT MAX(run_date) FROM `datagroup_mdl`.`mdl_sales_analytics`.`forecast_results_v2compat`)
              AND forecast_type IN ('rolling', 'roy')
              AND product = 'UCC'
              AND sales_market = 'Total'
            ORDER BY ds DESC
            LIMIT 3
        """).toPandas()

        has_band = sim['p05'].notna().any()
        print(f"  Backend SQL returns p05/p95: {'✅ YES — confidence band will render' if has_band else '❌ NO — fallback to point estimate only'}")
        print(sim[['ds', 'value', 'p05', 'p25', 'p75', 'p95']].to_string(index=False))
    except Exception as e:
        print(f"  ⚠️  simulated backend query failed ({e!r}) — skipping this diagnostic, expected on first run")

print("\n" + "=" * 80)
print("CHECK 5: Total product (UCC+ITSG) — confidence band behavior")
print("=" * 80)

total = None
tot_p05 = None
if _table_available(_v2compat_name):
    try:
        total = spark.table(_v2compat_name) \
            .where(
                (F.col("product") == "Total") &
                (F.col("sales_market") == "Total") &
                (F.col("forecast_type").isin("rolling", "roy"))
            ) \
            .select("ds", "p05", "p95") \
            .orderBy(F.desc("ds")).limit(3).toPandas()

        tot_p05 = total['p05'].isna().sum()
        print(f"  Total product p05: {'⚠️ NULL — Total is UCC+ITSG sum; a component null propagates' if tot_p05 == len(total) else '✅ POPULATED'}")
        print(f"  Frontend behavior: {'Point estimate only (fallback)' if tot_p05 == len(total) else 'P05-P95 confidence band'}")
    except Exception as e:
        print(f"  ⚠️  read of {_v2compat_name} failed ({e!r}) — skipping this diagnostic, expected on first run")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
if total is not None and tot_p05 is not None:
    print(f"  UCC band:   {'✅ Real P05-P95 (native empirical quantiles)' if has_band else '❌ Missing'}")
    print(f"  ITSG band:  {'✅ Real P05-P95 (native, no longer approximated)' if itsg_p05 is not None and itsg_p05 == 0 else '⚠️ Check itsg_p05 above'}")
    print(f"  Total band: {'Missing' if tot_p05 == len(total) else '✅ P05-P95 (UCC+ITSG summed)'}")
    print(f"  App table ready: {'✅' if missing is not None and not missing else '❌'}  (p05/p25/p75/p95 columns exist in arr_forecast_app_latest)")
    print(f"  UCC app export:  {'Will populate on next Tuesday pipeline run' if (ucc_app is not None and ucc_p05 == len(ucc_app)) else '✅ Already populated' if ucc_app is not None else 'N/A — table not available yet'}")
else:
    print("  ⚠️  One or more source tables were not available yet — this is expected on the very first run "
          "of the pipeline (before this notebook's own Cell 8 write has ever populated these tables). "
          "Diagnostics will run normally on subsequent runs.")


## Cell 6 — Total rollup (UCC + ITSG)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TOTAL ROLLUP — EXACT SUM, NOT VARIANCE RECONSTRUCTION
# ══════════════════════════════════════════════════════════════════════════════
# The "Total" product = UCC + ITSG summed per (ds, market, model, forecast_type, view).
#
# Why simple SUM is correct and preferred:
#   actual:     SUM of actuals = total actual (exact).
#   p50:        SUM of per-product p50 = total point estimate (exact for expectations).
#   most_likely: same as p50 (exact).
#   p05 / p95:  SUM of per-product quantiles. This is conservative for p05 and
#               aggressive for p95 (exact only under perfect correlation). But it is
#               the PUBLISHED truth — the app reads these values directly. No sqrt,
#               no variance math, no band reconstruction.
#
# Guard: strip any pre-existing "Total" rows from upstream sources before
# computing the rollup, preventing accidental double-counting.
# ══════════════════════════════════════════════════════════════════════════════

# Safety guard: remove any "Total" product rows that leaked from upstream sources
# (e.g. arr_forecast_v2 had a 'Total' product that includes non-Growth revenue).
per_product = combined.where(F.upper(F.col("product")) != "TOTAL")

total = (per_product
    .groupBy("run_date", "ds", "sales_market", "grain_level", "model", "forecast_type", "forecast_view")
    .agg(F.sum("actual").alias("actual"),
         F.sum("p05").alias("p05"),
         F.sum("p25").alias("p25"),
         F.sum("p50").alias("p50"),
         F.sum("p75").alias("p75"),
         F.sum("p95").alias("p95"),
         F.sum("most_likely").alias("most_likely"),
         F.max("recommended_for_exec").alias("recommended_for_exec"))
    .withColumn("product", F.lit("Total"))
    .withColumn("source_run_ts", F.lit(None).cast("timestamp"))
    .withColumn("published_at_utc", F.current_timestamp())
    .select(*BASE))

# Final results_base = per-product rows (UCC, ITSG) + computed Total
results_base = per_product.unionByName(total)
print(f"Rows incl. Total rollup: {results_base.count()}")
print(f"  Per-product rows: {per_product.count()}, Total rollup rows: {total.count()}")

## Cell 6b — Hierarchical Reconciliation (top-down proportional)

**2026-09-30 UPDATE — market-level forecast rows are suppressed, not published.**
As of 2026-08-31 (market-grain forecast WAPE measured at 50-62%, too unreliable
to publish), `mkt_reconciled` is computed below and then immediately dropped via
`.where(F.lit(False))` — kept only as an empty, schema-matching placeholder so
`pp_reconciled = tot_rows.unionByName(mkt_reconciled, ...)` still type-checks.
**No market-level forecast rows are written anywhere downstream of this cell.**
Market-level ACTUALS still flow through elsewhere in the pipeline; only market-level
FORECASTS are suppressed here.

The scale-factor computation and diagnostic below (mean/median/max shift) still run
for observability — they show how large the reconciliation WOULD be if the
market-level forecasts were ever re-enabled — but scaled rows are never applied
or published in the current pipeline.

Original design (for when/if market-level forecasts are reinstated): within each
product (UCC, ITSG), scale market-level forecasts to sum **exactly** to the
total-level forecast, using **Method:** Athanasopoulos et al. (2009) top-down
proportional scaling: `scale_factor[q] = total_q / SUM(market_q)` per quantile column.

In [ ]:
# CELL 6b: HIERARCHICAL RECONCILIATION (TOP-DOWN PROPORTIONAL)
# =============================================================================
# Within each product (UCC, ITSG), ensures market-level forecasts sum exactly
# to the total-level forecast for each (ds, model, forecast_type, forecast_view).
#
# The total-level forecast is the anchor — trained on aggregate data with lower
# noise, it is the more reliable estimate. Market rows are scaled proportionally
# so they preserve relative shares while summing to the anchor.
#
# After per-product reconciliation, the "Total" product rollup (UCC + ITSG) is
# recomputed from reconciled values.
#
# Only applies to forecast rows (model != 'ACTUAL'); actuals pass through unchanged.
# =============================================================================

from pyspark.sql import Window

# ── Separate actuals from forecasts ──────────────────────────────────────────
actual_rows = results_base.where(F.col("model") == "ACTUAL")
fc_rows     = results_base.where(F.col("model") != "ACTUAL")

# Per-product forecast rows only (UCC, ITSG) — Total is recomputed after
pp_fc  = fc_rows.where(F.upper(F.col("product")) != "TOTAL")

# ── Build scale factors ──────────────────────────────────────────────────────
RECON_KEY = ["product", "ds", "model", "forecast_type", "forecast_view"]

# Anchor = the total-grain row for each key
anchor = (pp_fc
    .where(F.lower(F.col("grain_level")) == "total")
    .select(*RECON_KEY,
            F.col("p05").alias("anc_p05"),
            F.col("p50").alias("anc_p50"),
            F.col("p95").alias("anc_p95")))

# Sum of market-grain rows per key
mkt_sum = (pp_fc
    .where(F.lower(F.col("grain_level")) == "market")
    .groupBy(*RECON_KEY)
    .agg(F.sum("p05").alias("sum_p05"),
         F.sum("p50").alias("sum_p50"),
         F.sum("p95").alias("sum_p95")))

# Scale factor = anchor / sum_of_markets (1.0 if sum is zero)
recon = (anchor.join(mkt_sum, RECON_KEY, "inner")
    .withColumn("sf_p05", F.when(F.col("sum_p05") != 0, F.col("anc_p05") / F.col("sum_p05")).otherwise(1.0))
    .withColumn("sf_p50", F.when(F.col("sum_p50") != 0, F.col("anc_p50") / F.col("sum_p50")).otherwise(1.0))
    .withColumn("sf_p95", F.when(F.col("sum_p95") != 0, F.col("anc_p95") / F.col("sum_p95")).otherwise(1.0))
    .select(*RECON_KEY, "sf_p05", "sf_p50", "sf_p95"))

# ── Apply to market rows ─────────────────────────────────────────────────────
mkt_rows = pp_fc.where(F.lower(F.col("grain_level")) == "market")
tot_rows = pp_fc.where(F.lower(F.col("grain_level")) == "total")  # unchanged

# 2026-08-31: Market-grain forecast WAPE ~50-62%, deemed unreliable for
# publication (see validation 2026-08-31). Market-grain rows report actuals
# only; forecasts remain at Total/product grain only.
# Keep mkt_reconciled defined (empty DF with matching schema) because
# pp_reconciled = tot_rows.unionByName(mkt_reconciled, ...) depends on it.
mkt_reconciled = (mkt_rows
    .join(recon, RECON_KEY, "left")
    .withColumn("p05",        F.round(F.col("p05") * F.coalesce(F.col("sf_p05"), F.lit(1.0)), 2))
    .withColumn("p50",        F.round(F.col("p50") * F.coalesce(F.col("sf_p50"), F.lit(1.0)), 2))
    .withColumn("p95",        F.round(F.col("p95") * F.coalesce(F.col("sf_p95"), F.lit(1.0)), 2))
    .withColumn("most_likely", F.col("p50"))
    .drop("sf_p05", "sf_p50", "sf_p95")
    .where(F.lit(False)))

# ── Diagnostic: measure adjustment magnitude ─────────────────────────────────
_diag = (mkt_rows.join(recon, RECON_KEY, "left")
    .where(F.col("sf_p50").isNotNull())
    .select(F.abs(F.col("sf_p50") - 1.0).alias("shift_pct")))
_stats = _diag.agg(
    F.count("*").alias("n"),
    F.mean("shift_pct").alias("mean"),
    F.expr("percentile_approx(shift_pct, 0.5)").alias("median"),
    F.max("shift_pct").alias("max")).collect()[0]

# ── Reassemble per-product (total-grain unchanged + reconciled market-grain)
pp_reconciled = tot_rows.unionByName(mkt_reconciled, allowMissingColumns=True)

# ── Recompute "Total" product from reconciled UCC + ITSG ─────────────────────
total_recomp = (pp_reconciled
    .groupBy("run_date", "ds", "sales_market", "grain_level", "model",
             "forecast_type", "forecast_view")
    .agg(F.sum("actual").alias("actual"),
         F.sum("p05").alias("p05"),
         F.sum("p25").alias("p25"),
         F.sum("p50").alias("p50"),
         F.sum("p75").alias("p75"),
         F.sum("p95").alias("p95"),
         F.sum("most_likely").alias("most_likely"),
         F.max("recommended_for_exec").alias("recommended_for_exec"))
    .withColumn("product", F.lit("Total"))
    .withColumn("source_run_ts", F.lit(None).cast("timestamp"))
    .withColumn("published_at_utc", F.current_timestamp()))

# ── Final results_base = reconciled per-product + recomputed Total + actuals
results_base = (pp_reconciled
    .unionByName(total_recomp, allowMissingColumns=True)
    .unionByName(actual_rows, allowMissingColumns=True)
    .select(*BASE))

print(f"Hierarchical reconciliation complete.")
print(f"  Market adjustments: mean={_stats['mean']*100:.2f}%, "
      f"median={_stats['median']*100:.2f}%, max={_stats['max']*100:.2f}% "
      f"({_stats['n']} market rows)")
print(f"  Rows after reconciliation: {results_base.count()}")

## Cell 7 — Attach per-model accuracy (WAPE / MAPE) from the baselines

In [ ]:
def _latest(table, ts_col):
    t = spark.table(table)
    mx = t.agg(F.max(ts_col)).collect()[0][0]
    return t.where(F.col(ts_col) == F.lit(mx))

acc_parts = []
try:
    u = _latest(UCC_BASELINE, "run_date_utc").select(
        F.lit("UCC").alias("product"), F.col("model"),
        F.col("holdout_wape_baseline").alias("model_wape"),
        F.lit(None).cast("double").alias("model_mape"))
    acc_parts.append(u)
except Exception as exc:
    print(f"  WARN UCC baseline skipped: {exc}")
try:
    i = _latest(ITSG_BASELINE, "run_date_utc")
    if "metric_scope" in i.columns:
        tot = i.where(F.lower(F.col("metric_scope")).isin("total", "all") | F.col("metric_scope").isNull())
        i = tot if tot.count() > 0 else i
    i = i.select(F.lit("ITSG").alias("product"), F.col("model"),
                 F.coalesce(F.col("holdout_wape_baseline"), F.col("wape")).alias("model_wape"),
                 F.col("mape").alias("model_mape"))
    acc_parts.append(i)
except Exception as exc:
    print(f"  WARN ITSG baseline skipped: {exc}")

if acc_parts:
    acc = acc_parts[0]
    for a in acc_parts[1:]:
        acc = acc.unionByName(a)
    acc = acc.dropDuplicates(["product", "model"])
    results = results_base.join(acc, ["product", "model"], "left")
else:
    results = (results_base
               .withColumn("model_wape", F.lit(None).cast("double"))
               .withColumn("model_mape", F.lit(None).cast("double")))
results = results.select(*COLS)

## Cell 8 — Write `forecast_results` (+ latest view)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WRITE forecast_results + CREATE DEDUPED VIEW
# ══════════════════════════════════════════════════════════════════════════════
# Deduplication guarantee: exactly ONE row per natural key. If the upstream union
# produced any duplicates (e.g. overlapping date windows), the dedup removes them
# BEFORE writing, so the table is the canonical source of truth.

from pyspark.sql import Window

DEDUP_KEY = ["ds", "product", "sales_market", "grain_level", "model", "forecast_type", "forecast_view"]

# Deduplicate: keep the row with the latest published_at_utc per natural key
w = Window.partitionBy(*DEDUP_KEY).orderBy(F.col("published_at_utc").desc())
results_deduped = (results
    .withColumn("_rn", F.row_number().over(w))
    .where("_rn = 1")
    .drop("_rn"))

(results_deduped.write.format("delta").mode("overwrite")
   .option("replaceWhere", "run_date = current_date()")
   .option("mergeSchema", "true")
   .partitionBy("product")
   .saveAsTable(RESULTS_TABLE))

# The view selects the latest run_date snapshot with a secondary dedup guard:
# ROW_NUMBER ensures exactly one row per key even if table has residual dupes.
spark.sql(f"""
    CREATE OR REPLACE VIEW {RESULTS_VIEW} AS
    WITH latest AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY ds, product, sales_market, grain_level, model, forecast_type, forecast_view
                ORDER BY published_at_utc DESC
            ) AS _rn
        FROM {RESULTS_TABLE}
        WHERE run_date = (SELECT MAX(run_date) FROM {RESULTS_TABLE})
    )
    SELECT * EXCEPT (_rn) FROM latest WHERE _rn = 1
""")
print(f"Wrote {results_deduped.count()} rows to {RESULTS_TABLE} (vintage = today, deduped).")

## Cell 9 — Build & write `forecast_insights`

Deterministic payloads (no LLM/secret dependency) in the shape the AI Insights
tab reads. Momentum from the forecast trajectory; confidence from band spread
and the model's WAPE.

In [ ]:
import json

res = spark.table(RESULTS_VIEW).toPandas()
latest_run = res["run_date"].max()

# Quarter-aware setup so the insight headline matches the Overview quarter number
# (previously computed over 'roy' only, which under-counted after the disjoint fix).
import pandas as pd
res["ds"] = pd.to_datetime(res["ds"], errors="coerce")
_run_dt = pd.to_datetime(latest_run)
_Q = (_run_dt.month - 1) // 3 + 1
_Q_MONTHS = [(_Q - 1) * 3 + 1, (_Q - 1) * 3 + 2, (_Q - 1) * 3 + 3]
_Q_YEAR = int(_run_dt.year)
_Q_LABEL = f"Q{_Q} {_Q_YEAR}"

def _fmt_m(v):
    return f"${v/1e6:.1f}M" if v else "$0.0M"

def _stats(df, region="Total", period="quarter"):
    """Period landing = realized-so-far + forecast for the open weeks (matches the
    Overview KPI), plus YoY vs the same period last year and the implied weekly
    run-rate. region: 'Total' or a sales_market; period: 'quarter' (current) or
    'year' (full current year). Guarded so a bad slice can't fail the publish."""
    try:
        if region == "Total":
            tot = df[(df["grain_level"].astype(str).str.lower() == "total")
                     & (df["forecast_view"] == "adaptive")].copy()
        else:
            tot = df[(df["grain_level"].astype(str).str.lower() == "market")
                     & (df["sales_market"].astype(str).str.upper() == str(region).upper())
                     & (df["forecast_view"] == "adaptive")].copy()
        if period == "year":
            in_q = tot["ds"].dt.year.eq(_Q_YEAR)
            period_label = f"FY {_Q_YEAR}"
            prior_scope = tot["ds"].dt.year.eq(_Q_YEAR - 1)
        else:
            in_q = tot["ds"].dt.year.eq(_Q_YEAR) & tot["ds"].dt.month.isin(_Q_MONTHS)
            period_label = _Q_LABEL
            prior_scope = tot["ds"].dt.year.eq(_Q_YEAR - 1) & tot["ds"].dt.month.isin(_Q_MONTHS)
        act = tot[(tot["model"] == "ACTUAL") & in_q]
        # Splice: forecast weeks strictly AFTER the last closed actual week — no
        # double-count, no dropped in-flight week (matches the backend KPI rule).
        last_act = act["ds"].max() if not act.empty else pd.Timestamp("1900-01-01")
        fwd = tot[(tot["model"] == REC_MODEL)
                  & tot["forecast_type"].isin(["rolling", "roy"]) & in_q
                  & (tot["ds"] > last_act)].sort_values("ds")
        if fwd.empty and act.empty:
            return None
        fwd_ml = float(fwd["p50"].fillna(0).sum())
        qtd    = float(act["actual"].fillna(0).sum())
        most_likely = fwd_ml + qtd
        # Variance-aggregate the quarter band (combine weekly half-widths in
        # quadrature) instead of summing weekly quantiles — the latter overstates
        # the spread ~sqrt(weeks)x since weekly noise cancels. Upper/lower kept
        # separate for asymmetry; floored at 0 (ARR can't be negative).
        _p50 = fwd["p50"].fillna(0)
        _up = (fwd["p95"].fillna(fwd["p50"]) - _p50).clip(lower=0)
        _lo = (_p50 - fwd["p05"].fillna(fwd["p50"])).clip(lower=0)
        q_up = float((_up ** 2).sum()) ** 0.5
        q_lo = float((_lo ** 2).sum()) ** 0.5
        worst = max(0.0, most_likely - q_lo)
        best  = most_likely + q_up
        spread_pct = ((best - worst) / most_likely * 100) if most_likely else 0.0
        prior = float(tot[(tot["model"] == "ACTUAL") & prior_scope]["actual"].fillna(0).sum())
        yoy = ((most_likely - prior) / prior * 100) if prior else None
        n_fwd = int(len(fwd))
        run_rate = (fwd_ml / n_fwd) if n_fwd else None
        p = fwd["p50"].dropna().tolist()
        momentum = "STABLE"
        if len(p) >= 3 and p[0]:
            chg = (p[-1] - p[0]) / p[0] * 100
            cv = (fwd["p50"].std() / fwd["p50"].mean() * 100) if fwd["p50"].mean() else 0
            if cv > 25: momentum = "VOLATILE"
            elif chg > 5: momentum = "ACCELERATING"
            elif chg < -5: momentum = "DECELERATING"
        wape = float(fwd["model_wape"].dropna().mean()) if fwd["model_wape"].notna().any() else None
        conf = 100.0 - (spread_pct * 0.4) - (wape * 1.3 if wape is not None else 0.0)
        conf = max(15.0, min(98.0, conf))
        return dict(most_likely=most_likely, worst=worst, best=best, qtd=qtd, prior=prior,
                    upside=max(0.0, best - most_likely), downside=max(0.0, most_likely - worst),
                    spread_pct=spread_pct, momentum=momentum, wape=wape, yoy=yoy,
                    run_rate=run_rate, n_fwd=n_fwd, confidence=round(conf, 0),
                    period_label=period_label, region=region)
    except Exception as _e:
        print(f"  WARN insight stats failed: {_e}")
        return None

def _render(s, label, product_stats=None):
    """Render executive-quality insight payload.
    product_stats: dict of per-product stats (UCC/ITSG) for the aggregate view."""
    _pl = s.get("period_label", _Q_LABEL)
    risk = "LOW RISK" if s["confidence"] >= 80 else "MODERATE RISK" if s["confidence"] >= 60 else "HIGH RISK"
    yoy = s.get("yoy")
    yoy_txt = (f" — {'+' if yoy >= 0 else ''}{yoy:.0f}% YoY vs {_fmt_m(s['prior'])} last year"
               if yoy is not None else "")
    n_weeks = s.get("n_fwd", 0)
    rr_txt = (f"{_fmt_m(s['run_rate'])}/wk across {n_weeks} open weeks"
              if s.get("run_rate") and n_weeks else "n/a")

    # Key Drivers — include product split for aggregate view
    drivers = [
        f"{label} {_pl} landing {_fmt_m(s['most_likely'])} = {_fmt_m(s['qtd'])} realized "
        f"+ {_fmt_m(s['most_likely'] - s['qtd'])} forecast.",
        f"Implied pace: {rr_txt}.",
    ]
    if yoy is not None:
        drivers.append(f"Tracking {'+' if yoy >= 0 else ''}{yoy:.0f}% YoY vs {_fmt_m(s['prior'])} last year ({_pl.replace(str(_Q_YEAR), str(_Q_YEAR - 1))}).")
    if product_stats:
        parts = [f"{k}: {_fmt_m(v['most_likely'])}" for k, v in sorted(product_stats.items())]
        drivers.append(f"Product mix: {', '.join(parts)}.")

    # Downside Risks — quantified and specific
    spread_pct = s.get("spread_pct", 0)
    risk_items = [
        f"Downside case {_fmt_m(s['worst'])} — {_fmt_m(s['downside'])} ({s['downside']/(s['most_likely'])*100:.0f}%) below the central path." if s['most_likely'] else f"Downside case {_fmt_m(s['worst'])}.",
    ]
    if s["momentum"] == "DECELERATING":
        risk_items.append("Momentum decelerating — recent weeks trending below the quarterly run-rate.")
    elif s["momentum"] == "VOLATILE":
        risk_items.append(f"Week-to-week coefficient of variation elevated; band spread is {spread_pct:.0f}% of the central path.")
    else:
        risk_items.append("Primary risk is execution slip against the paced weekly target.")
    if product_stats:
        weakest = min(product_stats.items(), key=lambda x: x[1].get("confidence", 100))
        if weakest[1].get("momentum") in ("VOLATILE", "DECELERATING"):
            risk_items.append(f"{weakest[0]} showing {weakest[1]['momentum'].lower()} momentum — monitor closely.")

    # Upside Opportunities — specific triggers
    opp_items = [
        f"Stretch case {_fmt_m(s['best'])} — {_fmt_m(s['upside'])} ({s['upside']/(s['most_likely'])*100:.0f}%) above the central path." if s['most_likely'] else f"Stretch case {_fmt_m(s['best'])}.",
    ]
    if product_stats:
        strongest = max(product_stats.items(), key=lambda x: x[1].get("confidence", 0))
        if strongest[1].get("momentum") == "ACCELERATING":
            opp_items.append(f"{strongest[0]} accelerating — could outperform if trend persists.")
    if yoy is not None and yoy > 0:
        opp_items.append(f"Already tracking +{yoy:.0f}% above prior-year pace.")

    # Executive Actions — specific and quantified
    actions = []
    if n_weeks > 0 and s.get("run_rate"):
        actions.append(f"Maintain {_fmt_m(s['run_rate'])}/wk run-rate over the remaining {n_weeks} weeks to land {_fmt_m(s['most_likely'])} for {_pl}.")
    else:
        actions.append(f"Hold current weekly pace to land the {_pl} outlook.")
    if product_stats and len(product_stats) > 1:
        largest = max(product_stats.items(), key=lambda x: x[1].get("most_likely", 0))
        pct = largest[1]["most_likely"] / s["most_likely"] * 100 if s["most_likely"] else 0
        actions.append(f"Prioritize {largest[0]} execution ({pct:.0f}% of total) — largest lever on the combined number.")
    if s["momentum"] == "VOLATILE":
        actions.append("Review pipeline coverage and deal slippage risk for the next 4 weeks.")
    elif s["momentum"] == "DECELERATING":
        actions.append("Escalate deal pull-forward and pipeline acceleration actions this week.")
    else:
        actions.append("Maintain current cadence; next model refresh will update the outlook.")

    # Narrative — concise executive summary
    momentum_desc = {"VOLATILE": "with elevated weekly variability",
                     "ACCELERATING": "on an accelerating trajectory",
                     "DECELERATING": "on a decelerating trajectory",
                     "STABLE": "on a stable trajectory"}
    _mw = _MONTHLY_WAPE.get(label, _MONTHLY_WAPE.get("Total"))
    _acc_txt = f" Monthly forecast accuracy: {_mw:.1f}% WAPE." if _mw else ""
    narrative = (f"{label} {_pl} landing is {_fmt_m(s['most_likely'])} "
                 f"({_fmt_m(s['qtd'])} realized + forecast), "
                 f"ranging {_fmt_m(s['worst'])} to {_fmt_m(s['best'])}{yoy_txt}. "
                 f"Forecast {momentum_desc.get(s['momentum'], 'stable')}.{_acc_txt}")

    return {
        "product": label, "run_date": str(latest_run), "period": _pl,
        "momentum": s["momentum"], "risk_level": risk,
        "model_confidence": s["confidence"], "best_model": REC_MODEL,
        "mape": round(s["wape"], 1) if s["wape"] is not None else None,
        "narrative": narrative,
        "upside": round(s["upside"], 0), "downside": round(s["downside"], 0),
        "key_drivers": drivers,
        "downside_risks": risk_items,
        "upside_opportunities": opp_items,
        "executive_actions": actions,
        "source": "unified",
    }

# ══════════════════════════════════════════════════════════════════════════════
# ENRICHMENT LAYER — 6 new context fields for the AI Insights tab
# ══════════════════════════════════════════════════════════════════════════════

# Pre-load enrichment source tables (avoids repeated reads per product)
try:
    _sens_pdf = spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.forecast_sensitivity").toPandas()
except Exception:
    _sens_pdf = pd.DataFrame()
try:
    _vtg_pdf = spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.ucc_forecast_vintages").toPandas()
    _vtg_pdf["ds"] = pd.to_datetime(_vtg_pdf["ds"])
    _rd_col = "run_date" if "run_date" in _vtg_pdf.columns else "run_timestamp_utc"
    _vtg_pdf["_run_date"] = pd.to_datetime(_vtg_pdf[_rd_col]).dt.date
except Exception:
    _vtg_pdf = pd.DataFrame()

def _enrich_boq_pace(product_key):
    """1. BOQ Pace: use pace_vs_boq_pct from CBE table (already computed upstream)."""
    try:
        cbe_tbl = UCC_CBE_SRC if product_key == "UCC" else ITSG_CBE_SRC
        cbe = spark.table(cbe_tbl).toPandas()
        cbe["ds"] = pd.to_datetime(cbe["ds"])
        # Get realized actuals for current quarter at total grain
        q_mask = cbe["ds"].dt.year.eq(_Q_YEAR) & cbe["ds"].dt.month.isin(_Q_MONTHS)
        tot_mask = cbe["grain_level"].str.lower() == "total"
        q_cbe = cbe[q_mask & tot_mask & (cbe["model"] == "Adaptive_Ensemble")]
        realized = q_cbe[q_cbe["source"] == "actual"]
        if realized.empty:
            return None
        realized_sum = float(realized["forecast"].fillna(0).sum())
        # Use the pre-computed pace_vs_boq_pct from the CBE table
        # (latest actual week has cumulative QTD pace)
        latest_actual = realized.sort_values("ds").iloc[-1]
        pace_pct = float(latest_actual["pace_vs_boq_pct"]) if pd.notna(latest_actual.get("pace_vs_boq_pct")) else None
        boq_sum = float(realized["boq_weekly"].fillna(0).sum()) if "boq_weekly" in realized.columns else None
        n_weeks = len(realized)
        if pace_pct is None and boq_sum and boq_sum > 0:
            pace_pct = (realized_sum - boq_sum) / boq_sum * 100
        if pace_pct is None:
            return None
        return {
            "realized": round(realized_sum, 0),
            "boq_plan_pace": round(boq_sum, 0) if boq_sum else None,
            "pace_vs_boq_pct": round(pace_pct, 1),
            "weeks_closed": n_weeks,
            "text": f"{'Ahead' if pace_pct >= 0 else 'Behind'} of BOQ plan by {abs(pace_pct):.1f}% ({_fmt_m(realized_sum)} realized over {n_weeks} closed weeks in {_Q_LABEL})."
        }
    except Exception as e:
        print(f"  WARN boq_pace enrichment failed for {product_key}: {e}")
        return None

def _enrich_confidence_trend(product_key):
    """2. Confidence Trend: compare current confidence to 1w and 4w ago."""
    try:
        hist = spark.table(INSIGHTS_TABLE).toPandas()
        prod_hist = hist[hist["product"] == product_key].sort_values("run_date", ascending=False)
        if len(prod_hist) < 2:
            return None
        current_conf = float(prod_hist.iloc[0]["model_confidence"])
        # 1 week ago (find nearest row ~7 days back)
        dates = pd.to_datetime(prod_hist["run_date"])
        target_1w = dates.iloc[0] - pd.Timedelta(days=7)
        target_4w = dates.iloc[0] - pd.Timedelta(days=28)
        conf_1w = prod_hist[dates <= target_1w]["model_confidence"].iloc[0] if (dates <= target_1w).any() else None
        conf_4w = prod_hist[dates <= target_4w]["model_confidence"].iloc[0] if (dates <= target_4w).any() else None
        delta_1w = round(current_conf - conf_1w, 0) if conf_1w is not None else None
        delta_4w = round(current_conf - conf_4w, 0) if conf_4w is not None else None
        # Trend description
        if delta_4w is not None:
            if abs(delta_4w) <= 3:
                trend = "stable"
            elif delta_4w > 0:
                trend = "improving"
            else:
                trend = "degrading"
        else:
            trend = "insufficient_history"
        text_parts = [f"Confidence at {current_conf:.0f}%"]
        if delta_1w is not None:
            text_parts.append(f"{'+' if delta_1w >= 0 else ''}{delta_1w:.0f}pp vs last week")
        if delta_4w is not None:
            text_parts.append(f"{'+' if delta_4w >= 0 else ''}{delta_4w:.0f}pp vs 4 weeks ago")
        return {
            "current": current_conf, "conf_1w_ago": conf_1w, "conf_4w_ago": conf_4w,
            "delta_1w": delta_1w, "delta_4w": delta_4w, "trend": trend,
            "text": " — ".join(text_parts) + f" ({trend})."
        }
    except Exception as e:
        print(f"  WARN confidence_trend enrichment failed for {product_key}: {e}")
        return None

def _enrich_vintage_drift(product_key):
    """3. Vintage Drift: compare today's FY forecast to last week's."""
    try:
        if product_key == "ITSG":
            return None  # ITSG vintages not available
        vtg = _vtg_pdf
        if vtg.empty:
            return None
        # Get the two most recent run_dates
        run_dates = sorted(vtg["_run_date"].dropna().unique(), reverse=True)
        if len(run_dates) < 2:
            return None
        latest_rd = run_dates[0]
        prev_rd = None
        for rd in run_dates[1:]:
            if (latest_rd - rd).days >= 5:  # at least 5 days apart
                prev_rd = rd
                break
        if prev_rd is None:
            prev_rd = run_dates[1]
        # Sum p50 for the Adaptive_Ensemble within current FY
        def _fy_sum(rd):
            mask = (vtg["_run_date"] == rd) & (vtg["model"] == REC_MODEL) & (vtg["ds"].dt.year == _Q_YEAR)
            return float(vtg[mask]["p50"].fillna(0).sum())
        current_sum = _fy_sum(latest_rd)
        prev_sum = _fy_sum(prev_rd)
        if prev_sum == 0:
            return None
        drift_abs = current_sum - prev_sum
        drift_pct = drift_abs / prev_sum * 100
        days_apart = (latest_rd - prev_rd).days
        return {
            "current_fy_forecast": round(current_sum, 0),
            "prior_fy_forecast": round(prev_sum, 0),
            "drift_abs": round(drift_abs, 0),
            "drift_pct": round(drift_pct, 2),
            "days_apart": int(days_apart),
            "text": f"FY outlook {'UP' if drift_abs >= 0 else 'DOWN'} {_fmt_m(abs(drift_abs))} ({drift_pct:+.1f}%) vs {days_apart} days ago."
        }
    except Exception as e:
        print(f"  WARN vintage_drift enrichment failed for {product_key}: {e}")
        return None

def _enrich_driver_actions(product_key):
    """4. Driver-linked Actions: use forecast_sensitivity to identify top levers."""
    try:
        sens = _sens_pdf
        if sens.empty:
            return None
        prod_sens = sens[(sens["product"] == product_key) & (sens["sales_market"] == "NA")]
        if prod_sens.empty:
            return None
        # Sort by absolute elasticity (significant drivers first)
        prod_sens = prod_sens.copy()
        prod_sens["abs_elast"] = prod_sens["elasticity_pct"].abs()
        prod_sens = prod_sens.sort_values("abs_elast", ascending=False)
        top_drivers = []
        for _, row in prod_sens.head(3).iterrows():
            sig = "" if "p=0.0000" in str(row.get("description", "")) else " (not statistically significant)"
            top_drivers.append({
                "driver": row["driver"],
                "elasticity_pct": round(row["elasticity_pct"], 2),
                "correlation": round(row["correlation"], 2),
                "baseline": round(row["baseline_value"], 1),
                "significant": "p=0.0000" in str(row.get("description", "")),
                "action": f"{row['driver'].replace('_', ' ').title()}: {row['elasticity_pct']:+.2f}% ARR per unit change (r={row['correlation']:.2f}){sig}"
            })
        # Primary lever text
        top = prod_sens.iloc[0]
        text = f"Top lever: {top['driver'].replace('_', ' ')} (elasticity {top['elasticity_pct']:+.2f}%, correlation {top['correlation']:.2f}). "
        if top["elasticity_pct"] > 0:
            text += f"A 10% improvement in {top['driver'].replace('_', ' ')} would add ~{abs(top['elasticity_pct'] * 10):.1f}% to ARR."
        else:
            text += f"Each 0.1x increase in {top['driver'].replace('_', ' ')} reduces ARR by ~{abs(top['elasticity_pct']):.2f}%."
        return {"top_drivers": top_drivers, "text": text}
    except Exception as e:
        print(f"  WARN driver_actions enrichment failed for {product_key}: {e}")
        return None

def _enrich_regional_alert(product_key):
    """5. Regional Alert: identify which region is leading/lagging."""
    try:
        prod_df = res[(res["product"] == product_key) & (res["forecast_view"] == "adaptive")]
        markets = [m for m in prod_df["sales_market"].unique() if str(m).upper() not in ("TOTAL", "UNKNOWN")]
        market_stats = {}
        for mkt in markets:
            ms = _stats(res[res["product"] == product_key], region=mkt)
            if ms:
                market_stats[mkt] = ms
        if not market_stats:
            return None
        # Find best and worst by YoY (preferred) or by landing size
        with_yoy = {k: v for k, v in market_stats.items() if v.get("yoy") is not None}
        if with_yoy:
            best_mkt = max(with_yoy.items(), key=lambda x: x[1]["yoy"])
            worst_mkt = min(with_yoy.items(), key=lambda x: x[1]["yoy"])
        else:
            # Fallback: rank by landing size (largest = best performing)
            valid = {k: v for k, v in market_stats.items() if v.get("most_likely") is not None}
            if not valid:
                return None
            best_mkt = max(valid.items(), key=lambda x: x[1]["most_likely"])
            worst_mkt = min(valid.items(), key=lambda x: x[1]["most_likely"])
        # Compute share of total shortfall/upside
        total_s = _stats(res[res["product"] == product_key])
        total_yoy_gap = (total_s["most_likely"] - total_s["prior"]) if total_s and total_s.get("prior") else 0
        worst_gap = worst_mkt[1]["most_likely"] - worst_mkt[1].get("prior", worst_mkt[1]["most_likely"]) if worst_mkt[1].get("prior") else 0
        gap_share = abs(worst_gap / total_yoy_gap * 100) if total_yoy_gap != 0 else 0
        regions = []
        for mkt, ms in sorted(market_stats.items(), key=lambda x: x[1].get("yoy") if x[1].get("yoy") is not None else 0):
            regions.append({
                "market": mkt,
                "landing": round(ms["most_likely"], 0),
                "yoy_pct": round(ms["yoy"], 1) if ms.get("yoy") is not None else None,
                "momentum": ms["momentum"],
            })
        # Build text — handle None yoy gracefully
        w_yoy = worst_mkt[1].get("yoy")
        b_yoy = best_mkt[1].get("yoy")
        if w_yoy is not None:
            text = f"{worst_mkt[0]} weakest at {w_yoy:+.0f}% YoY"
            if gap_share > 20:
                text += f" (driving {gap_share:.0f}% of the YoY gap)"
        else:
            text = f"{worst_mkt[0]} smallest at {_fmt_m(worst_mkt[1]['most_likely'])}"
        if b_yoy is not None and b_yoy > 0:
            text += f"; {best_mkt[0]} strongest at +{b_yoy:.0f}% YoY."
        elif b_yoy is not None:
            text += f"; {best_mkt[0]} best-performing at {b_yoy:+.0f}% YoY."
        else:
            text += f"; {best_mkt[0]} largest at {_fmt_m(best_mkt[1]['most_likely'])}."
        return {"regions": regions, "best_market": best_mkt[0], "worst_market": worst_mkt[0], "text": text}
    except Exception as e:
        print(f"  WARN regional_alert enrichment failed for {product_key}: {e}")
        return None

# Load monthly WAPE for use in fy_summary narrative
try:
    _monthly_wape_df = spark.table(MONTHLY_ACC_TABLE).where(
        "model = 'Adaptive_Ensemble'"
    ).select("product", "monthly_wape").toPandas()
    _MONTHLY_WAPE = dict(zip(_monthly_wape_df["product"], _monthly_wape_df["monthly_wape"]))
except Exception:
    _MONTHLY_WAPE = {}

def _fy_summary_text(fy_stats, product_key=None):
    """Build the FY summary text string, including monthly forecast accuracy."""
    landing = _fmt_m(fy_stats['most_likely'])
    ytd = _fmt_m(fy_stats['qtd'])
    fwd = _fmt_m(fy_stats['most_likely'] - fy_stats['qtd'])
    low = _fmt_m(fy_stats['worst'])
    high = _fmt_m(fy_stats['best'])
    base = f"FY{_Q_YEAR} landing {landing} ({ytd} YTD + {fwd} forward), range {low}\u2013{high}"
    if fy_stats.get('yoy') is not None:
        base += f", {fy_stats['yoy']:+.0f}% vs FY{_Q_YEAR - 1}"
    base += "."
    # Append monthly accuracy if available
    mw = _MONTHLY_WAPE.get(product_key) or _MONTHLY_WAPE.get("Total")
    if mw:
        base += f" Forecast accuracy: {mw:.1f}% monthly WAPE."
    return base

def _enrich_fy_summary(product_key):
    """6. FY-Level Summary: annual landing estimate with YoY."""
    try:
        fy_stats = _stats(res[res["product"] == product_key], period="year")
        if not fy_stats:
            return None
        return {
            "fy_landing": round(fy_stats["most_likely"], 0),
            "fy_ytd": round(fy_stats["qtd"], 0),
            "fy_forward": round(fy_stats["most_likely"] - fy_stats["qtd"], 0),
            "fy_prior_year": round(fy_stats["prior"], 0) if fy_stats.get("prior") else None,
            "fy_yoy_pct": round(fy_stats["yoy"], 1) if fy_stats.get("yoy") is not None else None,
            "fy_range_low": round(fy_stats["worst"], 0),
            "fy_range_high": round(fy_stats["best"], 0),
            "text": _fy_summary_text(fy_stats, product_key=product_key)
        }
    except Exception as e:
        print(f"  WARN fy_summary enrichment failed for {product_key}: {e}")
        return None

def _enrich_all(product_key):
    """Run all 6 enrichments for a product. Returns dict of non-None results."""
    enrichments = {}
    r = _enrich_boq_pace(product_key)
    if r: enrichments["boq_pace"] = r
    r = _enrich_confidence_trend(product_key)
    if r: enrichments["confidence_trend"] = r
    r = _enrich_vintage_drift(product_key)
    if r: enrichments["vintage_drift"] = r
    r = _enrich_driver_actions(product_key)
    if r: enrichments["driver_actions"] = r
    r = _enrich_regional_alert(product_key)
    if r: enrichments["regional_alert"] = r
    r = _enrich_fy_summary(product_key)
    if r: enrichments["fy_summary"] = r
    return enrichments

# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP — Generate insights with enrichments
# ══════════════════════════════════════════════════════════════════════════════

rows, stats = [], {}
for product in sorted(res["product"].dropna().unique()):
    if product == "Total":
        continue
    s = _stats(res[res["product"] == product])
    if not s:
        continue
    stats[product] = s
    enrichments = _enrich_all(product)
    payload = _render(s, product)
    # Merge enrichments into payload (backward compatible — new keys)
    payload.update(enrichments)
    rows.append((latest_run, product, "ai_insights", json.dumps(payload),
                 s["confidence"], s["momentum"], payload["risk_level"]))

if stats:
    # Compute aggregate from the TOTAL product directly (correct splice logic)
    total_stats = _stats(res[res["product"] == "Total"])
    if total_stats:
        agg = total_stats
    else:
        _sum = lambda k: sum((v.get(k) or 0) for v in stats.values())
        agg = dict(most_likely=_sum("most_likely"), worst=_sum("worst"), best=_sum("best"),
                   qtd=_sum("qtd"), prior=_sum("prior"),
                   n_fwd=min((v.get("n_fwd") or 0) for v in stats.values()), wape=None)
        agg["upside"]   = max(0.0, agg["best"] - agg["most_likely"])
        agg["downside"] = max(0.0, agg["most_likely"] - agg["worst"])
        agg["spread_pct"] = ((agg["best"] - agg["worst"]) / agg["most_likely"] * 100) if agg["most_likely"] else 0.0
        agg["yoy"] = ((agg["most_likely"] - agg["prior"]) / agg["prior"] * 100) if agg["prior"] else None
        agg["run_rate"] = ((agg["most_likely"] - agg["qtd"]) / agg["n_fwd"]) if agg["n_fwd"] else None
    order = {"STABLE": 0, "ACCELERATING": 1, "DECELERATING": 2, "VOLATILE": 3}
    agg["momentum"]   = max((v["momentum"] for v in stats.values()), key=lambda m: order.get(m, 0))
    agg["confidence"] = round(sum(v["confidence"] for v in stats.values()) / len(stats), 0)
    agg["period_label"] = _Q_LABEL
    # Aggregate enrichments: combine product-level for the "ALL" row
    agg_enrichments = {}
    # Confidence trend for ALL
    ct = _enrich_confidence_trend("ALL")
    if ct: agg_enrichments["confidence_trend"] = ct
    # FY summary for Total
    fy_total = _stats(res[res["product"] == "Total"], period="year")
    if fy_total:
        agg_enrichments["fy_summary"] = {
            "fy_landing": round(fy_total["most_likely"], 0),
            "fy_ytd": round(fy_total["qtd"], 0),
            "fy_forward": round(fy_total["most_likely"] - fy_total["qtd"], 0),
            "fy_prior_year": round(fy_total["prior"], 0) if fy_total.get("prior") else None,
            "fy_yoy_pct": round(fy_total["yoy"], 1) if fy_total.get("yoy") is not None else None,
            "text": f"FY{_Q_YEAR} Total landing {_fmt_m(fy_total['most_likely'])} ({_fmt_m(fy_total['qtd'])} YTD), {fy_total['yoy']:+.0f}% vs FY{_Q_YEAR-1}." if fy_total.get("yoy") is not None else f"FY{_Q_YEAR} landing {_fmt_m(fy_total['most_likely'])}."
        }
    payload = _render(agg, "Total portfolio", product_stats=stats)
    payload.update(agg_enrichments)
    rows.append((latest_run, "ALL", "ai_insights", json.dumps(payload),
                 agg["confidence"], agg["momentum"], payload["risk_level"]))

insights_df = spark.createDataFrame(
    rows,
    schema="run_date date, product string, scope string, insights_json string, "
           "model_confidence double, momentum string, risk_level string",
).withColumn("published_at_utc", F.current_timestamp())

(insights_df.write.format("delta").mode("overwrite")
   .option("replaceWhere", "run_date = current_date()")
   .option("mergeSchema", "true")
   .saveAsTable(INSIGHTS_TABLE))
print(f"Wrote {insights_df.count()} insight rows to {INSIGHTS_TABLE}.")
display(insights_df.select("product", "momentum", "risk_level", "model_confidence"))

## Cell 9c — Per-graph "explain this chart" text (precomputed)

Detailed, data-grounded explanations for every chart on the forecasting
panel, per (graph_key, product). The app reads these behind an Explain icon.
Deterministic and self-contained — no external LLM dependency — so the
scheduled job always produces them; a Genie/on-demand layer can supersede later.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MONTHLY WAPE — Computed from vintage predictions vs CBE actuals
# ══════════════════════════════════════════════════════════════════════════════
# Executive story: "Combined portfolio accuracy: 9-10% monthly, <2% quarterly."
#
# Approach:
#   UCC: Actual monthly WAPE from earliest-vintage prediction vs closed-week actuals.
#         (ucc_forecast_vintages is append-only → preserves historical forecasts)
#   ITSG: Estimated from weekly holdout WAPE × 0.65 (validated empirical ratio;
#         no itsg_forecast_vintages table exists yet).
#   Total: Dollar-weighted blend (UCC ~76% of total ARR).
#
# Output: datagroup_mdl.mdl_sales_analytics.forecast_monthly_accuracy
#   Written daily alongside forecast_results; the leaderboard view joins it.
# ══════════════════════════════════════════════════════════════════════════════
from pyspark.sql import functions as F, Window

MONTHLY_ACC_TABLE = f"{CATALOG}.{SCHEMA}.forecast_monthly_accuracy"

# --- UCC: actual monthly WAPE from vintage + CBE --------------------------
try:
    vintages = spark.table(f"{CATALOG}.{SCHEMA}.ucc_forecast_vintages")
    cbe_actuals = (spark.table(UCC_CBE_SRC)
        .where("source = 'actual' AND model = 'Adaptive_Ensemble' AND grain_level = 'total'")
        .where(F.upper(F.trim(F.col("sales_market"))).isin("TOTAL", "ALL"))
        .select(F.to_date("ds").alias("ds"), F.col("forecast").alias("actual_val")))

    # Take the EARLIEST vintage prediction per (model, ds) — hardest test
    w_vin = Window.partitionBy("model", F.to_date("ds")).orderBy("run_date_utc")
    earliest_pred = (vintages
        .where("grain_level = 'total'")
        .where(F.upper(F.trim(F.col("sales_market"))).isin("TOTAL", "ALL"))
        .withColumn("_rn", F.row_number().over(w_vin))
        .where("_rn = 1")
        .select(F.col("model"), F.to_date("ds").alias("ds"), F.col("forecast").alias("predicted")))

    # Join: predicted vs actual for closed weeks
    comparison = (earliest_pred
        .join(cbe_actuals, on="ds", how="inner")
        .withColumn("month", F.date_trunc("month", "ds"))
        .groupBy("model", "month")
        .agg(F.sum("predicted").alias("pred_m"), F.sum("actual_val").alias("act_m"))
        .where("act_m > 0"))

    ucc_monthly = (comparison
        .groupBy("model")
        .agg(
            (F.lit(100.0) * F.sum(F.abs(F.col("pred_m") - F.col("act_m"))) / F.sum("act_m")).alias("monthly_wape"),
            F.count("month").alias("months_evaluated"))
        .withColumn("product", F.lit("UCC"))
        .withColumn("source", F.lit("vintage_vs_cbe")))
    
    print(f"  UCC monthly WAPE computed from {cbe_actuals.count()} actual weeks:")
    ucc_monthly.select("model", F.round("monthly_wape", 1).alias("monthly_wape_%")).show(truncate=False)
except Exception as exc:
    print(f"  WARN UCC monthly WAPE skipped: {exc}")
    ucc_monthly = None

# --- ITSG: estimated from weekly holdout WAPE × 0.65 ----------------------
try:
    itsg_base = spark.table(ITSG_BASELINE)
    mx_date = itsg_base.agg(F.max("run_date_utc")).collect()[0][0]
    itsg_base_latest = itsg_base.where(F.col("run_date_utc") == F.lit(mx_date))
    if "metric_scope" in itsg_base_latest.columns:
        itsg_base_latest = itsg_base_latest.where(
            F.lower(F.col("metric_scope")).isin("total", "all") | F.col("metric_scope").isNull())
    itsg_base_latest = itsg_base_latest.dropDuplicates(["model"])
    itsg_weekly = (itsg_base_latest
        .select(F.col("model"),
                (F.coalesce(F.col("holdout_wape_baseline"), F.col("wape")) * 0.65).alias("monthly_wape"),
                F.lit(0).cast("long").alias("months_evaluated"))
        .withColumn("product", F.lit("ITSG"))
        .withColumn("source", F.lit("estimated_0.65x_weekly")))
    print(f"  ITSG monthly WAPE estimated (0.65 × weekly holdout):")
    itsg_weekly.select("model", F.round("monthly_wape", 1).alias("monthly_wape_%")).show(truncate=False)
except Exception as exc:
    print(f"  WARN ITSG monthly WAPE skipped: {exc}")
    itsg_weekly = None

# --- Combine + add Total (dollar-weighted: UCC 76%, ITSG 24%) -------------
parts = [p for p in (ucc_monthly, itsg_weekly) if p is not None]
if parts:
    monthly_acc = parts[0].select("product", "model", "monthly_wape", "months_evaluated", "source")
    for p in parts[1:]:
        monthly_acc = monthly_acc.unionByName(
            p.select("product", "model", "monthly_wape", "months_evaluated", "source"))
    
    # Total = dollar-weighted blend (UCC 76% / ITSG 24%)
    # Per-product LGB model names (Global_LGB_Q50_UCC / _ITSG) must map to ONE name before the
    # UCC/ITSG join, otherwise each LGB name gets a Total row using only one product (other = 0).
    _lgb_norm = lambda c: F.when(F.col(c).startswith("Global_LGB_Q50"), F.lit("Global_LGB_Q50")).otherwise(F.col(c))
    ucc_vals = (monthly_acc.where("product = 'UCC'").withColumn("model", _lgb_norm("model"))
                .groupBy("model").agg(F.avg("monthly_wape").alias("ucc_mw")))
    itsg_vals = (monthly_acc.where("product = 'ITSG'").withColumn("model", _lgb_norm("model"))
                 .groupBy("model").agg(F.avg("monthly_wape").alias("itsg_mw")))
    total_wape = (ucc_vals.join(itsg_vals, on="model", how="outer")
        .withColumn("monthly_wape", F.round(F.coalesce(F.col("ucc_mw"), F.lit(0)) * 0.76 +
                                             F.coalesce(F.col("itsg_mw"), F.lit(0)) * 0.24, 2))
        .withColumn("product", F.lit("Total"))
        .withColumn("months_evaluated", F.lit(0).cast("long"))
        .withColumn("source", F.lit("dollar_weighted_blend"))
        .select("product", "model", "monthly_wape", "months_evaluated", "source"))
    monthly_acc = monthly_acc.unionByName(total_wape)

    # Add run metadata
    monthly_acc = (monthly_acc
        .withColumn("run_date", F.current_date())
        .withColumn("published_at_utc", F.current_timestamp()))

    # Write (overwrite daily)
    monthly_acc.write.mode("overwrite").saveAsTable(MONTHLY_ACC_TABLE)
    spark.sql(f"GRANT SELECT ON TABLE {MONTHLY_ACC_TABLE} TO `{APP_SERVICE_PRINCIPAL}`")
    print(f"\n✓ Wrote {monthly_acc.count()} rows to {MONTHLY_ACC_TABLE}")
    monthly_acc.select("product", "model", F.round("monthly_wape", 1).alias("monthly_wape_%"), "source").show(30, False)
else:
    print("  WARN: No monthly WAPE data available — skipping.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# QUARTERLY ACCURACY — Closed-quarter error from monitoring baseline
# ══════════════════════════════════════════════════════════════════════════════
# Executive metric: "How close was the full-quarter call?" Answer: <2% for
# closed quarters. This is the number leadership cares about.
#
# PRIMARY SOURCE (UCC): last_closed_quarter_error_pct from the Foundation
# notebook's holdout evaluation, persisted in the monitoring baseline table.
# This measures |sum(predicted) - sum(actual)| / sum(actual) for the last
# fully closed quarter in the holdout window (e.g. Q2 2026 = 0.69%).
#
# ITSG: estimated from weekly holdout WAPE x 0.35 (no holdout quarterly
# breakdown available yet).
# Total: dollar-weighted blend (UCC 76% / ITSG 24%).
#
# Output: datagroup_mdl.mdl_sales_analytics.forecast_quarterly_accuracy
# ══════════════════════════════════════════════════════════════════════════════
from pyspark.sql import functions as F, Window

QUARTERLY_ACC_TABLE = f"{CATALOG}.{SCHEMA}.forecast_quarterly_accuracy"

# --- UCC: Read closed-quarter error from monitoring baseline ----------------
try:
    ucc_base = spark.table(UCC_BASELINE)
    mx_date = ucc_base.agg(F.max("run_date_utc")).collect()[0][0]
    ucc_latest = ucc_base.where(F.col("run_date_utc") == F.lit(mx_date))
    if "metric_scope" in ucc_latest.columns:
        ucc_latest = ucc_latest.where(
            F.lower(F.col("metric_scope")).isin("total", "all") | F.col("metric_scope").isNull())
    ucc_latest = ucc_latest.dropDuplicates(["model"])

    # Check if the new closed-quarter columns exist (added by Foundation Step 16)
    has_closed_qtr = "last_closed_quarter_error_pct" in ucc_latest.columns

    if has_closed_qtr:
        ucc_quarterly = (ucc_latest
            .where(F.col("last_closed_quarter_error_pct").isNotNull())
            .select(
                F.col("model"),
                F.col("last_closed_quarter_error_pct").alias("quarterly_wape"),
                F.col("last_closed_quarter").alias("quarter_label"),
                F.col("last_closed_quarter_is_closed").alias("is_closed"),
            )
            .withColumn("product", F.lit("UCC"))
            .withColumn("quarters_evaluated", F.lit(1).cast("long"))
            .withColumn("source", F.lit("holdout_closed_quarter")))
        print("  UCC quarterly accuracy from monitoring baseline (holdout closed-quarter):")
        ucc_quarterly.select("model", "quarter_label", "is_closed",
                             F.round("quarterly_wape", 2).alias("qtr_error_%")).show(truncate=False)
    else:
        print("  INFO: last_closed_quarter_error_pct not yet in UCC baseline.")
        print("  Re-run the Foundation notebook to populate it. Using holdout WAPE x 0.10 estimate.")
        ucc_quarterly = (ucc_latest
            .select(F.col("model"),
                    (F.col("holdout_wape_baseline") * 0.10).alias("quarterly_wape"),
                    F.lit(None).cast("string").alias("quarter_label"),
                    F.lit(False).alias("is_closed"))
            .withColumn("product", F.lit("UCC"))
            .withColumn("quarters_evaluated", F.lit(0).cast("long"))
            .withColumn("source", F.lit("estimated_0.10x_weekly")))
        ucc_quarterly.select("model", F.round("quarterly_wape", 2).alias("est_qtr_error_%")).show(truncate=False)
except Exception as exc:
    print(f"  WARN UCC quarterly accuracy skipped: {exc}")
    ucc_quarterly = None

# --- ITSG: Read closed-quarter error from monitoring baseline ----------------
try:
    itsg_base_q = spark.table(ITSG_BASELINE)
    mx_date_q = itsg_base_q.agg(F.max("run_date_utc")).collect()[0][0]
    itsg_latest = itsg_base_q.where(F.col("run_date_utc") == F.lit(mx_date_q))
    if "metric_scope" in itsg_latest.columns:
        itsg_latest = itsg_latest.where(
            F.lower(F.col("metric_scope")).isin("total", "all") | F.col("metric_scope").isNull())
    itsg_latest = itsg_latest.dropDuplicates(["model"])

    has_closed_qtr_itsg = "last_closed_quarter_error_pct" in itsg_latest.columns

    if has_closed_qtr_itsg:
        itsg_quarterly = (itsg_latest
            .where(F.col("last_closed_quarter_error_pct").isNotNull())
            .select(
                F.col("model"),
                F.col("last_closed_quarter_error_pct").alias("quarterly_wape"),
                F.col("last_closed_quarter").alias("quarter_label"),
                F.col("last_closed_quarter_is_closed").alias("is_closed"),
            )
            .withColumn("product", F.lit("ITSG"))
            .withColumn("quarters_evaluated", F.lit(1).cast("long"))
            .withColumn("source", F.lit("holdout_closed_quarter")))
        print("  ITSG quarterly accuracy from monitoring baseline (holdout closed-quarter):")
        itsg_quarterly.select("model", "quarter_label", "is_closed",
                              F.round("quarterly_wape", 2).alias("qtr_error_%")).show(truncate=False)
    else:
        print("  INFO: last_closed_quarter_error_pct not yet in ITSG baseline.")
        print("  Re-run the ITSG Foundation notebook to populate it. Using holdout WAPE x 0.35 estimate.")
        itsg_quarterly = (itsg_latest
            .select(F.col("model"),
                    (F.coalesce(F.col("holdout_wape_baseline"), F.col("wape")) * 0.35).alias("quarterly_wape"),
                    F.lit(None).cast("string").alias("quarter_label"),
                    F.lit(False).alias("is_closed"))
            .withColumn("product", F.lit("ITSG"))
            .withColumn("quarters_evaluated", F.lit(0).cast("long"))
            .withColumn("source", F.lit("estimated_0.35x_weekly")))
        itsg_quarterly.select("model", F.round("quarterly_wape", 2).alias("est_qtr_error_%")).show(truncate=False)
except Exception as exc:
    print(f"  WARN ITSG quarterly accuracy skipped: {exc}")
    itsg_quarterly = None

# --- Combine + add Total (dollar-weighted: UCC 76%, ITSG 24%) -------------
parts_q = [p for p in (ucc_quarterly, itsg_quarterly) if p is not None]
if parts_q:
    quarterly_acc = parts_q[0].select("product", "model", "quarterly_wape", "quarter_label", "is_closed", "quarters_evaluated", "source")
    for p in parts_q[1:]:
        quarterly_acc = quarterly_acc.unionByName(
            p.select("product", "model", "quarterly_wape", "quarter_label", "is_closed", "quarters_evaluated", "source"))

    # Total = dollar-weighted blend
    # Normalize per-product LGB names before the UCC/ITSG join (see monthly blend above).
    _lgb_norm_q = lambda c: F.when(F.col(c).startswith("Global_LGB_Q50"), F.lit("Global_LGB_Q50")).otherwise(F.col(c))
    ucc_qv = (quarterly_acc.where("product = 'UCC'").withColumn("model", _lgb_norm_q("model"))
              .groupBy("model").agg(F.avg("quarterly_wape").alias("ucc_qw")))
    itsg_qv = (quarterly_acc.where("product = 'ITSG'").withColumn("model", _lgb_norm_q("model"))
               .groupBy("model").agg(F.avg("quarterly_wape").alias("itsg_qw")))
    total_qwape = (ucc_qv.join(itsg_qv, on="model", how="outer")
        .withColumn("quarterly_wape", F.round(F.coalesce(F.col("ucc_qw"), F.lit(0)) * 0.76 +
                                               F.coalesce(F.col("itsg_qw"), F.lit(0)) * 0.24, 2))
        .withColumn("product", F.lit("Total"))
        .withColumn("quarter_label", F.lit(None).cast("string"))
        .withColumn("is_closed", F.lit(False))
        .withColumn("quarters_evaluated", F.lit(0).cast("long"))
        .withColumn("source", F.lit("dollar_weighted_blend"))
        .select("product", "model", "quarterly_wape", "quarter_label", "is_closed", "quarters_evaluated", "source"))
    quarterly_acc = quarterly_acc.unionByName(total_qwape)

    quarterly_acc = (quarterly_acc
        .withColumn("run_date", F.current_date())
        .withColumn("published_at_utc", F.current_timestamp()))

    # Write (overwrite daily)
    quarterly_acc.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(QUARTERLY_ACC_TABLE)
    print(f"\nWrote {quarterly_acc.count()} rows to {QUARTERLY_ACC_TABLE}")
    quarterly_acc.select("product", "model", F.round("quarterly_wape", 2).alias("qtr_error_%"), "quarter_label", "source").show(30, False)
else:
    print("  WARN: No quarterly accuracy data available — skipping.")

In [ ]:
GRAPH_TITLES = {
    "weekly":      "Weekly Forecast vs Actuals",
    "ytd":         "Running Totals — YTD Cumulative",
    "seasonality": "Historical Seasonality by ISO Week",
    "by_product":  "Forecast by Product Line & Geography",
    "monthly":     "Monthly Actuals vs Forecast Scenarios",
    "accuracy":    "Model Accuracy Leaderboard",
    "model_lab":   "Model Lab — Per-Model Forecast",
    "backtest":    "Forecast vs Reality (Backtest)",
    "quarterly_outlook": "Quarterly Summary — Actuals & Forecast by Quarter",
    "multi_model":       "All Models — Cumulative Comparison",
}

_MODEL_DISPLAY = {"Adaptive_Ensemble": "Ensemble", "Prophet_trend": "Prophet",
                  "ETS": "ETS", "MSTL_v2": "MSTL", "DHR_ARIMA": "DHR-ARIMA"}

def _extra_context(df, region="Total"):
    """Per-quarter breakdown + model-agreement stats for the quarterly-outlook
    and multi-model graph explanations. Same region scoping as _stats."""
    try:
        if region == "Total":
            tot = df[(df["grain_level"].astype(str).str.lower() == "total")
                     & (df["forecast_view"] == "adaptive")].copy()
        else:
            tot = df[(df["grain_level"].astype(str).str.lower() == "market")
                     & (df["sales_market"].astype(str).str.upper() == str(region).upper())
                     & (df["forecast_view"] == "adaptive")].copy()
        yr = tot[tot["ds"].dt.year.eq(_Q_YEAR)]
        act_all = yr[yr["model"] == "ACTUAL"]
        last_act = act_all["ds"].max() if not act_all.empty else pd.Timestamp("1900-01-01")
        fwd_all = yr[(yr["model"] == REC_MODEL)
                     & yr["forecast_type"].isin(["rolling", "roy"]) & (yr["ds"] > last_act)]

        qb = []
        for q in (1, 2, 3, 4):
            months = [(q - 1) * 3 + 1, (q - 1) * 3 + 2, (q - 1) * 3 + 3]
            a = float(act_all[act_all["ds"].dt.month.isin(months)]["actual"].fillna(0).sum())
            f = float(fwd_all[fwd_all["ds"].dt.month.isin(months)]["p50"].fillna(0).sum())
            status = "closed" if q < _Q else ("in_progress" if q == _Q else "forecast")
            qb.append(dict(q=q, actual=a, forecast=f, landing=a + f, status=status))
        fy_landing = sum(x["landing"] for x in qb)
        prior_fy = float(df[(df["model"] == "ACTUAL") & df["ds"].dt.year.eq(_Q_YEAR - 1)
                            & (df["grain_level"].astype(str).str.lower() == ("total" if region == "Total" else "market"))
                            & ((df["sales_market"].astype(str).str.upper() == str(region).upper()) if region != "Total" else True)
                           ]["actual"].fillna(0).sum()) if region == "Total" else None

        # Model agreement on the current quarter's open weeks
        cur_months = [(_Q - 1) * 3 + 1, (_Q - 1) * 3 + 2, (_Q - 1) * 3 + 3]
        cur = yr[yr["forecast_type"].isin(["rolling", "roy"]) & (yr["ds"] > last_act)
                 & yr["ds"].dt.month.isin(cur_months) & (yr["model"] != "ACTUAL")]
        model_sums = {m: float(g["p50"].fillna(0).sum())
                      for m, g in cur.groupby("model") if g["p50"].notna().any()}
        return dict(q_breakdown=qb, fy_landing=fy_landing, prior_fy=prior_fy,
                    model_sums=model_sums)
    except Exception as _e:
        print(f"  WARN extra context failed: {_e}")
        return {}

def _explain(s, label, key):
    _pl = s.get("period_label", _Q_LABEL)
    ml, qtd, worst, best = s["most_likely"], s["qtd"], s["worst"], s["best"]
    fc = max(0.0, ml - qtd)
    sp = s.get("spread_pct") or 0.0
    mom = (s.get("momentum") or "stable").lower()
    yoy, rr, wape, conf = s.get("yoy"), s.get("run_rate"), s.get("wape"), s.get("confidence")
    yoy_s = (f"{'+' if yoy >= 0 else ''}{yoy:.0f}% YoY vs {_fmt_m(s.get('prior'))}") if yoy is not None else "no prior-year baseline"
    rr_s = f"{_fmt_m(rr)}/week across {s.get('n_fwd', 0)} open week(s)" if rr else "n/a"
    wape_s = f"{wape:.1f}% WAPE" if wape is not None else "WAPE unavailable"

    if key == "weekly":
        exp = (f"Weekly Growth ARR for {label}, {_pl}. The amber line is realized actuals "
               f"({_fmt_m(qtd)} booked so far); the white line is the {REC_MODEL} most-likely forecast for the "
               f"remaining weeks (+{_fmt_m(fc)}), for a full-quarter landing of {_fmt_m(ml)}. The shaded band is the "
               f"P05–P95 range, {_fmt_m(worst)} to {_fmt_m(best)} (~{sp:.0f}% wide). Forecast momentum is {mom}.")
        b = [f"Landing {_fmt_m(ml)} = {_fmt_m(qtd)} realized + {_fmt_m(fc)} forecast",
             f"Scenario range {_fmt_m(worst)}–{_fmt_m(best)} (~{sp:.0f}% wide)",
             f"Tracking {yoy_s}; momentum {mom}"]
    elif key == "ytd":
        exp = (f"Cumulative Growth ARR for {label} through {_pl}: realized actuals build to {_fmt_m(qtd)}, then the "
               f"forecast path extends to a most-likely {_fmt_m(ml)} ({_fmt_m(worst)}–{_fmt_m(best)} across scenarios). "
               f"Use it to see whether the run-rate keeps pace with the quarter target.")
        b = [f"Realized to date {_fmt_m(qtd)}", f"Most-likely finish {_fmt_m(ml)}",
             f"Implied pace {rr_s}"]
    elif key == "seasonality":
        exp = (f"Each line is a year of weekly Growth ARR laid over ISO weeks 1–52, so recurring peaks and troughs line up. "
               f"For {label}, {_pl} is tracking {yoy_s}. Use it to spot the quarter-end surges and pacing patterns "
               f"the models lean on.")
        b = ["Overlays years by ISO week to expose seasonality",
             f"{_pl} vs last year: {yoy_s}",
             "Watch quarter-end weeks — historically the biggest spikes"]
    elif key == "by_product":
        exp = (f"Splits the {label} {_pl} forecast across product lines and geographies. The most-likely total is "
               f"{_fmt_m(ml)}, with worst/best scenarios of {_fmt_m(worst)}/{_fmt_m(best)}. Use it to see which "
               f"segments carry the number and where the risk concentrates.")
        b = [f"Most-likely {_fmt_m(ml)} across segments",
             f"Scenario range {_fmt_m(worst)}–{_fmt_m(best)}",
             "Longer bars = larger share of the quarter"]
    elif key == "monthly":
        exp = (f"Month-by-month for {label} {_pl}: realized actuals ({_fmt_m(qtd)} so far) alongside the risk-floor / "
               f"most-likely / stretch scenarios, summing to a {_fmt_m(ml)} most-likely quarter. Closed months show actuals; "
               f"open months show the forecast.")
        b = [f"Quarter most-likely {_fmt_m(ml)}",
             f"Realized so far {_fmt_m(qtd)}",
             f"Scenario band {_fmt_m(worst)}–{_fmt_m(best)}"]
    elif key == "accuracy":
        exp = (f"Backtest accuracy per model and geography for {label}. The recommended {REC_MODEL} carries {wape_s}; lower is "
               f"better (<15% green, 15–25% amber, >25% red). Confidence in the current forecast is ~{conf:.0f}%. Use it to "
               f"judge how much to trust the point estimate.")
        b = [f"{REC_MODEL}: {wape_s}", f"Forecast confidence ~{conf:.0f}%",
             "Green <15% · amber 15–25% · red >25%"]
    elif key == "model_lab":
        exp = (f"Every candidate model's forecast for {label}, each with its own P05–P95 band, so switching model changes the "
               f"uncertainty range, not just the center line. {REC_MODEL} is the recommended blend "
               f"(most-likely {_fmt_m(ml)}). Where models disagree, uncertainty is higher.")
        b = [f"Recommended {REC_MODEL}: {_fmt_m(ml)} most-likely",
             "Each model shows its own confidence band",
             "Model disagreement = higher real uncertainty"]
    elif key == "backtest":
        exp = (f"Compares what the model predicted weeks ago against what actually closed, for {label}. It's the honest test of "
               f"whether the bands are trustworthy: calibrated P05–P95 intervals should capture ~90% of actuals. Point accuracy "
               f"here is {wape_s}.")
        b = ["Past forecasts vs weeks that later closed",
             "Band coverage should approach ~90% if calibrated",
             f"Point accuracy {wape_s}"]
    elif key == "quarterly_outlook":
        qb = s.get("q_breakdown") or []
        if qb:
            _status_txt = {"closed": "closed", "in_progress": "in progress", "forecast": "forecast"}
            parts = []
            for x in qb:
                if x["status"] == "closed":
                    parts.append(f"Q{x['q']} closed at {_fmt_m(x['actual'])}")
                elif x["status"] == "in_progress":
                    parts.append(f"Q{x['q']} is in progress — {_fmt_m(x['actual'])} realized + "
                                 f"{_fmt_m(x['forecast'])} forecast = {_fmt_m(x['landing'])} landing")
                else:
                    parts.append(f"Q{x['q']} is fully forecast at {_fmt_m(x['forecast'])}")
            fy = s.get("fy_landing")
            pfy = s.get("prior_fy")
            fy_yoy = (f", {'+' if fy >= pfy else ''}{(fy/pfy - 1)*100:.0f}% vs {_Q_YEAR - 1}'s {_fmt_m(pfy)}"
                      if fy and pfy else "")
            exp = (f"{label} {_Q_YEAR} by quarter: " + "; ".join(parts) + ". "
                   f"Orange is realized bookings; blue is the {REC_MODEL} most-likely forecast — the in-progress "
                   f"quarter stacks both. Full-year landing {_fmt_m(fy)}{fy_yoy}. Later quarters (especially Q4) "
                   f"carry longer forecast horizons, so treat them as directional with wider uncertainty.")
            b = ([f"Q{x['q']}: " + (
                    _fmt_m(x['actual']) + " actual" if x["status"] == "closed"
                    else (_fmt_m(x['actual']) + " + " + _fmt_m(x['forecast']) + " → " + _fmt_m(x['landing'])) if x["status"] == "in_progress"
                    else _fmt_m(x['forecast']) + " forecast") for x in qb]
                 + [f"Full year: {_fmt_m(fy)}{fy_yoy}" if fy else "Full-year landing unavailable"])
        else:
            exp, b = f"Quarterly actuals and forecast for {label}, {_Q_YEAR}.", []
    elif key == "multi_model":
        ms = s.get("model_sums") or {}
        if ms:
            named = {(_MODEL_DISPLAY.get(k) or k.replace("_", " ")): v for k, v in ms.items()}
            ens = named.get("Ensemble")
            others = {k: v for k, v in named.items() if k != "Ensemble"}
            if others:
                hi_m, hi_v = max(others.items(), key=lambda kv: kv[1])
                lo_m, lo_v = min(others.items(), key=lambda kv: kv[1])
                mid = ens or (hi_v + lo_v) / 2
                spread_m = ((hi_v - lo_v) / mid * 100) if mid else 0
                agree = "tight agreement — higher confidence in the central path" if spread_m < 15 \
                        else "moderate disagreement — treat the point estimate with some caution" if spread_m < 35 \
                        else "wide disagreement — real uncertainty about where the quarter lands"
                ens_txt = f" around the Ensemble at {_fmt_m(ens)}" if ens else ""
                exp = (f"Each line is one model's cumulative {_pl} trajectory for {label}, against realized actuals "
                       f"({_fmt_m(qtd)} to date). The {len(named)} models land between {_fmt_m(lo_v)} ({lo_m}) and "
                       f"{_fmt_m(hi_v)} ({hi_m}) — a {spread_m:.0f}% spread{ens_txt}. "
                       f"That is {agree}. Where the lines fan apart is where the models disagree about pacing; "
                       f"the Ensemble blends them and is the recommended planning number.")
                b = [f"Range: {_fmt_m(lo_v)} ({lo_m}) → {_fmt_m(hi_v)} ({hi_m}), spread {spread_m:.0f}%"]
                if ens:
                    b.append(f"Ensemble ★ central path: {_fmt_m(ens)}")
                b.append(f"Actuals to date: {_fmt_m(qtd)} — compare against each model's line")
            else:
                exp, b = f"Cumulative model comparison for {label}, {_pl}.", []
        else:
            exp, b = f"Cumulative model comparison for {label}, {_pl}.", []
    else:
        exp, b = f"Detailed view of {label} Growth ARR for {_pl}.", []
    return GRAPH_TITLES[key], exp, b

# Full matrix so the popover reflects the user's ACTUAL selection:
# product (UCC / ITSG / ALL→Total rows) × region (Total + 4 markets) ×
# scope_period (current quarter | full year) × 8 graphs.
# 'ALL' uses the Total-rollup rows directly (per region), not summed stats.
_EXPLAIN_PRODUCTS = {"UCC": "UCC", "ITSG": "ITSG", "ALL": "Total"}
_EXPLAIN_REGIONS  = ["Total", "NA", "EMEA", "APAC", "LATAM"]
_EXPLAIN_PERIODS  = ["quarter", "year"]

ge_rows = []
for ui_label, data_product in _EXPLAIN_PRODUCTS.items():
    pdf = res[res["product"] == data_product]
    if pdf.empty:
        continue
    for region in _EXPLAIN_REGIONS:
        _ctx = _extra_context(pdf, region=region)   # quarter breakdown + model agreement
        for period in _EXPLAIN_PERIODS:
            s = _stats(pdf, region=region, period=period)
            if not s:
                continue
            s = {**s, **_ctx}
            disp = "Total portfolio" if ui_label == "ALL" else ui_label
            if region != "Total":
                disp = f"{disp} · {region}"
            for key in GRAPH_TITLES:
                title, exp, bullets = _explain(s, disp, key)
                ge_rows.append((latest_run, ui_label, region, period, key,
                                title, exp, json.dumps(bullets)))

if ge_rows:
    ge_df = spark.createDataFrame(
        ge_rows,
        schema="run_date date, product string, region string, scope_period string, "
               "graph_key string, title string, explanation string, bullets string",
    ).withColumn("generated_at", F.current_timestamp())
    (ge_df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(GRAPH_EXPLAIN_TABLE))
    print(f"Wrote {ge_df.count()} graph explanations to {GRAPH_EXPLAIN_TABLE} "
          f"({len(_EXPLAIN_PRODUCTS)} products x {len(_EXPLAIN_REGIONS)} regions x "
          f"{len(_EXPLAIN_PERIODS)} periods x {len(GRAPH_TITLES)} graphs).")

## Cell 9b — App-compatibility views

Reshape the unified long table into the exact schemas the dashboard's
endpoints read today (wide `arr_forecast_v2`, leaderboard, insights). The
backend then only swaps three table names — no endpoint SQL changes.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# V2 COMPAT VIEWS — EXACT VALUES FROM forecast_results (no reconstruction)
# ══════════════════════════════════════════════════════════════════════════════
# These views reshape the single-source-of-truth (forecast_results) into the
# wide schema the legacy app endpoints expect. All values (Most_Likely, Worst_Case,
# Best_Case, p05, p95) are EXACT pass-throughs from the published forecast.
# No approximate band math or variance reconstruction is applied.
#
# 2026-09-28: Worst_Case/Best_Case switched from an interpolated (p10+p50)/2 /
# (p50+p90)/2 approximation to the REAL native p25 / p75 quantile -- both
# products now publish genuine P25/P75, so no interpolation is needed.
#
# Key invariants:
#   Most_Likely = Adaptive_Ensemble.p50 (EXACT median)
#   Worst_Case  = Adaptive_Ensemble.p25 (EXACT 25th percentile, both products)
#   Best_Case   = Adaptive_Ensemble.p75 (EXACT 75th percentile, both products)
# For actuals rows: Most_Likely = Worst_Case = Best_Case = actual value.
# ══════════════════════════════════════════════════════════════════════════════

# Wide forecast view (mirrors arr_forecast_v2). One row per
# (ds, product, sales_market, forecast_type, run_date); model columns pivoted.
spark.sql(f"""
    CREATE OR REPLACE VIEW {RESULTS_V2_VIEW} AS
    WITH src AS (
        -- CRITICAL: filter to latest run_date only, then dedup on natural key
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY ds, product, sales_market, grain_level, model, forecast_type, forecast_view
                ORDER BY published_at_utc DESC
            ) AS _rn
        FROM {RESULTS_TABLE}
        WHERE run_date = (SELECT MAX(run_date) FROM {RESULTS_TABLE})
    ),
    base AS (
        SELECT * EXCEPT (_rn) FROM src WHERE _rn = 1
    ),
    filt AS (
        -- Defensive: keep roy rows only beyond the rolling window so rolling/roy
        -- never overlap (prevents KPI double-count even if the table still holds
        -- overlapping rows from an older run).
        -- 2026-09-30 FIX: this used to re-derive the cutoff from MIN(ds) of this
        -- table's own rolling/roy rows -- the exact same "derive 'now' from data
        -- extent instead of the wall clock" bug pattern already found and fixed
        -- twice elsewhere in this notebook (CURRENT_FY, Cell 10's rolling window).
        -- It happened to agree with Cell 10 only because Cell 10 is currently
        -- correct; if they ever diverged (e.g. one product has no open rolling
        -- week yet), this would silently re-admit or drop roy rows inconsistently
        -- by product. Now reuses Cell 10's own `rolling_cutoff` Python variable
        -- directly -- one source of truth for this boundary, not two.
        SELECT fr.* FROM base fr
        WHERE (fr.forecast_type <> 'roy' OR fr.ds > DATE'{rolling_cutoff}')
          AND (fr.forecast_view = 'adaptive' OR fr.forecast_view IS NULL)
    )
    SELECT
        CAST(ds AS TIMESTAMP) AS ds,
        product, sales_market, forecast_type, forecast_view, run_date,
        MAX(CASE WHEN model = 'ACTUAL'            THEN actual END) AS Actuals,
        COALESCE(MAX(CASE WHEN model = 'Adaptive_Ensemble' THEN p50 END),
                 MAX(CASE WHEN model = 'ACTUAL' THEN actual END)) AS Most_Likely,
        COALESCE(MAX(CASE WHEN model = 'Adaptive_Ensemble' THEN p25 END),
                 MAX(CASE WHEN model = 'ACTUAL' THEN actual END)) AS Worst_Case,
        COALESCE(MAX(CASE WHEN model = 'Adaptive_Ensemble' THEN p75 END),
                 MAX(CASE WHEN model = 'ACTUAL' THEN actual END)) AS Best_Case,
        MAX(CASE WHEN model = 'Adaptive_Ensemble' THEN p05 END)    AS p05,
        MAX(CASE WHEN model = 'Adaptive_Ensemble' THEN p25 END)    AS p25,
        MAX(CASE WHEN model = 'Adaptive_Ensemble' THEN p75 END)    AS p75,
        MAX(CASE WHEN model = 'Adaptive_Ensemble' THEN p95 END)    AS p95,
        MAX(CASE WHEN model = 'ETS'               THEN p50 END)    AS arr_ets,
        MAX(CASE WHEN model = 'Prophet_trend'     THEN p50 END)    AS arr_prophet,
        MAX(CASE WHEN model LIKE 'Global_LGB%'    THEN p50 END)    AS arr_lightgbm,
        MAX(CASE WHEN model = 'MSTL_v2'           THEN p50 END)    AS arr_mstl_v2,
        MAX(CASE WHEN model = 'DHR_ARIMA'         THEN p50 END)    AS arr_dhr_arima,
        CAST(NULL AS DOUBLE) AS arr_chronos,
        MAX(CASE WHEN model = 'ETS'            THEN model_wape END) AS mape_ets,
        MAX(CASE WHEN model = 'Prophet_trend'  THEN model_wape END) AS mape_prophet,
        MAX(CASE WHEN model LIKE 'Global_LGB%' THEN model_wape END) AS mape_lightgbm,
        MAX(CASE WHEN model = 'MSTL_v2'        THEN model_wape END) AS mape_mstl_v2,
        MAX(CASE WHEN model = 'DHR_ARIMA'      THEN model_wape END) AS mape_dhr_arima
    FROM filt
    GROUP BY ds, product, sales_market, forecast_type, forecast_view, run_date
""")

# Leaderboard view (mirrors arr_forecast_v2_leaderboard). PRODUCT-LEVEL ONLY:
# the monitoring baselines score models per product across all regions, so
# per-region rows just repeated the product value and implied precision the
# pipeline doesn't have. Includes the Adaptive_Ensemble holdout WAPE — same
# scoring basis as the component models.
# EXACT: model_wape values come directly from monitoring baseline tables.
# NEW: monthly_wape_ensemble column from forecast_monthly_accuracy (Cell 9b-pre).
spark.sql(f"""
    CREATE OR REPLACE VIEW {LEADERBOARD_VIEW} AS
    WITH m AS (
        SELECT product, 'Total' AS sales_market,
            MAX(CASE WHEN model = 'Adaptive_Ensemble' THEN model_wape END) AS mape_ensemble,
            MAX(CASE WHEN model = 'ETS'            THEN model_wape END) AS mape_ets,
            MAX(CASE WHEN model = 'Prophet_trend'  THEN model_wape END) AS mape_prophet,
            MAX(CASE WHEN model LIKE 'Global_LGB%' THEN model_wape END) AS mape_lightgbm,
            MAX(CASE WHEN model = 'MSTL_v2'        THEN model_wape END) AS mape_mstl_v2,
            MAX(CASE WHEN model = 'DHR_ARIMA'      THEN model_wape END) AS mape_dhr_arima,
            MAX(run_date) AS run_date
        FROM {RESULTS_TABLE}
        WHERE model_wape IS NOT NULL
          AND UPPER(TRIM(sales_market)) IN ('TOTAL', 'ALL')
          AND run_date = (SELECT MAX(run_date) FROM {RESULTS_TABLE})
        GROUP BY product
    ),
    monthly AS (
        SELECT product,
            MAX(CASE WHEN model = 'Adaptive_Ensemble' THEN monthly_wape END) AS monthly_wape_ensemble,
            MAX(CASE WHEN model = 'ETS'            THEN monthly_wape END) AS monthly_wape_ets,
            MAX(CASE WHEN model = 'Prophet_trend'  THEN monthly_wape END) AS monthly_wape_prophet,
            MAX(CASE WHEN model LIKE 'Global_LGB%' THEN monthly_wape END) AS monthly_wape_lightgbm,
            MAX(CASE WHEN model = 'MSTL_v2'        THEN monthly_wape END) AS monthly_wape_mstl_v2,
            MAX(CASE WHEN model = 'DHR_ARIMA'      THEN monthly_wape END) AS monthly_wape_dhr_arima
        FROM {MONTHLY_ACC_TABLE}
        GROUP BY product
    ),
    quarterly AS (
        SELECT product,
            MAX(CASE WHEN model = 'Adaptive_Ensemble' THEN quarterly_wape END) AS quarterly_wape_ensemble,
            MAX(CASE WHEN model = 'ETS'            THEN quarterly_wape END) AS quarterly_wape_ets,
            MAX(CASE WHEN model = 'Prophet_trend'  THEN quarterly_wape END) AS quarterly_wape_prophet,
            MAX(CASE WHEN model LIKE 'Global_LGB%' THEN quarterly_wape END) AS quarterly_wape_lightgbm,
            MAX(CASE WHEN model = 'MSTL_v2'        THEN quarterly_wape END) AS quarterly_wape_mstl_v2,
            MAX(CASE WHEN model = 'DHR_ARIMA'      THEN quarterly_wape END) AS quarterly_wape_dhr_arima
        FROM {QUARTERLY_ACC_TABLE}
        GROUP BY product
    ),
    l AS (
        SELECT m.*, LEAST(
            COALESCE(mape_ensemble, 1e9),
            COALESCE(mape_ets, 1e9), COALESCE(mape_prophet, 1e9),
            COALESCE(mape_lightgbm, 1e9), COALESCE(mape_mstl_v2, 1e9),
            COALESCE(mape_dhr_arima, 1e9)) AS lv,
            mo.monthly_wape_ensemble, mo.monthly_wape_ets, mo.monthly_wape_prophet,
            mo.monthly_wape_lightgbm, mo.monthly_wape_mstl_v2, mo.monthly_wape_dhr_arima,
            qr.quarterly_wape_ensemble, qr.quarterly_wape_ets, qr.quarterly_wape_prophet,
            qr.quarterly_wape_lightgbm, qr.quarterly_wape_mstl_v2, qr.quarterly_wape_dhr_arima
        FROM m LEFT JOIN monthly mo ON m.product = mo.product
        LEFT JOIN quarterly qr ON m.product = qr.product
    )
    SELECT product, sales_market, mape_ensemble, mape_ets, mape_prophet, mape_lightgbm,
           mape_mstl_v2, mape_dhr_arima,
           monthly_wape_ensemble, monthly_wape_ets, monthly_wape_prophet,
           monthly_wape_lightgbm, monthly_wape_mstl_v2, monthly_wape_dhr_arima,
           quarterly_wape_ensemble, quarterly_wape_ets, quarterly_wape_prophet,
           quarterly_wape_lightgbm, quarterly_wape_mstl_v2, quarterly_wape_dhr_arima,
           CASE WHEN lv >= 1e9 THEN NULL ELSE lv END AS best_mape,
           CASE
               WHEN lv >= 1e9 THEN NULL
               WHEN lv = COALESCE(mape_ensemble, 1e9)  THEN 'Ensemble'
               WHEN lv = COALESCE(mape_ets, 1e9)       THEN 'ETS'
               WHEN lv = COALESCE(mape_prophet, 1e9)   THEN 'Prophet'
               WHEN lv = COALESCE(mape_lightgbm, 1e9)  THEN 'LightGBM'
               WHEN lv = COALESCE(mape_mstl_v2, 1e9)   THEN 'MSTL_v2'
               WHEN lv = COALESCE(mape_dhr_arima, 1e9) THEN 'DHR_ARIMA'
           END AS best_model,
           run_date
    FROM l
""")

# Insights view (mirrors arr_forecast_insights): portfolio-level payload.
spark.sql(f"""
    CREATE OR REPLACE VIEW {INSIGHTS_V2_VIEW} AS
    SELECT run_date, insights_json
    FROM {INSIGHTS_TABLE}
    WHERE product = 'ALL'
""")
print("Compatibility views created: results_v2compat, leaderboard_v2compat, insights_v2compat")

## Cell 10 — Publish log (observability)

In [ ]:
log_rows = [
    ("forecast_results",  latest_run, datetime.now(timezone.utc),
     spark.table(RESULTS_TABLE).where(F.col("run_date") == F.current_date()).count(),
     SCHEMA_VERSION, "SUCCESS"),
    ("forecast_insights", latest_run, datetime.now(timezone.utc),
     int(insights_df.count()), SCHEMA_VERSION, "SUCCESS"),
]
log_df = spark.createDataFrame(
    log_rows,
    schema="dataset_name string, run_date date, published_at_utc timestamp, "
           "row_count long, schema_version string, status string",
)
(log_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(PUBLISH_LOG))
display(log_df)

## Cell 11 — Grant read access to the app service principal

The dashboard connects as `324a6ec7-...`; it cannot read a table until granted.

In [ ]:
# All tables/views the app needs access to:
# 1. Outputs written by this notebook
# 2. Upstream planning tables (BOQ, CBE, V5, V4, vintages, monitoring baselines)
SCHEMA_PREFIX = f"{CATALOG}.{SCHEMA}"

GRANT_TARGETS = [
    # --- Outputs of this notebook ---
    ("TABLE", RESULTS_TABLE),
    ("VIEW",  RESULTS_VIEW),
    ("TABLE", INSIGHTS_TABLE),
    ("TABLE", PUBLISH_LOG),
    ("VIEW",  RESULTS_V2_VIEW),
    ("VIEW",  LEADERBOARD_VIEW),
    ("VIEW",  INSIGHTS_V2_VIEW),
    ("TABLE", GRAPH_EXPLAIN_TABLE),
    ("TABLE", QUARTERLY_ACC_TABLE),
    # --- UCC upstream planning tables ---
    ("TABLE", f"{SCHEMA_PREFIX}.ucc_forecast_v5"),
    ("TABLE", f"{SCHEMA_PREFIX}.ucc_forecast_v4"),
    ("TABLE", f"{SCHEMA_PREFIX}.ucc_forecast_boq"),
    ("TABLE", f"{SCHEMA_PREFIX}.ucc_forecast_cbe"),
    ("TABLE", f"{SCHEMA_PREFIX}.ucc_forecast_vintages"),
    ("TABLE", f"{SCHEMA_PREFIX}.ucc_forecast_monitoring_baseline"),
    # --- ITSG upstream planning tables ---
    ("TABLE", f"{SCHEMA_PREFIX}.itsg_forecast_v5"),
    ("TABLE", f"{SCHEMA_PREFIX}.itsg_forecast_v4"),
    ("TABLE", f"{SCHEMA_PREFIX}.itsg_forecast_boq"),
    ("TABLE", f"{SCHEMA_PREFIX}.itsg_forecast_cbe"),
    ("TABLE", f"{SCHEMA_PREFIX}.itsg_forecast_monitoring_baseline"),
    # --- Scenario / Exec Mode tables ---
    ("TABLE", SCENARIO_GRID_TABLE),
    ("TABLE", SENSITIVITY_TABLE),
    ("TABLE", MODEL_SCENARIO_TABLE),
    ("TABLE", HIST_SCENARIO_TABLE),
    # --- ITSG app-facing output ---
    ("TABLE", f"{SCHEMA_PREFIX}.arr_forecast_app_latest"),
]

for kind, obj in GRANT_TARGETS:
    spark.sql(f"GRANT SELECT ON {kind} {obj} TO `{APP_SERVICE_PRINCIPAL}`")
    print(f"  GRANT SELECT on {kind} {obj}")
print(f"\nGranted SELECT on {len(GRANT_TARGETS)} objects to app service principal.")

## Cell 12 — Validation & summary

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPREHENSIVE VALIDATION
# ══════════════════════════════════════════════════════════════════════════════
# Validates:
#   1. Schema presence
#   2. Latest run_date
#   3. Row counts by forecast_type
#   4. Sample values for actuals, p05, p50, p95
#   5. No nulls in key dimensions
#   6. No duplicates on natural key
# ══════════════════════════════════════════════════════════════════════════════

# Refresh view schema — the underlying table may have had legacy columns dropped
# (zombie-column cleanup in step 7b), which invalidates the view's cached schema.
spark.sql(f"""
    CREATE OR REPLACE VIEW {RESULTS_VIEW} AS
    WITH latest AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY ds, product, sales_market, grain_level, model, forecast_type, forecast_view
                ORDER BY published_at_utc DESC
            ) AS _rn
        FROM {RESULTS_TABLE}
        WHERE run_date = (SELECT MAX(run_date) FROM {RESULTS_TABLE})
    )
    SELECT * EXCEPT (_rn) FROM latest WHERE _rn = 1
""")

r = spark.table(RESULTS_VIEW)
n = r.count()
products = sorted([row["product"] for row in r.select("product").distinct().collect()])
ftypes   = sorted([row["forecast_type"] for row in r.select("forecast_type").distinct().collect()])
models   = sorted([row["model"] for row in r.select("model").distinct().collect()])

# --- 1. Basic assertions ---
assert n > 0, "forecast_results_latest is empty."
assert r.where(F.col("p50").isNotNull()).count() > 0, "No non-null p50."
assert "Total" in products, "Total rollup missing."
assert any(ft in ftypes for ft in ("rolling", "roy", "forward_1yr")), "No forecast rows."
assert "actuals" in ftypes, "No actuals rows — check CBE/arr_forecast_v2 source."

# --- 2. Latest run_date ---
latest_rd = r.agg(F.max("run_date")).collect()[0][0]
print(f"Latest run_date: {latest_rd}")

# --- 3. Counts by forecast_type ---
print("\nRow counts by forecast_type:")
r.groupBy("forecast_type").count().orderBy("forecast_type").show(truncate=False)

# --- 4. Sample totals (Adaptive_Ensemble, Total product, latest week) ---
print("Sample values (Adaptive_Ensemble, Total product, rolling):")
sample = (r
    .where("model = 'Adaptive_Ensemble' AND product = 'Total' AND forecast_type = 'rolling'")
    .orderBy("ds")
    .select("ds", "actual", "p05", "p50", "p95", "most_likely")
    .limit(3))
sample.show(truncate=False)

# Verify most_likely == p50 exactly (no approximation)
ml_mismatch = r.where(
    (F.col("model") != "ACTUAL") &
    F.col("p50").isNotNull() &
    (F.col("most_likely") != F.col("p50"))
).count()
assert ml_mismatch == 0, f"FAIL: {ml_mismatch} rows where most_likely != p50 (expected EXACT equality)"
print(f"most_likely == p50 check: PASS (0 mismatches in {n} rows)")

# --- 5. Null check on key dimensions ---
KEY_DIMS = ["ds", "product", "sales_market", "model", "forecast_type", "forecast_view"]
null_counts = {}
for col_name in KEY_DIMS:
    nc = r.where(F.col(col_name).isNull()).count()
    null_counts[col_name] = nc
nulls_found = {k: v for k, v in null_counts.items() if v > 0}
if nulls_found:
    print(f"WARNING: Nulls in key dimensions: {nulls_found}")
else:
    print(f"Null check on key dimensions: PASS (0 nulls in {KEY_DIMS})")

# --- 6. Duplicate check on natural key ---
dedup_key = ["ds", "product", "sales_market", "grain_level", "model", "forecast_type", "forecast_view"]
total_rows = r.count()
distinct_keys = r.select(*dedup_key).distinct().count()
dupes = total_rows - distinct_keys
assert dupes == 0, f"FAIL: {dupes} duplicate rows on natural key {dedup_key}"
print(f"Duplicate check: PASS ({total_rows} rows, {distinct_keys} distinct keys, 0 dupes)")

# --- 7. Schema ---
print(f"\nSchema ({len(r.columns)} columns): {r.columns}")

# --- 7b. Legacy zombie-column check (2026-09-30) ---
# Delta's mergeSchema=true (used on the Cell 6b write) is additive-only: it
# can add new columns but never drops old ones. This table's schema included
# the full legacy p10/p20/p30/p40/p60/p70/p80/p90-by-10s grid before the
# 2026-09-28 conversion to the canonical P05/P25/P50/P75/P95 columns -- since
# then, every writer stopped POPULATING those columns, but they are still
# PRESENT on the physical Delta table as permanent all-NULL zombie columns
# (a `SELECT *` still returns them). This mirrors the same check Cell 3's
# app-table diagnostic (Check 3) already runs against arr_forecast_app_latest
# -- this table never had the equivalent check on itself until now.
_LEGACY_QUANTILE_COLS = ["p10", "p20", "p30", "p40", "p60", "p70", "p80", "p90"]
_legacy_present = [c for c in _LEGACY_QUANTILE_COLS if c in r.columns]
if _legacy_present:
    print(f"Legacy quantile columns still present on {RESULTS_TABLE} / {RESULTS_VIEW}: "
          f"{_legacy_present}. These are permanent all-NULL zombie columns left over from before "
          f"the 2026-09-28 P05-P95 conversion -- mergeSchema=true only adds columns, it never drops "
          f"them. Rebuilding the table now with only the canonical columns.")
    # 2026-09-30: actually remove them via CREATE OR REPLACE TABLE ... AS SELECT
    # (works on any Delta table, doesn't require enabling column mapping mode
    # the way ALTER TABLE ... DROP COLUMNS would). Rebuilds from the table's
    # own current contents, keeping only COLS (Cell 2's canonical 20-column
    # schema) and preserving the original partitioning (partitionBy("product")
    # in Cell 6b's write). Guarded by `if _legacy_present` so this only runs
    # the one time it's actually needed -- every subsequent day's run will see
    # the PASS branch below and skip straight past it at no cost. Wrapped in
    # try/except so a failure here (e.g. a permissions issue) logs and moves
    # on rather than crashing the rest of the publish, same pattern as this
    # notebook's other non-critical-failure handling (e.g. federated.sales).
    try:
        _select_list = ", ".join(COLS)
        spark.sql(f"""
            CREATE OR REPLACE TABLE {RESULTS_TABLE}
            PARTITIONED BY (product)
            AS SELECT {_select_list} FROM {RESULTS_TABLE}
        """)
        _post_cols = spark.table(RESULTS_TABLE).columns
        _still_present = [c for c in _LEGACY_QUANTILE_COLS if c in _post_cols]
        if _still_present:
            print(f"WARNING: rebuild ran but {_still_present} still present -- investigate manually.")
        else:
            print(f"Legacy quantile columns removed from {RESULTS_TABLE} "
                  f"({len(_post_cols)} columns remain, matching Cell 2's canonical COLS list).")
    except Exception as _exc:
        print(f"WARNING: legacy-column rebuild failed ({_exc}) -- table still has {_legacy_present} "
              f"present as zombie columns; will retry automatically on the next run.")
else:
    print(f"Legacy quantile-column check: PASS (none of {_LEGACY_QUANTILE_COLS} present on {RESULTS_TABLE})")

# --- Summary ---
print(f"\nforecast_results_latest: {n} rows")
print(f"  products      : {products}")
print(f"  forecast_types: {ftypes}")
print(f"  models        : {models}")
print(f"  WAPE attached : {r.where(F.col('model_wape').isNotNull()).count()} rows")
print(f"  insight scopes: {[row['product'] for row in spark.table(INSIGHTS_TABLE).where(F.col('run_date')==F.current_date()).select('product').collect()]}")
print('\nPublish complete — app reads EXACT values from forecast_results / forecast_insights.')

## Cell 13 — Scenario Analysis Engine

Creates four tables that power the improved Exec Mode scenario-analysis UI:

| Table | Purpose |
|-------|--------|
| `forecast_scenario_grid` | Named scenarios (Conservative/Baseline/Stretch) using real model quantiles, quarterly |
| `forecast_sensitivity` | Per-driver elasticity coefficients computed from historical pipeline data |
| `forecast_model_scenarios` | Maps each model to a scenario archetype with directional metadata |
| `forecast_historical_scenarios` | Quarterly pipeline metrics + actual outcomes for "what-if" validation |

The UI reads these to:
1. **One-click scenarios** — leadership picks Conservative/Baseline/Stretch and ALL tabs update
2. **Real sliders** — sensitivity coefficients ground the simulator in actual data relationships
3. **Model-as-scenario** — each model output becomes a named scenario for comparison
4. **Historical anchoring** — "When conditions looked like X, ARR was Y"

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SCENARIO ANALYSIS ENGINE — 4 tables for exec scenario analysis
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType

SCENARIO_GRID_TABLE = f"{CATALOG}.{SCHEMA}.forecast_scenario_grid"
SENSITIVITY_TABLE   = f"{CATALOG}.{SCHEMA}.forecast_sensitivity"
MODEL_SCENARIO_TABLE = f"{CATALOG}.{SCHEMA}.forecast_model_scenarios"
HIST_SCENARIO_TABLE  = f"{CATALOG}.{SCHEMA}.forecast_historical_scenarios"

# ─── 1. NAMED SCENARIO GRID ─────────────────────────────────────────────────
# Pre-compute quarterly outcomes at 3 named scenarios from real model quantiles.
# Conservative = P05, Baseline = P50 (most_likely), Stretch = P95.
# (2026-09-28: widened from P10/P90 to the real P05/P95 edge quantiles, now
# that both products publish them natively.)
# Broken down by product, quarter, and sales_market.

def build_scenario_grid():
    """Build quarterly scenario grid from forecast_results (adaptive view)."""
    r = spark.table(RESULTS_VIEW)
    
    # Only forecast rows (not actuals) in the adaptive view, current FY
    fwd = r.where(
        "forecast_view = 'adaptive' AND forecast_type IN ('rolling', 'roy') "
        f"AND ds <= '{FY_END_DATE}' AND model != 'ACTUAL'"
    )
    
    # 2026-09-30 SAFETY GUARD: fiscal_quarter is grouped as "Q1".."Q4" with NO
    # year component (kept for app/dashboard compatibility -- changing that
    # format is a separate, bigger change). That is only safe as long as `fwd`
    # spans exactly one calendar year, which depends entirely on FY_END_DATE
    # being correct. A real run once had FY_END_DATE a full year too late (see
    # Cell 2's CURRENT_FY fix note) which silently merged 2026-Q1 and 2027-Q1
    # into one "Q1" bucket (weeks_in_quarter=26 instead of 13). Fail loudly
    # instead of silently double-counting if that ever recurs.
    _fwd_years = sorted({r["yr"] for r in fwd.select(F.year("ds").alias("yr")).distinct().collect()})
    assert len(_fwd_years) <= 1, (
        f"build_scenario_grid(): 'fwd' spans multiple calendar years {_fwd_years} -- "
        f"fiscal_quarter grouping (no year component) would silently merge quarters "
        f"across years. Check FY_END_DATE (currently {FY_END_DATE}) / CURRENT_FY "
        f"(currently {CURRENT_FY}) before proceeding."
    )

    # Assign fiscal quarter (GoTo FY = calendar year)
    fwd = fwd.withColumn("fiscal_quarter", F.concat(F.lit("Q"), F.quarter("ds").cast("string")))
    
    # Aggregate by (product, sales_market, quarter, model)
    # Then across models: use recommended_for_exec=1 (Adaptive_Ensemble) for named scenarios
    rec = fwd.where(f"model = '{REC_MODEL}'")
    
    # NOTE (2026-09-30): after the Cell 4 fix, `fwd` (and therefore `rec`) only
    # contains OPEN weeks (ds > today) -- closed weeks are no longer tagged
    # rolling/roy at all. So `weeks_in_quarter` here now means "open/forecast
    # weeks remaining in the quarter", not the quarter's full week count.
    # `actual_weeks` (below) is the complementary closed-week count; the two
    # sum to the quarter's true length.
    grid = rec.groupBy("product", "sales_market", "fiscal_quarter").agg(
        F.sum("p05").alias("conservative"),
        F.sum("p50").alias("baseline"),
        F.sum("p95").alias("stretch"),
        F.sum("most_likely").alias("most_likely"),
        F.count("*").alias("weeks_in_quarter"),
    )
    
    # Also compute actuals for closed quarters
    act = r.where("forecast_type = 'actuals' AND model = 'ACTUAL'")
    act_q = (act
        .withColumn("fiscal_quarter", F.concat(F.lit("Q"), F.quarter("ds").cast("string")))
        .where(f"YEAR(ds) = {CURRENT_FY}")
        .groupBy("product", "sales_market", "fiscal_quarter")
        .agg(F.sum("actual").alias("actual_to_date"), F.count("*").alias("actual_weeks"))
    )
    
    # 2026-09-30 FIX (FULL OUTER, not LEFT): once a quarter's every week has
    # closed (no rows left in `rec`/`grid` at all -- e.g. Q3 the day it ends),
    # a LEFT join from `grid` produced ZERO rows for that quarter, because
    # there was no base row on the left side to join actuals onto in the
    # first place. That silently dropped just-closed quarters from
    # forecast_scenario_grid entirely until the next quarter's open weeks
    # existed. FULL OUTER + coalescing both sides keeps a row for every
    # quarter that has EITHER open/forecast weeks OR closed/actual weeks.
    grid = grid.join(act_q, ["product", "sales_market", "fiscal_quarter"], "full")
    grid = (grid
        .withColumn("conservative",     F.coalesce(F.col("conservative"), F.lit(0.0)))
        .withColumn("baseline",         F.coalesce(F.col("baseline"), F.lit(0.0)))
        .withColumn("stretch",          F.coalesce(F.col("stretch"), F.lit(0.0)))
        .withColumn("most_likely",      F.coalesce(F.col("most_likely"), F.lit(0.0)))
        .withColumn("weeks_in_quarter", F.coalesce(F.col("weeks_in_quarter"), F.lit(0)))
    )
    
    # 2026-09-30 FIX: make conservative/baseline/stretch/most_likely PACE-AWARE
    # full-quarter totals instead of pure re-estimates of the whole quarter.
    # Previously these columns summed the model's own p05/p50/p95 across every
    # week the grid grouped in -- for a partially-elapsed quarter that included
    # the model's stale self-prediction for weeks that have SINCE closed,
    # rather than what was actually booked. `actual_to_date` was computed and
    # joined in but never blended into the headline numbers, so a partial
    # quarter's "baseline" could read lower than the real pace already banked
    # (the raw model under-predicts, which is the whole reason BOQ/CBE
    # overrides and period-end uplift exist elsewhere in this notebook). Now:
    # real actual-to-date (0 for a fully-open future quarter, the FULL
    # quarter total once every week has closed) + model quantile forecast
    # for any remaining OPEN weeks (0 once the quarter is fully closed) =
    # true full-quarter scenario total in every case.
    _actual_fill = F.coalesce(F.col("actual_to_date"), F.lit(0.0))
    grid = (grid
        .withColumn("conservative", F.col("conservative") + _actual_fill)
        .withColumn("baseline",     F.col("baseline") + _actual_fill)
        .withColumn("stretch",      F.col("stretch") + _actual_fill)
        .withColumn("most_likely",  F.col("most_likely") + _actual_fill)
    )
    
    # Add metadata
    grid = (grid
        .withColumn("run_date", F.current_date())
        .withColumn("fiscal_year", F.lit(CURRENT_FY))
        .withColumn("scenario_source", F.lit("model_quantiles"))
        .withColumn("source_model", F.lit(REC_MODEL))
    )
    
    return grid

scenario_grid = build_scenario_grid()
scenario_grid.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SCENARIO_GRID_TABLE)
print(f"1. Wrote {scenario_grid.count()} rows to {SCENARIO_GRID_TABLE}")
scenario_grid.show(truncate=False)

# ─── 2. SENSITIVITY COEFFICIENTS (Weekly grain, SFDC source) ────────────────
# Compute elasticity from WEEKLY pipeline data (trailing 52-week rolling window).
# Uses the same SFDC source as the V5 forecast for consistency.
# ~254-270 observations per product/market (vs old 15 quarterly obs).

def build_sensitivity_coefficients():
    """Compute per-driver elasticity from weekly SFDC pipeline -> ARR relationships.
    Uses scipy linregress for p-values; marks significance at p<0.10."""
    from scipy import stats as sp_stats

    SFDC = "datagroup.datalake_transform.cds_sfdc_opp_products_latest"
    weekly_stats = spark.sql(f"""
        SELECT
            product_group as product,
            sales_market,
            DATE_TRUNC('week', close_date) as ds,
            SUM(CASE WHEN is_won = TRUE AND is_closed = TRUE
                     AND UPPER(COALESCE(purchase_type_rollup, '')) = 'GROWTH'
                     AND sales_channel NOT IN ('Care', 'Sales Other')
                THEN total_price2_converted ELSE 0 END) as won_arr,
            COUNT(CASE WHEN is_won = TRUE AND is_closed = TRUE
                       AND UPPER(COALESCE(purchase_type_rollup, '')) = 'GROWTH'
                       AND sales_channel NOT IN ('Care', 'Sales Other')
                THEN 1 END) as won_deals,
            COUNT(CASE WHEN stage_name NOT IN ('Closed Won', 'Closed Lost')
                THEN 1 END) as open_opps,
            SUM(CASE WHEN stage_name NOT IN ('Closed Won', 'Closed Lost')
                THEN total_price2_converted ELSE 0 END) as open_pipeline,
            AVG(CASE WHEN is_won = TRUE AND is_closed = TRUE
                     AND UPPER(COALESCE(purchase_type_rollup, '')) = 'GROWTH'
                     AND sales_channel NOT IN ('Care', 'Sales Other')
                THEN total_price2_converted END) as avg_deal_size
        FROM {SFDC}
        WHERE product_group IN ('UCC', 'ITSG')
          AND sales_market IN ('NA', 'EMEA', 'APAC', 'LATAM')
          AND close_date >= '2021-01-01'
          AND close_date <= CURRENT_DATE()
        GROUP BY product_group, sales_market, DATE_TRUNC('week', close_date)
        HAVING SUM(CASE WHEN is_won = TRUE AND is_closed = TRUE
                        AND UPPER(COALESCE(purchase_type_rollup, '')) = 'GROWTH'
                        AND sales_channel NOT IN ('Care', 'Sales Other')
                   THEN total_price2_converted ELSE 0 END) > 0
    """).toPandas()

    # Compute derived metrics
    weekly_stats['win_rate_pct'] = (
        weekly_stats['won_deals'] * 100.0 /
        weekly_stats['open_opps'].replace(0, np.nan)
    )
    weekly_stats['coverage_x'] = (
        weekly_stats['open_pipeline'] /
        weekly_stats['won_arr'].replace(0, np.nan)
    )
    weekly_stats['pipeline_volume'] = weekly_stats['open_pipeline']

    results = []
    drivers = [
        ('win_rate',        'win_rate_pct',   'percentage_point', '1pp higher win rate'),
        ('deal_size',       'avg_deal_size',  'percent_change',   '1% larger deals'),
        ('coverage',        'coverage_x',     '0.1x_change',      '+0.1x coverage'),
        ('pipeline_volume', 'pipeline_volume', 'percent_change',  '1% more pipeline'),
    ]

    for product in weekly_stats['product'].unique():
        for market in weekly_stats['sales_market'].unique():
            subset = weekly_stats[
                (weekly_stats['product'] == product) &
                (weekly_stats['sales_market'] == market)
            ].dropna(subset=['won_arr']).sort_values('ds')

            # Trailing 52-week rolling window (most recent data)
            subset = subset.tail(52 * 5)  # up to 5 years
            if len(subset) < 30:
                continue

            won_arr = subset['won_arr'].values.astype(float)
            mean_arr = won_arr.mean()
            if mean_arr == 0:
                continue

            for driver_name, col_name, unit, desc_prefix in drivers:
                if col_name not in subset.columns:
                    continue
                x = subset[col_name].values.astype(float)
                valid = ~(np.isnan(x) | np.isinf(x) | np.isnan(won_arr))
                x_v, y_v = x[valid], won_arr[valid]
                if len(x_v) < 30 or np.std(x_v) == 0:
                    continue

                slope, intercept, r_value, p_value, std_err = sp_stats.linregress(x_v, y_v)
                corr = r_value

                # Elasticity: % change in ARR per unit driver change
                if unit == 'percent_change':
                    elasticity = (slope * (np.mean(x_v) * 0.01)) / mean_arr * 100
                elif unit == '0.1x_change':
                    elasticity = (slope * 0.1) / mean_arr * 100
                else:  # percentage_point
                    elasticity = (slope / mean_arr) * 100

                results.append({
                    'product': product, 'sales_market': market,
                    'driver': driver_name, 'driver_unit': unit,
                    'baseline_value': round(float(np.mean(x_v)), 2),
                    'elasticity_pct': round(float(elasticity), 3),
                    'correlation': round(float(corr), 3),
                    'n_observations': int(len(x_v)),
                    'description': f'{desc_prefix} -> {elasticity:+.2f}% ARR (p={p_value:.4f})'
                })

    import pandas as pd
    sens_pdf = pd.DataFrame(results)
    sens_pdf['run_date'] = pd.Timestamp.now().date()
    return spark.createDataFrame(sens_pdf)

sens_df = build_sensitivity_coefficients()
sens_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SENSITIVITY_TABLE)
print(f"2. Wrote {sens_df.count()} rows to {SENSITIVITY_TABLE}")
sens_df.select("product", "sales_market", "driver", "elasticity_pct", "correlation", "n_observations").show(30, truncate=False)


# ─── 3. MODEL-AS-SCENARIO MAPPING ───────────────────────────────────────────
# Map individual models to named scenario archetypes based on their behavior.
# Uses backtest/holdout bias direction to classify as conservative/neutral/stretch.

def build_model_scenarios():
    """Classify each model as a scenario archetype based on its bias profile."""
    # Get model performance from monitoring baselines
    # UCC baseline uses holdout_wape_baseline/cv_wape_baseline (no metric_scope)
    # Deduplicate to one row per model (take latest by run_timestamp_utc)
    w_ucc = Window.partitionBy("model").orderBy(F.desc("run_timestamp_utc"))
    ucc_bl = (spark.table(UCC_BASELINE)
        .withColumn("_rn", F.row_number().over(w_ucc))
        .where("_rn = 1").drop("_rn")
        .select(
            F.lit("UCC").alias("product"),
            F.col("model"),
            F.col("holdout_wape_baseline").cast("double").alias("wape"),
            F.col("cv_wape_baseline").cast("double").alias("cv_wape"),
        )
    )
    
    # ITSG baseline has wape/mape/metric_scope — dedup same way
    w_itsg = Window.partitionBy("model").orderBy(F.desc("run_timestamp_utc"))
    itsg_bl = (spark.table(ITSG_BASELINE)
        .withColumn("_rn", F.row_number().over(w_itsg))
        .where("_rn = 1").drop("_rn")
        .select(
            F.lit("ITSG").alias("product"),
            F.col("model"),
            F.col("holdout_wape_baseline").cast("double").alias("wape"),
            F.col("cv_wape_baseline").cast("double").alias("cv_wape"),
        )
    )
    
    baselines = ucc_bl.unionByName(itsg_bl, allowMissingColumns=True)
    
    # Also look at actual FY2026 sum by model from forecast_results
    model_totals = (spark.table(RESULTS_VIEW)
        .where(f"forecast_view='adaptive' AND forecast_type IN ('rolling','roy') AND ds <= '{FY_END_DATE}' AND model != 'ACTUAL' AND sales_market = 'Total' AND grain_level = 'total'")
        .groupBy("product", "model")
        .agg(F.sum("p50").alias("fy_forecast_p50"))
    )
    # Total-product rows carry one partial row per per-product LGB name (Global_LGB_Q50_UCC / _ITSG),
    # each covering only that product. Merge them into one full-Total LGB row so pct_vs_ensemble
    # compares like with like. UCC/ITSG rows keep their names (needed for the baseline join).
    model_totals = (model_totals
        .withColumn("model", F.when((F.col("product") == "Total") & F.col("model").startswith("Global_LGB_Q50"),
                                    F.lit("Global_LGB_Q50")).otherwise(F.col("model")))
        .groupBy("product", "model")
        .agg(F.sum("fy_forecast_p50").alias("fy_forecast_p50"))
    )
    
    # Get the ensemble (baseline) total per product
    ensemble_totals = model_totals.where(f"model = '{REC_MODEL}'").select(
        F.col("product"), F.col("fy_forecast_p50").alias("ensemble_p50")
    )
    
    # Join and compute relative positioning
    model_pos = model_totals.join(ensemble_totals, "product", "left")
    model_pos = model_pos.withColumn(
        "pct_vs_ensemble",
        F.round((F.col("fy_forecast_p50") - F.col("ensemble_p50")) / F.col("ensemble_p50") * 100, 2)
    )
    
    # Classify: <-5% = conservative, >+5% = stretch, else = neutral/baseline
    model_pos = model_pos.withColumn(
        "scenario_archetype",
        F.when(F.col("pct_vs_ensemble") < -5, F.lit("conservative"))
         .when(F.col("pct_vs_ensemble") > 5, F.lit("stretch"))
         .otherwise(F.lit("baseline"))
    )
    
    # Join accuracy
    result = model_pos.join(baselines, ["product", "model"], "left")
    result = (result
        .withColumn("run_date", F.current_date())
        .withColumn("description", 
            F.when(F.col("scenario_archetype") == "conservative", 
                   F.concat(F.lit("Tends "), F.round(F.abs(F.col("pct_vs_ensemble")), 1).cast("string"), F.lit("% below ensemble — use for downside planning")))
             .when(F.col("scenario_archetype") == "stretch",
                   F.concat(F.lit("Tends "), F.round(F.col("pct_vs_ensemble"), 1).cast("string"), F.lit("% above ensemble — use for stretch/upside targets")))
             .otherwise(F.lit("Aligned with ensemble baseline — primary planning model"))
        )
    )
    return result

model_scenarios = build_model_scenarios()
model_scenarios.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(MODEL_SCENARIO_TABLE)
print(f"\n3. Wrote {model_scenarios.count()} rows to {MODEL_SCENARIO_TABLE}")
model_scenarios.select("product", "model", "scenario_archetype", "pct_vs_ensemble", "wape").orderBy("product", "pct_vs_ensemble").show(truncate=False)

# ─── 4. HISTORICAL SCENARIOS (WHAT-IF TABLE) ────────────────────────────────
# Quarterly pipeline conditions alongside actual ARR outcomes for anchoring.
# "When win rate was 35% and coverage was 3.2x, we closed $X"

def build_historical_scenarios():
    """Build quarterly history of pipeline conditions vs actual outcomes."""
    hist = spark.sql(f"""
        SELECT 
            product_group as product,
            CONCAT(YEAR(dte), '-Q', QUARTER(dte)) as period,
            YEAR(dte) as fiscal_year,
            CONCAT('Q', QUARTER(dte)) as fiscal_quarter,
            sales_market,
            ROUND(SUM(won_amount), 2) as actual_arr,
            ROUND(SUM(opened_amount), 2) as pipeline_created,
            CAST(SUM(won_opps) AS INT) as deals_won,
            CAST(SUM(opened_opps) AS INT) as deals_opened,
            ROUND(SUM(won_amount) / NULLIF(SUM(won_opps), 0), 2) as avg_deal_size,
            ROUND(SUM(won_opps) * 100.0 / NULLIF(SUM(opened_opps), 0), 2) as win_rate_pct,
            ROUND(SUM(opened_amount) / NULLIF(SUM(won_amount), 0), 2) as coverage_x,
            -- QoQ growth for context
            LAG(SUM(won_amount)) OVER (
                PARTITION BY product_group, sales_market 
                ORDER BY YEAR(dte), QUARTER(dte)
            ) as prev_q_arr
        FROM datagroup_mdl.mdl_sales_analytics.kpi_pacing_sfdc
        WHERE product_grain = 'genus'
          AND product_group IN ('UCC', 'ITSG')
          AND purchase_type IN ('New', 'Add-On')
          AND sales_market IN ('NA', 'EMEA', 'APAC', 'LATAM')
          AND YEAR(dte) BETWEEN 2023 AND {CURRENT_FY}
        GROUP BY product_group, YEAR(dte), QUARTER(dte), sales_market
        HAVING SUM(won_amount) > 0 AND SUM(opened_opps) > 10
    """)
    
    # Add QoQ growth rate
    hist = hist.withColumn(
        "qoq_growth_pct",
        F.round((F.col("actual_arr") - F.col("prev_q_arr")) / F.col("prev_q_arr") * 100, 1)
    ).drop("prev_q_arr")
    
    # Add Total product = UCC + ITSG rollup
    total = (hist
        .groupBy("period", "fiscal_year", "fiscal_quarter", "sales_market")
        .agg(
            F.sum("actual_arr").alias("actual_arr"),
            F.sum("pipeline_created").alias("pipeline_created"),
            F.sum("deals_won").alias("deals_won"),
            F.sum("deals_opened").alias("deals_opened"),
            F.round(F.sum("actual_arr") / F.sum("deals_won"), 2).alias("avg_deal_size"),
            F.round(F.sum("deals_won") * 100.0 / F.sum("deals_opened"), 2).alias("win_rate_pct"),
            F.round(F.sum("pipeline_created") / F.sum("actual_arr"), 2).alias("coverage_x"),
        )
        .withColumn("product", F.lit("Total"))
        .withColumn("qoq_growth_pct", F.lit(None).cast("double"))
    )
    
    combined = hist.unionByName(total, allowMissingColumns=True)
    combined = combined.withColumn("run_date", F.current_date())
    return combined

hist_scenarios = build_historical_scenarios()
hist_scenarios.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(HIST_SCENARIO_TABLE)
print(f"\n4. Wrote {hist_scenarios.count()} rows to {HIST_SCENARIO_TABLE}")
hist_scenarios.where("product = 'UCC' AND sales_market = 'NA'").orderBy("period").show(truncate=False)

print("\n" + "="*70)
print("SCENARIO ANALYSIS ENGINE COMPLETE")
print(f"  {SCENARIO_GRID_TABLE}: Named scenario outcomes (Conservative/Baseline/Stretch)")
print(f"  {SENSITIVITY_TABLE}: Per-driver elasticity coefficients")
print(f"  {MODEL_SCENARIO_TABLE}: Model-to-scenario archetype mapping")
print(f"  {HIST_SCENARIO_TABLE}: Historical what-if anchoring data")
print("="*70)

## Cell 14 — Unified All-Models Table for Power BI

Creates `forecast_all_models` — a single table that combines:
- **Prophet** results from `forecast_prophet` (native P05/P25/P50/P75/P95 quantiles)
- **All other models** (MSTL, DHR_ARIMA, ETS, LightGBM, Adaptive_Ensemble) from `forecast_results`

### Quantile Handling (non-Prophet models) — updated 2026-09-30

**Stale as of 2026-09-28:** the P10/P90-interpolation formula below is no longer
what Cell 40 does. `forecast_results` now natively carries real P05/P25/P50/P75/P95
for every model (the legacy P10/P90 grid was removed everywhere upstream), so Cell 40
prefers those real values directly — no interpolation needed in the normal case.

Interpolation only happens as a **fallback**, and it is WAPE-based, not a P10/P90
normal-error spread (that column pair no longer exists to derive sigma from):
- σ = model_wape / 100 × |P50|  (per-model holdout WAPE from the monitoring baseline)
- P05 = P50 − 1.6449 × σ, P25 = P50 − 0.6745 × σ, P75 = P50 + 0.6745 × σ, P95 = P50 + 1.6449 × σ

Prophet rows keep their EXACT native quantiles (no interpolation, same as before).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FORECAST_ALL_MODELS — Unified table for Power BI (all models in one place)
# ══════════════════════════════════════════════════════════════════════════════
# Combines:
#   1. Prophet results from forecast_prophet (native 5-quantile fan)
#   2. All other models from forecast_results (real p05/p25/p50/p75/p95, WAPE-derived fallback)
# Actuals & driver features joined model-independently from forecast_prophet.
# ══════════════════════════════════════════════════════════════════════════════

ALL_MODELS_TABLE = f"{CATALOG}.{SCHEMA}.forecast_all_models"

# Z_05 / Z_25 now come from the shared config cell (Cell 2) — do not
# redefine them here, to avoid drift between the two locations. (There is no
# Z_10: that was the old P10/P90 z-score, removed along with the legacy grid.)

# ─── 1. PROPHET ROWS (native quantiles from forecast_prophet) ────────────────

prophet_src = spark.table(f"{CATALOG}.{SCHEMA}.forecast_prophet")

prophet_rows = (prophet_src
    .select(
        F.col("ds").cast("date").alias("ds"),
        F.col("product"),
        F.when(F.upper(F.col("sales_market")) == "UNKNOWN", F.lit("Unknown"))
         .otherwise(F.col("sales_market")).alias("sales_market"),
        F.lit("Prophet").alias("model_name"),
        F.col("year"),
        F.col("month"),
        F.quarter("ds").alias("quarter"),
        F.col("week_number"),
        F.weekofyear("ds").alias("iso_week"),
        F.year(F.col("ds") + F.expr("INTERVAL 3 DAYS")).alias("iso_year"),
        F.col("summer").cast("int").alias("summer"),
        F.col("winter").cast("int").alias("winter"),
        F.col("isweek1").cast("int").alias("isweek1"),
        F.col("Forecast_P05").alias("forecast_p05"),
        F.col("Forecast_P25").alias("forecast_p25"),
        F.col("Forecast_Median").alias("forecast_p50"),
        F.col("Forecast_P75").alias("forecast_p75"),
        F.col("Forecast_P95").alias("forecast_p95"),
        F.when(F.col("Actuals") > 0, F.col("Actuals")).otherwise(F.lit(None).cast("double")).alias("actuals"),
        F.col("magna_carta_actuals"),
        F.col("miradore_plan"),
        F.col("avg_deal_size"),
        F.col("avg_sales_cycle"),
        F.col("all_created_arr"),
        F.col("total_partner_pipeline"),
        F.col("avg_dials"),
        F.col("avg_touches"),
        F.col("avg_push_counter"),
        F.col("number_of_opps"),
        F.col("headcount"),
        F.col("avg_opportunity_age"),
        F.lit("prophet_native").alias("interval_method"),
        F.lit(None).cast("double").alias("model_wape"),
        F.lit(True).alias("is_current"),
        F.when(F.col("Actuals").isNotNull() & (F.col("Actuals") > 0), F.lit("actuals"))
         .otherwise(F.lit("forecast")).alias("forecast_type"),
        F.col("pe_account_flag"),
        F.current_timestamp().alias("published_at_utc"),
    )
)

# ─── 1b. ADD 'Total' MARKET ROLLUP ──────────────────────────────────────────
# Removed pe_account_flag and published_at_utc from groupBy — they vary between
# markets for the same ds, causing split Total rows (16 instead of 13 per quarter).
prophet_total = (prophet_rows
    .groupBy("ds", "product", "model_name", "year", "month", "quarter",
             "week_number", "iso_week", "iso_year", "summer", "winter", "isweek1",
             "interval_method", "model_wape", "is_current", "forecast_type")
    .agg(
        F.sum("forecast_p05").alias("forecast_p05"),
        F.sum("forecast_p25").alias("forecast_p25"),
        F.sum("forecast_p50").alias("forecast_p50"),
        F.sum("forecast_p75").alias("forecast_p75"),
        F.sum("forecast_p95").alias("forecast_p95"),
        F.sum("actuals").alias("actuals"),
        F.sum("magna_carta_actuals").alias("magna_carta_actuals"),
        F.sum("miradore_plan").alias("miradore_plan"),
        F.avg("avg_deal_size").alias("avg_deal_size"),
        F.avg("avg_sales_cycle").alias("avg_sales_cycle"),
        F.sum("all_created_arr").alias("all_created_arr"),
        F.sum("total_partner_pipeline").alias("total_partner_pipeline"),
        F.avg("avg_dials").alias("avg_dials"),
        F.avg("avg_touches").alias("avg_touches"),
        F.avg("avg_push_counter").alias("avg_push_counter"),
        F.sum("number_of_opps").alias("number_of_opps"),
        F.sum("headcount").alias("headcount"),
        F.avg("avg_opportunity_age").alias("avg_opportunity_age"),
        F.first("pe_account_flag").alias("pe_account_flag"),
        F.first("published_at_utc").alias("published_at_utc"),
    )
    .withColumn("sales_market", F.lit("Total"))
)

prophet_rows = prophet_rows.unionByName(prophet_total, allowMissingColumns=True)
print(f"Prophet rows from forecast_prophet (with Total rollup): {prophet_rows.count()}")

# ─── 2. OTHER MODELS (from forecast_results, interpolated quantiles) ─────────

fr = spark.table(RESULTS_VIEW)
other_models = fr.where(
    "model != 'ACTUAL' AND model != 'Prophet_trend' AND forecast_view = 'adaptive'"
)

# ── Prefer REAL native quantiles; fall back to model_wape-derived sigma ────
# forecast_results now carries genuine p05/p25/p75/p95 for BOTH products
# natively (UCC's real empirical deciles, ITSG's real native P05/P25/P75/P95)
# — the legacy p10/p20/.../p90 grid this used to derive a Gaussian sigma from
# (p90-p10) has been removed entirely (2026-09-28), so that fallback path is
# no longer possible. In practice this fallback should now rarely if ever
# trigger, since both products always populate real edge quantiles; where a
# row genuinely lacks them, sigma is instead derived from that model's
# holdout WAPE (model_wape, as a % of p50) — a coarser but always-available
# substitute, rather than hard-failing on a missing p10/p90 column.
other_rows = (other_models
    .withColumn("_sigma", F.col("model_wape") / 100.0 * F.abs(F.col("p50")))
    .withColumn("_gauss_p05", F.col("p50") - Z_05 * F.col("_sigma"))
    .withColumn("_gauss_p25", F.col("p50") - Z_25 * F.col("_sigma"))
    .withColumn("_gauss_p75", F.col("p50") + Z_25 * F.col("_sigma"))
    .withColumn("_gauss_p95", F.col("p50") + Z_05 * F.col("_sigma"))
    .select(
        F.col("ds"),
        F.col("product"),
        F.col("sales_market"),
        F.col("model").alias("model_name"),
        F.year("ds").alias("year"),
        F.month("ds").alias("month"),
        F.quarter("ds").alias("quarter"),
        F.weekofyear("ds").alias("week_number"),
        F.weekofyear("ds").alias("iso_week"),
        F.year(F.col("ds") + F.expr("INTERVAL 3 DAYS")).alias("iso_year"),
        F.when(F.month("ds").isin(6, 7, 8), 1).otherwise(0).alias("summer"),
        F.when(F.month("ds").isin(12, 1, 2), 1).otherwise(0).alias("winter"),
        F.when(F.weekofyear("ds") == 1, 1).otherwise(0).alias("isweek1"),
        F.coalesce(F.col("p05"), F.col("_gauss_p05")).alias("forecast_p05"),
        F.coalesce(F.col("p25"), F.col("_gauss_p25")).alias("forecast_p25"),
        F.col("p50").alias("forecast_p50"),
        F.coalesce(F.col("p75"), F.col("_gauss_p75")).alias("forecast_p75"),
        F.coalesce(F.col("p95"), F.col("_gauss_p95")).alias("forecast_p95"),
        F.lit(None).cast("double").alias("actuals"),
        F.lit(None).cast("double").alias("magna_carta_actuals"),
        F.lit(None).cast("double").alias("miradore_plan"),
        F.lit(None).cast("double").alias("avg_deal_size"),
        F.lit(None).cast("double").alias("avg_sales_cycle"),
        F.lit(None).cast("double").alias("all_created_arr"),
        F.lit(None).cast("double").alias("total_partner_pipeline"),
        F.lit(None).cast("double").alias("avg_dials"),
        F.lit(None).cast("double").alias("avg_touches"),
        F.lit(None).cast("double").alias("avg_push_counter"),
        F.lit(None).cast("bigint").alias("number_of_opps"),
        F.lit(None).cast("bigint").alias("headcount"),
        F.lit(None).cast("double").alias("avg_opportunity_age"),
        # interval_method now records whether the REAL native quantile was
        # used or the Gaussian fallback was needed, so this is auditable:
        #   native_quantile        -> real p05/p25/p75/p95 all present, used as-is
        #   <method>_partial_native -> some real edge quantiles present, some
        #                              still Gaussian-derived (mixed row)
        #   quantile_regression / monte_carlo / analytic_ci (unchanged tags)
        #                          -> no real edge quantiles at all; pure
        #                             Gaussian sigma-derivation, same as before
        F.when(F.col("p05").isNotNull() & F.col("p25").isNotNull()
               & F.col("p75").isNotNull() & F.col("p95").isNotNull(),
               F.lit("native_quantile"))
         .when(F.col("p05").isNotNull() | F.col("p25").isNotNull()
               | F.col("p75").isNotNull() | F.col("p95").isNotNull(),
               F.concat(
                   F.when(F.col("model").contains("LGB"), F.lit("quantile_regression"))
                    .when(F.col("model") == "Adaptive_Ensemble", F.lit("monte_carlo"))
                    .otherwise(F.lit("analytic_ci")),
                   F.lit("_partial_native")))
         .when(F.col("model").contains("LGB"), F.lit("quantile_regression"))
         .when(F.col("model") == "Adaptive_Ensemble", F.lit("monte_carlo"))
         .otherwise(F.lit("analytic_ci")).alias("interval_method"),
        F.col("model_wape"),
        F.lit(True).alias("is_current"),
        F.col("forecast_type"),
        F.lit(None).cast("boolean").alias("pe_account_flag"),
        F.current_timestamp().alias("published_at_utc"),
    )
)

# Join actuals & driver features from forecast_prophet
driver_features = (prophet_src
    .select(
        F.col("ds").cast("date").alias("ds"),
        "product", "sales_market",
        F.col("Actuals").alias("_actuals"),
        F.col("magna_carta_actuals").alias("_magna_carta_actuals"),
        F.col("miradore_plan").alias("_miradore_plan"),
        F.col("avg_deal_size").alias("_avg_deal_size"),
        F.col("avg_sales_cycle").alias("_avg_sales_cycle"),
        F.col("all_created_arr").alias("_all_created_arr"),
        F.col("total_partner_pipeline").alias("_total_partner_pipeline"),
        F.col("avg_dials").alias("_avg_dials"),
        F.col("avg_touches").alias("_avg_touches"),
        F.col("avg_push_counter").alias("_avg_push_counter"),
        F.col("number_of_opps").alias("_number_of_opps"),
        F.col("headcount").alias("_headcount"),
        F.col("avg_opportunity_age").alias("_avg_opportunity_age"),
    )
)

other_rows_enriched = (other_rows
    .join(driver_features, on=["ds", "product", "sales_market"], how="left")
    .withColumn("actuals", F.coalesce(F.col("_actuals"), F.col("actuals")))
    .withColumn("magna_carta_actuals", F.coalesce(F.col("_magna_carta_actuals"), F.col("magna_carta_actuals")))
    .withColumn("miradore_plan", F.coalesce(F.col("_miradore_plan"), F.col("miradore_plan")))
    .withColumn("avg_deal_size", F.coalesce(F.col("_avg_deal_size"), F.col("avg_deal_size")))
    .withColumn("avg_sales_cycle", F.coalesce(F.col("_avg_sales_cycle"), F.col("avg_sales_cycle")))
    .withColumn("all_created_arr", F.coalesce(F.col("_all_created_arr"), F.col("all_created_arr")))
    .withColumn("total_partner_pipeline", F.coalesce(F.col("_total_partner_pipeline"), F.col("total_partner_pipeline")))
    .withColumn("avg_dials", F.coalesce(F.col("_avg_dials"), F.col("avg_dials")))
    .withColumn("avg_touches", F.coalesce(F.col("_avg_touches"), F.col("avg_touches")))
    .withColumn("avg_push_counter", F.coalesce(F.col("_avg_push_counter"), F.col("avg_push_counter")))
    .withColumn("number_of_opps", F.coalesce(F.col("_number_of_opps"), F.col("number_of_opps")))
    .withColumn("headcount", F.coalesce(F.col("_headcount"), F.col("headcount")))
    .withColumn("avg_opportunity_age", F.coalesce(F.col("_avg_opportunity_age"), F.col("avg_opportunity_age")))
)
cols_to_drop = [c for c in other_rows_enriched.columns if c.startswith("_")]
other_rows_enriched = other_rows_enriched.drop(*cols_to_drop)

print(f"Other model rows (enriched with actuals/features): {other_rows_enriched.count()}")

# ─── 3. UNION Prophet + Other Models ─────────────────────────────────────────

final_cols = prophet_rows.columns
for c in final_cols:
    if c not in other_rows_enriched.columns:
        other_rows_enriched = other_rows_enriched.withColumn(c, F.lit(None))

other_rows_enriched = other_rows_enriched.select(*final_cols)
unified = prophet_rows.unionByName(other_rows_enriched, allowMissingColumns=True)

print(f"\nUnified forecast_all_models: {unified.count()} rows")
print(f"  Models: {sorted([r[0] for r in unified.select('model_name').distinct().collect()])}")
print(f"  Products: {sorted([r[0] for r in unified.select('product').distinct().collect()])}")
print(f"  Date range: {unified.agg(F.min('ds')).collect()[0][0]} to {unified.agg(F.max('ds')).collect()[0][0]}")

# ─── 4. WRITE TABLE ──────────────────────────────────────────────────────────

(unified.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ALL_MODELS_TABLE)
)
print(f"\n✓ Written to {ALL_MODELS_TABLE}")

# ─── 5. QUANTILE MONOTONICITY VALIDATION ─────────────────────────────────────

validation = spark.table(ALL_MODELS_TABLE)
violations = validation.where(
    (F.col("forecast_p05") > F.col("forecast_p25")) |
    (F.col("forecast_p25") > F.col("forecast_p50")) |
    (F.col("forecast_p50") > F.col("forecast_p75")) |
    (F.col("forecast_p75") > F.col("forecast_p95"))
).count()

if violations > 0:
    print(f"  ⚠️  WARNING: {violations} rows violate quantile monotonicity")
else:
    print(f"  ✓ Quantile monotonicity: PASS")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GRANTS + COMPANION VIEWS
# ══════════════════════════════════════════════════════════════════════════════

CATALOG = "datagroup_mdl"
SCHEMA = "mdl_sales_analytics"
ALL_MODELS_TABLE = f"{CATALOG}.{SCHEMA}.forecast_all_models"
APP_SERVICE_PRINCIPAL = "324a6ec7-e988-42c7-8a7f-55465f5bea37"

spark.sql(f"GRANT SELECT ON TABLE {ALL_MODELS_TABLE} TO `{APP_SERVICE_PRINCIPAL}`")
print(f"✓ Granted SELECT on {ALL_MODELS_TABLE} to {APP_SERVICE_PRINCIPAL}")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.v_forecast_all_models_current AS
SELECT 
    ds, product, sales_market, model_name,
    year, month, quarter, week_number, iso_week, iso_year,
    summer, winter, isweek1,
    forecast_p05, forecast_p25, forecast_p50,
    forecast_p50 AS Forecast_Median,
    forecast_p75, forecast_p95,
    actuals AS Actuals,
    magna_carta_actuals, miradore_plan,
    avg_deal_size, avg_sales_cycle, all_created_arr,
    total_partner_pipeline, avg_dials, avg_touches,
    avg_push_counter, number_of_opps, headcount, avg_opportunity_age,
    interval_method, model_wape, forecast_type, pe_account_flag
FROM {ALL_MODELS_TABLE}
WHERE is_current = TRUE
""")
print(f"✓ Created view {CATALOG}.{SCHEMA}.v_forecast_all_models_current")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.v_forecast_model_comparison AS
SELECT
    ds, product, sales_market, model_name,
    forecast_p50,
    (forecast_p95 - forecast_p05) AS band_width_90,
    (forecast_p75 - forecast_p25) AS band_width_50,
    actuals,
    CASE WHEN actuals IS NOT NULL 
         THEN ABS(actuals - forecast_p50) / NULLIF(actuals, 0) * 100 
    END AS abs_pct_error,
    model_wape, interval_method
FROM {ALL_MODELS_TABLE}
WHERE is_current = TRUE AND forecast_type != 'actuals'
""")
print(f"✓ Created view {CATALOG}.{SCHEMA}.v_forecast_model_comparison")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.v_forecast_accuracy AS
SELECT
    ds, product, sales_market, model_name, actuals, forecast_p50,
    CASE WHEN actuals IS NOT NULL AND actuals != 0
         THEN ABS(actuals - forecast_p50) / ABS(actuals) * 100
    END AS ape,
    CASE WHEN actuals IS NOT NULL
         THEN (forecast_p50 - actuals)
    END AS signed_error,
    CASE WHEN actuals IS NOT NULL
         THEN CASE WHEN actuals BETWEEN forecast_p25 AND forecast_p75 THEN 1 ELSE 0 END
    END AS in_p25_p75,
    CASE WHEN actuals IS NOT NULL
         THEN CASE WHEN actuals BETWEEN forecast_p05 AND forecast_p95 THEN 1 ELSE 0 END
    END AS in_p05_p95,
    CASE WHEN actuals IS NOT NULL
         THEN 0.5 * ABS(actuals - forecast_p50)
    END AS pinball_p50,
    interval_method
FROM {ALL_MODELS_TABLE}
WHERE is_current = TRUE AND actuals IS NOT NULL
""")
print(f"✓ Created view {CATALOG}.{SCHEMA}.v_forecast_accuracy")

for vw in ['v_forecast_all_models_current', 'v_forecast_model_comparison', 'v_forecast_accuracy']:
    spark.sql(f"GRANT SELECT ON VIEW {CATALOG}.{SCHEMA}.{vw} TO `{APP_SERVICE_PRINCIPAL}`")
    print(f"  ✓ Granted SELECT on {vw}")

print(f"\n✓ All views created and grants applied.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PURE FORECAST PREDICTIONS — Jan 2026 → Sep 2027, full p05-p95
# ══════════════════════════════════════════════════════════════════════════════
# Single table: every week × every model × UCC + ITSG, with p05 through p95.
# Pure (STATIC) model predictions only -- this is THE source of the "static
# forecast" view: every closed week keeps its original model prediction,
# never overwritten by the actual. (Actuals live in the separate `actual`
# column here, and in `actuals`/`most_likely`/`blended_p50` downstream in
# PowerBI_Wide_Forecast_Export, which is the "dynamic" view: actual where
# closed, forecast where still open.)
# Updated each pipeline run — forward portion refreshes, historical is stable.
#
# 2026-09-30 FIX: this table's own "pure model predictions only" promise was
# silently broken for UCC. Section 3 (V5 CURRENT) read ucc_forecast_v5/
# itsg_forecast_v5 with no filter distinguishing genuine forward-forecast
# rows from actual-value rows. UCC's own v5 table blends real actuals into
# every quantile column for closed weeks (by design, for UCC's own CBE/pace
# views) -- reading it unfiltered meant UCC's "pure" p50 (and p05/p25/p75/p95)
# for every closed week silently became the actual value, for every model,
# all the way back to each fiscal year's start. That also meant there was no
# surviving static forecast anywhere downstream for UCC's closed weeks.
# Fixed by excluding source='actual' rows from the V5 read (itsg_forecast_v5
# has no such rows to begin with) and by computing the hindcast gap window's
# V5 boundary PER PRODUCT instead of from a single value shared across both
# products (see _v5_min_by_product below) -- the shared value, dragged down
# by UCC's early actual-collapsed rows, was also silently suppressing ITSG's
# hindcast generation for its Jul-Aug gap (both products were printing 0
# hindcast rows before this fix).
#
# SOURCES & BAND METHODS (approximate -- actual boundaries are now derived
# per-product from each table's genuine forward-forecast start, not fixed
# calendar dates):
#   1. Jan 5 – Jun 29 2026 (both products): backtest_results → sigma from CV error std
#   2. Jun 29 → each product's own genuine V5 start: vintage hindcast (or
#      backtest-bias fallback where no vintage exists) → sigma from backtest
#   3. Each product's genuine V5 start → Dec 28 2026: V5 tables → native p05-p95
#   4. Jan 4 – Sep 13 2027 : BOQ tables → native p05-p95
# ══════════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from datetime import date

CATALOG = "datagroup_mdl"
SCHEMA  = "mdl_sales_analytics"
PURE_TABLE = f"{CATALOG}.{SCHEMA}.forecast_pure_predictions"

# Normal Z-scores: p_q = p50 ± |z_q| * sigma
Z = {5: 1.6449, 25: 0.6745, 75: 0.6745, 95: 1.6449}

def _add_bands(df, sigma_col='_sigma'):
    """Compute p05/p25/p75/p95 from p50 and sigma (symmetric normal)."""
    for q in [5, 25]:
        df[f'p{q:02d}'] = df['p50'] - Z[q] * df[sigma_col]
    for q in [75, 95]:
        df[f'p{q:02d}'] = df['p50'] + Z[q] * df[sigma_col]
    return df

# ── 1. BACKTEST (Jan 5 – Jun 29 2026) — bands from CV error std ──────────
parts = []
for product, tbl in [('UCC', 'ucc_backtest_results'), ('ITSG', 'itsg_backtest_results')]:
    bt = spark.sql(f"""
        SELECT model, CAST(ds AS DATE) AS ds, forecast AS p50, actual,
               (forecast - actual) AS error
        FROM {CATALOG}.{SCHEMA}.{tbl}
        WHERE ds >= '2026-01-05'
    """).toPandas()
    # Rolling sigma — varies by season instead of flat constant
    bt = bt.sort_values(['model', 'ds'])
    bt['_sigma'] = bt.groupby('model')['error'].transform(
        lambda x: x.expanding(min_periods=4).std()
    )
    # Fill early weeks (< 4 obs) with model's overall std
    bt['_sigma'] = bt['_sigma'].fillna(bt.groupby('model')['error'].transform('std'))
    bt = _add_bands(bt)
    bt['product'] = product
    bt['source'] = 'backtest_cv'
    bt['bands_method'] = 'wape_reconstructed'
    parts.append(bt)
backtest = pd.concat(parts, ignore_index=True)
print(f"1. Backtest (Jan-Jun 2026): {len(backtest)} rows")

# ── 2. HINDCAST (gap weeks) — vintage weekly p50 + backtest sigma → bands ─
# The old approach read from forecast_all_models (cumulative QTD values —
# unusable for weekly predictions). Instead, read from VINTAGES which store
# correct weekly p50 for every model. Use per-model error sigma from the
# backtest period (step 1) to generate symmetric normal bands (p05-p95).
# This gives ALL models full quantile spreads, not just Ensemble & LGB.
_bt_max = backtest['ds'].max()  # end of backtest_cv coverage

# 2026-09-30 FIX: _v5_min was a single value shared across BOTH products,
# computed as MIN(ds) over each v5 table's ENTIRE contents with no filter on
# whether a row is a genuine forward forecast or an actual-value row. UCC's
# own v5 table (Step 14E-V5 in UCC_Forecast_Foundation_V5) deliberately
# blends actuals into weekly rows back to the start of the fiscal year
# (source='actual'), so UCC's raw MIN(ds) resolves to ~Jan of the current
# year -- far earlier than UCC's real forward-forecast start. Because this
# single _v5_min was then reused for ITSG's gap-detection window too
# (`ds > _bt_max AND ds < _v5_min`), ITSG's genuinely-open gap (its own v5
# table has no 'actual' rows at all and legitimately starts later, e.g.
# ~Sep) was being collapsed to nothing -- `ds < _v5_min` was true for almost
# no dates, so the hindcast step silently produced 0 rows for either product
# and both the vintage and fallback-estimate branches below never ran.
# Fix: compute _v5_min PER PRODUCT, and only from rows that are genuine
# forward forecasts (exclude any 'actual'-sourced row if the table has a
# source column distinguishing them -- ucc_forecast_v5 does, itsg_forecast_v5
# does not and its rows are all forward forecasts already).
def _genuine_v5_min(tbl):
    _full_tbl = f"{CATALOG}.{SCHEMA}.{tbl}"
    _has_source_col = "source" in spark.table(_full_tbl).columns
    _actual_filter = "AND (source IS NULL OR source != 'actual')" if _has_source_col else ""
    # 2026-10-01 FIX: this was missing the same grain='weekly' filter that
    # section 3's own V5 read applies just below. UCC's v5 table (Step 14E-V5)
    # is multi-grain -- weekly rows plus monthly/quarterly/yearly rollup rows
    # in the same table, distinguished only by a `grain` column -- and a
    # forward-looking yearly or quarterly rollup row can carry an anchor `ds`
    # at the START of that period (e.g. a FY2026 rollup row dated 2026-01-01)
    # even though it summarizes future weeks too. Without this filter, that
    # early-anchored rollup row won (as the "genuine" MIN(ds)) over UCC's real
    # weekly forward-forecast start, collapsing the hindcast gap-detection
    # window to nothing for UCC -- confirmed in a real run: UCC silently got
    # 0 hindcast rows and ended up 12 weeks short of ITSG/Total's coverage
    # (79 vs 91 weeks) because of exactly this. itsg_forecast_v5 has no
    # `grain` column (weekly-only table), so this is a no-op there, same as
    # the grain filter already is in section 3.
    _has_grain_col = "grain" in spark.table(_full_tbl).columns
    _grain_filter = "AND grain = 'weekly'" if _has_grain_col else ""
    _row = spark.sql(f"""
        SELECT MIN(ds) AS min_ds FROM {_full_tbl}
        WHERE run_date_utc = (SELECT MAX(run_date_utc) FROM {_full_tbl})
        {_actual_filter}
        {_grain_filter}
    """).collect()[0]
    return pd.Timestamp(_row['min_ds']) if _row['min_ds'] else pd.Timestamp('2099-01-01')

_v5_min_by_product = {
    'UCC': _genuine_v5_min('ucc_forecast_v5'),
    'ITSG': _genuine_v5_min('itsg_forecast_v5'),
}
print(f"  Genuine (forward-only) V5 start per product: "
      f"UCC={_v5_min_by_product['UCC'].date()}  ITSG={_v5_min_by_product['ITSG'].date()}")
# Kept for any other code in this cell that still refers to a single _v5_min
# (section 3 below now looks up its own product's value directly).
_v5_min = min(_v5_min_by_product.values())

hc_parts = []
for _hc_product, _hc_vint_tbl in [('UCC', 'ucc_forecast_vintages'), ('ITSG', 'itsg_forecast_vintages')]:
    # Per-model sigma from backtest errors (already computed in step 1)
    _bt_prod = backtest[backtest['product'] == _hc_product]
    _model_sigma = _bt_prod.groupby('model')['error'].std().to_dict()

    # 2026-09-30 FIX: use this product's own genuine forward-forecast start,
    # not the shared/global _v5_min (see fix above _v5_min_by_product).
    _v5_min_prod = _v5_min_by_product[_hc_product]

    # Earliest vintage prediction per (ds, model) for gap weeks
    _vint = spark.sql(f"""
        WITH ranked AS (
            SELECT CAST(ds AS DATE) AS ds, model, p50,
                   ROW_NUMBER() OVER (PARTITION BY ds, model ORDER BY run_date_utc ASC) AS rn
            FROM {CATALOG}.{SCHEMA}.{_hc_vint_tbl}
            WHERE grain_level = 'total'
              AND ds > '{_bt_max.strftime('%Y-%m-%d')}'
              AND ds < '{_v5_min_prod.strftime('%Y-%m-%d')}'
        )
        SELECT ds, model, p50 FROM ranked WHERE rn = 1
    """).toPandas()

    _ens_sigma = _model_sigma.get('Adaptive_Ensemble', 100_000)
    _vint_dates = set()

    if len(_vint) > 0:
        _vint['ds'] = pd.to_datetime(_vint['ds'])
        _vint['_sigma'] = _vint['model'].map(_model_sigma)
        _ens_sigma = _model_sigma.get('Adaptive_Ensemble', _vint['p50'].std() * 0.3)
        _vint['_sigma'] = _vint['_sigma'].fillna(_ens_sigma)
        _vint = _add_bands(_vint)
        _vint['product'] = _hc_product
        _vint['source'] = 'hindcast'
        _vint['bands_method'] = 'vintage_backtest_sigma'
        _vint['actual'] = np.nan
        hc_parts.append(_vint)
        _vint_dates = set(_vint['ds'].unique())
        print(f"  {_hc_product}: {len(_vint)} vintage hindcast rows ({_vint['model'].nunique()} models)")

    # ── Fallback: gap weeks WITHOUT vintage predictions ──────────────────
    # (e.g. ITSG vintages started late → Jul 6-Aug 17 missing)
    # Estimate each model's p50 from its backtest bias ratio × CBE actual.
    # bias_ratio = median(model_forecast / actual) over backtest period.
    _cbe_tbl = f"{CATALOG}.{SCHEMA}.{'ucc' if _hc_product == 'UCC' else 'itsg'}_forecast_cbe"
    _cbe_gap = spark.sql(f"""
        SELECT CAST(ds AS DATE) AS ds, p50 AS actual
        FROM {_cbe_tbl}
        WHERE model = 'Adaptive_Ensemble' AND source = 'actual'
          AND ds > '{_bt_max.strftime('%Y-%m-%d')}'
          AND ds < '{_v5_min_prod.strftime('%Y-%m-%d')}'
          AND run_date_utc = (SELECT MAX(run_date_utc) FROM {_cbe_tbl})
    """).toPandas()
    _cbe_gap['ds'] = pd.to_datetime(_cbe_gap['ds'])
    _cbe_missing = _cbe_gap[~_cbe_gap['ds'].isin(_vint_dates)]

    if len(_cbe_missing) > 0:
        # Per-model bias ratio from backtest: median(forecast / actual)
        _bt_with_act = _bt_prod[_bt_prod['actual'].notna() & (_bt_prod['actual'].abs() > 0)]
        _bias_ratios = {}
        for _mdl, _grp in _bt_with_act.groupby('model'):
            _bias_ratios[_mdl] = float((_grp['p50'] / _grp['actual']).median())

        _fb_rows = []
        for _, _miss in _cbe_missing.iterrows():
            for _mdl, _ratio in _bias_ratios.items():
                _fb_rows.append({
                    'ds': _miss['ds'], 'model': _mdl,
                    'p50': _miss['actual'] * _ratio,
                    '_sigma': _model_sigma.get(_mdl, _ens_sigma),
                })
        if _fb_rows:
            _fb_df = pd.DataFrame(_fb_rows)
            _fb_df = _add_bands(_fb_df)
            _fb_df['product'] = _hc_product
            _fb_df['source'] = 'hindcast'
            _fb_df['bands_method'] = 'backtest_bias_estimated'
            _fb_df['actual'] = np.nan
            hc_parts.append(_fb_df)
            print(f"  {_hc_product}: {len(_fb_df)} estimated rows for {len(_cbe_missing)} weeks without vintages")

if hc_parts:
    hc = pd.concat(hc_parts, ignore_index=True)
else:
    hc = pd.DataFrame(columns=['product', 'model', 'ds', 'p50', '_sigma'] +
                       ['p05', 'p25', 'p75', 'p95'] +
                       ['source', 'bands_method', 'actual'])
print(f"2. Hindcast (gap weeks from vintages): {len(hc)} rows")

# ── 3. V5 CURRENT (Sep 14+) — native p05-p95 ────────────────────────────
# 2026-09-28 CRITICAL FIX: ucc_forecast_v5 mixes weekly rows together with
# monthly/quarterly/yearly rollup rows in the same table (distinguished only
# by a `grain` column) -- reading it unfiltered double/triple/quadruple-
# counted the affected weeks in any quarterly sum (e.g. Q4 2026 summed to
# ~$28.5M instead of the correct ~$14M). Filter to grain='weekly' for any
# source table that has that column. itsg_forecast_v5 has no `grain` column
# (it only ever writes weekly rows), so this is checked per table.
v5_parts = []
for product, tbl in [('UCC', 'ucc_forecast_v5'), ('ITSG', 'itsg_forecast_v5')]:
    _full_tbl = f"{CATALOG}.{SCHEMA}.{tbl}"
    _has_grain_col = "grain" in spark.table(_full_tbl).columns
    _grain_filter = "AND grain = 'weekly'" if _has_grain_col else ""
    # 2026-09-30 FIX: forecast_pure_predictions is documented as "pure model
    # predictions only (no actuals in the quantile columns)" -- but UCC's own
    # v5 table (Step 14E-V5) blends real actuals into p05-p95 for closed
    # weeks (source='actual', by design, for UCC's own CBE/pace-tracking
    # use). Reading this table here with no filter meant every UCC closed
    # week's "pure" forecast was actually the actual value repeated across
    # all 5 quantiles for every model -- i.e. there was no surviving static
    # forecast anywhere downstream for UCC's closed weeks (PowerBI export's
    # Adaptive_Ensemble_P50 etc. exactly equalled actuals). Exclude
    # source='actual' rows so only genuine forward model predictions feed
    # the pure table; itsg_forecast_v5 has no source column (all its rows
    # are already forward-only), so the filter is a no-op there.
    _has_source_col = "source" in spark.table(_full_tbl).columns
    _actual_filter = "AND (source IS NULL OR source != 'actual')" if _has_source_col else ""
    df = spark.sql(f"""
        SELECT model, CAST(ds AS DATE) AS ds,
               p05, p25, p50, p75, p95
        FROM {_full_tbl}
        WHERE run_date_utc = (SELECT MAX(run_date_utc) FROM {_full_tbl})
          AND (UPPER(sales_market) IN ('ALL','TOTAL'))
          {_grain_filter}
          {_actual_filter}
    """).toPandas()
    df['product'] = product
    df['source'] = 'v5_current'
    df['bands_method'] = 'real'
    v5_parts.append(df)
v5 = pd.concat(v5_parts, ignore_index=True)
print(f"3. V5 forward (Sep 2026+): {len(v5)} rows")

# ── 4. BOQ (beyond V5 max ds) — native p05-p95 ──────────────────────────
boq_parts = []
for product, tbl in [('UCC', 'ucc_forecast_boq'), ('ITSG', 'itsg_forecast_boq')]:
    v5_max = v5[v5['product'] == product]['ds'].max()
    df = spark.sql(f"""
        SELECT model, CAST(ds AS DATE) AS ds,
               p05, p25, p50, p75, p95
        FROM {CATALOG}.{SCHEMA}.{tbl}
        WHERE source = 'forecast' AND ds > '{v5_max}' AND grain_level = 'total'
    """).toPandas()
    df['product'] = product
    df['source'] = 'boq_forward'
    df['bands_method'] = 'real'
    boq_parts.append(df)
boq = pd.concat(boq_parts, ignore_index=True)
print(f"4. BOQ forward (2027+): {len(boq)} rows")

# ── 5. UNION + DEDUPLICATE ───────────────────────────────────────────────
cols = ['product', 'model', 'ds',
        'p05', 'p25', 'p50', 'p75', 'p95',
        'source', 'bands_method']

# Add actual column where available
for df in [v5, boq]:
    df['actual'] = np.nan
for df in [hc]:
    pass  # already has actual

all_cols = cols + ['actual']
for df in [backtest, hc, v5, boq]:
    for c in all_cols:
        if c not in df.columns:
            df[c] = np.nan

unified = pd.concat([backtest[all_cols], hc[all_cols], v5[all_cols], boq[all_cols]],
                    ignore_index=True)
unified['ds'] = pd.to_datetime(unified['ds'])

# Priority: real > interpolated > wape_reconstructed
priority = {'real': 0, 'vintage_backtest_sigma': 1, 'interpolated_p05_p95': 1,
            'backtest_bias_estimated': 2, 'wape_reconstructed': 3}
unified['_prio'] = unified['bands_method'].map(priority)
unified = unified.sort_values(['product', 'model', 'ds', '_prio'])
unified = unified.drop_duplicates(subset=['product', 'model', 'ds'], keep='first')
unified = unified.drop(columns=['_prio'])

# ── 5a. COVERAGE-AWARE CALIBRATION ────────────────────────────────────────
# Two references per (product, model):
#   1. V5/BOQ native median band width (from real quantiles)
#   2. Empirical 80%-coverage width (from backtest errors)
# Use max(v5, empirical) — ensures:
#   - ITSG: V5 width dominates (already wide enough) → consistent with real bands
#   - UCC: empirical width dominates (V5 underestimates error) → correct coverage
_real = unified[(unified['bands_method'] == 'real') & (unified['source'] == 'v5_current')].copy()
_real['_bw'] = (_real['p95'] - _real['p05']) / _real['p50'].clip(lower=1)
v5_bw = _real.groupby(['product', 'model'])['_bw'].median()

# Empirical: band width needed for ~90% p05–p95 coverage on backtest actuals
bt_act = unified[unified['actual'].notna() & (unified['source'] == 'backtest_cv')]
emp_parts = []
for (prod, mdl), grp in bt_act.groupby(['product', 'model']):
    sigma_emp = (grp['actual'] - grp['p50']).std()
    avg_p50 = grp['p50'].clip(lower=1).mean()
    emp_parts.append({'product': prod, 'model': mdl,
                      '_emp_bw': 2 * 1.6449 * sigma_emp / avg_p50})
emp_bw = pd.DataFrame(emp_parts).set_index(['product', 'model'])['_emp_bw']

# Combine: max(v5, empirical) per (product, model)
ref_df = pd.DataFrame({'_v5': v5_bw, '_emp': emp_bw}).fillna(0)
ref_df['_ref_bw'] = ref_df[['_v5', '_emp']].max(axis=1)
ref_bw = ref_df['_ref_bw'].reset_index()

unified = unified.merge(ref_bw, on=['product', 'model'], how='left')
nr = (unified['bands_method'] != 'real') & unified['_ref_bw'].notna()
_curr_bw = (unified.loc[nr, 'p95'] - unified.loc[nr, 'p05']) / unified.loc[nr, 'p50'].clip(lower=1)
_scale = unified.loc[nr, '_ref_bw'] / _curr_bw.clip(lower=0.01)

for q in ['p05', 'p25', 'p75', 'p95']:
    delta = unified.loc[nr, q] - unified.loc[nr, 'p50']
    unified.loc[nr, q] = unified.loc[nr, 'p50'] + delta * _scale

unified.loc[nr & (unified['bands_method'] == 'wape_reconstructed'), 'bands_method'] = 'calibrated_wape'
unified.loc[nr & (unified['bands_method'] == 'interpolated_p05_p95'), 'bands_method'] = 'calibrated_interp'
unified.loc[nr & (unified['bands_method'] == 'vintage_backtest_sigma'), 'bands_method'] = 'calibrated_vintage'
unified.loc[nr & (unified['bands_method'] == 'backtest_bias_estimated'), 'bands_method'] = 'calibrated_bias_est'
unified = unified.drop(columns=['_ref_bw'])

for prod in ['UCC', 'ITSG']:
    key = (prod, 'Adaptive_Ensemble')
    v5v = ref_df.loc[key, '_v5'] if key in ref_df.index else 0
    empv = ref_df.loc[key, '_emp'] if key in ref_df.index else 0
    winner = 'empirical' if empv > v5v else 'V5 native'
    print(f"  {prod} Ensemble: V5={v5v:.1%} emp={empv:.1%} → {winner}")
print(f"5a. Calibrated {nr.sum()} rows (coverage-aware)")

# ── 5a-BOQ. CALIBRATE BOQ BANDS to match V5 band widths ─────────────────
# BOQ native bands are narrower than V5 calibrated bands (UCC 53% vs 79%,
# ITSG 60% vs 91%). Scale BOQ outward to match V5 per (product, model).
# Uses V5 median band width (not max with empirical) because BOQ is also
# a forward prediction — V5 conformal calibration is the right reference.
boq_mask = unified['source'] == 'boq_forward'
if boq_mask.any():
    _boq_calibrated = 0
    for (prod, mdl), grp in unified[boq_mask].groupby(['product', 'model']):
        key = (prod, mdl)
        if key not in ref_df.index:
            continue
        v5_target = ref_df.loc[key, '_v5']
        if v5_target <= 0:
            continue
        idx = grp.index
        curr_bw = (unified.loc[idx, 'p95'] - unified.loc[idx, 'p05']) / unified.loc[idx, 'p50'].clip(lower=1)
        scale = v5_target / curr_bw.clip(lower=0.01)
        scale = scale.clip(lower=1.0)  # only widen, never shrink
        for q in ['p05', 'p25', 'p75', 'p95']:
            delta = unified.loc[idx, q] - unified.loc[idx, 'p50']
            unified.loc[idx, q] = unified.loc[idx, 'p50'] + delta * scale
        _boq_calibrated += len(idx)
    unified.loc[boq_mask, 'bands_method'] = 'calibrated_boq'
    # Report
    _boq_cal = unified[unified['source'] == 'boq_forward']
    for prod in ['UCC', 'ITSG']:
        _em = _boq_cal[((_boq_cal['product'] == prod) & (_boq_cal['model'] == 'Adaptive_Ensemble'))]
        if not _em.empty:
            _bw = ((_em['p95'] - _em['p05']) / _em['p50'].clip(lower=1)).mean()
            print(f"  {prod} BOQ Ensemble: calibrated bw = {_bw:.1%}")
    print(f"5a-BOQ. Calibrated {_boq_calibrated} BOQ rows to match V5 band widths")

# ── 5a-2. BLENDED P50 (actual where closed, forecast where future) ──────
# FY totals from blended_p50 match the dynamic table exactly
unified['blended_p50'] = unified['actual'].combine_first(unified['p50'])

# ── 5b. TOTAL ROLLUP (UCC + ITSG summed per model × week) ────────────────
# LGB product-specific names → common name for Total
LGB_MAP = {'Global_LGB_Q50_UCC': 'Global_LGB_Q50', 'Global_LGB_Q50_ITSG': 'Global_LGB_Q50'}
tmp = unified.copy()
tmp['_model_key'] = tmp['model'].replace(LGB_MAP)

q_and_actual = ['p05', 'p25', 'p50', 'p75', 'p95', 'actual', 'blended_p50']
src_prio = {'real': 0, 'calibrated_boq': 0, 'calibrated_interp': 1, 'interpolated_p05_p95': 1,
            'calibrated_vintage': 1, 'vintage_backtest_sigma': 1,
            'calibrated_bias_est': 2, 'backtest_bias_estimated': 2,
            'calibrated_wape': 3, 'wape_reconstructed': 3}
src_rev  = {v: k for k, v in src_prio.items()}

total_rows = []
for (mk, ds), grp in tmp.groupby(['_model_key', 'ds']):
    # 2026-09-30 fix: previously required nunique()>=2 (both UCC AND ITSG
    # present for this model/week) before building a Total row at all --
    # any week where one product's source table didn't cover that date
    # (backtest/hindcast/v5/boq windows differ per product) silently
    # dropped from Total's static baseline entirely, undercounting the
    # cross-check's "static" FY total by the missing product's full
    # contribution for those weeks (confirmed: components summed to more
    # than the reported Total row). Total = UCC + ITSG is defined as an
    # exact sum (see Cell 16 rollup note) -- sum whatever product(s) are
    # actually present rather than requiring both.
    row = {'product': 'Total', 'model': mk, 'ds': ds, 'source': 'total_rollup'}
    for c in q_and_actual:
        vals = grp[c].dropna()
        row[c] = vals.sum() if len(vals) > 0 else np.nan
    # bands_method = worst quality of the two components
    mapped = grp['bands_method'].map(src_prio)
    if mapped.isna().any():
        # Unknown bands_method — use worst priority
        worst_val = max(src_prio.values()) + 1
    else:
        worst_val = int(mapped.max())
    # Pick the actual bands_method string from the row with worst priority
    row['bands_method'] = grp.loc[mapped.idxmax(), 'bands_method'] if not mapped.isna().any() else grp['bands_method'].iloc[0]
    total_rows.append(row)

total_df = pd.DataFrame(total_rows)
unified = pd.concat([unified, total_df], ignore_index=True)
print(f"5b. Total rollup: {len(total_df)} rows added ({unified['ds'].nunique()} weeks)")

# Floor all quantiles at zero — ARR cannot be negative
q_cols = ['p05', 'p25', 'p50', 'p75', 'p95']
unified[q_cols] = unified[q_cols].clip(lower=0)

unified['run_date'] = date.today()

print(f"\n{'='*70}")
print(f"PURE FORECAST PREDICTIONS: {len(unified)} rows")
print(f"  Products: {sorted(unified['product'].unique())}")
print(f"  Models:   {sorted(unified['model'].unique())}")
print(f"  Period:   {unified['ds'].min().date()} → {unified['ds'].max().date()}")
print(f"  Weeks:    {unified['ds'].nunique()}")
print(f"  Per product: {unified.groupby('product')['ds'].nunique().to_dict()}")
print(f"\n  By source:")
for src, cnt in unified.groupby('source').size().items():
    print(f"    {src}: {cnt}")
print(f"  By bands_method:")
for m, cnt in unified.groupby('bands_method').size().items():
    print(f"    {m}: {cnt}")

# Verify no nulls in quantiles
null_check = unified[['p05','p25','p50','p75','p95','blended_p50']].isnull().sum()
print(f"\n  Null check: {null_check.to_dict()}")
print(f"{'='*70}")

# ── 6. WRITE TO DELTA ────────────────────────────────────────────────────
sdf = spark.createDataFrame(unified)
sdf.write.format('delta').mode('overwrite').option('overwriteSchema', 'true') \
    .saveAsTable(PURE_TABLE)
print(f"\n✓ Written to: {PURE_TABLE}")

# Grant to app SP
spark.sql(f"GRANT SELECT ON TABLE {PURE_TABLE} TO `324a6ec7-e988-42c7-8a7f-55465f5bea37`")
print(f"✓ Granted SELECT to app service principal")

# ── 7. SUMMARY TABLE ─────────────────────────────────────────────────────
print("\n--- Coverage by product × source (Ensemble only) ---")
summary = (unified[unified['model'] == 'Adaptive_Ensemble']
    .groupby(['product', 'source', 'bands_method'])
    .agg(weeks=('ds', 'nunique'),
         from_ds=('ds', lambda x: x.min().date()),
         to_ds=('ds', lambda x: x.max().date()),
         p50_sum_M=('p50', lambda x: round(x.sum()/1e6, 2)))
    .reset_index())
display(spark.createDataFrame(summary))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# APP SNAPSHOT V2 — Rebuilt from pure predictions + v2compat history
# ══════════════════════════════════════════════════════════════════════════════
# MOVED here from Cell 15 to fix stale-data bug: the snapshot must read
# forecast_pure_predictions AFTER Cell 16 writes it (not before).
#
# The app backend (forecast_v2_impl.py) reads from this table.
# Architecture:
#   Pre-2026: v2compat actuals (historical)
#   2026+:    forecast_pure_predictions (calibrated bands, all models)
# ══════════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from datetime import date

CATALOG = "datagroup_mdl"
SCHEMA  = "mdl_sales_analytics"
PURE    = f"{CATALOG}.{SCHEMA}.forecast_pure_predictions"
V2C     = f"{CATALOG}.{SCHEMA}.forecast_results_v2compat"
SNAP    = f"{CATALOG}.{SCHEMA}.forecast_app_snapshot"
APP_SP  = "324a6ec7-e988-42c7-8a7f-55465f5bea37"
RUN     = date.today()
CUTOFF  = pd.Timestamp('2026-01-05')

# ═══ A. 2026+ rows from FRESHLY-WRITTEN pure predictions ═════════════════
raw = spark.table(PURE).toPandas()
raw['ds'] = pd.to_datetime(raw['ds'])
print(f"Pure predictions (fresh): {len(raw)} rows, {raw['ds'].nunique()} weeks")

ens   = raw[raw['model'] == 'Adaptive_Ensemble'].copy()
indiv = raw[raw['model'] != 'Adaptive_Ensemble'].copy()

# Scaffold from Ensemble
s = ens[['product', 'ds', 'p05', 'p25', 'p50', 'p75', 'p95',
         'actual', 'blended_p50', 'source', 'bands_method']].copy()

s['Most_Likely']   = s['blended_p50']
s['Actuals']       = s['actual']
# Worst_Case/Best_Case aligned to P25/P75 (2026-09-28) to match the
# PowerBI_Wide_Forecast_Export notebook's worst_case/best_case definition
# (Adaptive_Ensemble_P25 / Adaptive_Ensemble_P75) — that is what's already
# live in Power BI, so both consumers now show the same numbers.
# forecast_pure_predictions now carries real, native p05/p25/p50/p75/p95
# (legacy by-10s grid removed 2026-09-28), so no interpolation is needed.
s['Worst_Case']    = s['p25']
s['Best_Case']     = s['p75']
s['forecast_type'] = np.where(
    s['actual'].notna(), 'actuals',
    np.where(s['source'] == 'boq_forward', 'boq_forward', 'forecast'))

# Propagate boq_forward to Total product
_boq_dates = set(s.loc[
    s['product'].isin(['UCC', 'ITSG']) & (s['forecast_type'] == 'boq_forward'), 'ds'])
_total_boq = (s['product'] == 'Total') & s['ds'].isin(_boq_dates) & (s['forecast_type'] != 'actuals')
s.loc[_total_boq, 'forecast_type'] = 'boq_forward'

s['forecast_view'] = 'adaptive'
s['sales_market']  = 'Total'
s['run_date']      = RUN

# ── Pivot non-LGB models → arr_* / mape_* ───────────────────────────────
_MODELS = {
    'Prophet_trend': 'prophet',
    'MSTL_v2':       'mstl_v2',
    'ETS':           'ets',
    'DHR_ARIMA':     'dhr_arima',
}
for mname, key in _MODELS.items():
    acol, mcol = f'arr_{key}', f'mape_{key}'
    mx = indiv[indiv['model'] == mname][['product', 'ds', 'p50', 'actual']].copy()
    mx[acol] = mx['p50']
    mx[mcol] = np.where(
        mx['actual'].notna() & (mx['actual'].abs() > 0),
        (mx['p50'] - mx['actual']).abs() / mx['actual'].abs() * 100, np.nan)
    s = s.merge(mx[['product', 'ds', acol, mcol]], on=['product', 'ds'], how='left')

# ── Pivot LGB (product-specific names → single arr_lightgbm column) ─────
lgb_names = [m for m in indiv['model'].unique() if 'LGB' in m]
if lgb_names:
    lgb = indiv[indiv['model'].isin(lgb_names)][['product', 'ds', 'p50', 'actual']].copy()
    lgb['arr_lightgbm'] = lgb['p50']
    lgb['mape_lightgbm'] = np.where(
        lgb['actual'].notna() & (lgb['actual'].abs() > 0),
        (lgb['p50'] - lgb['actual']).abs() / lgb['actual'].abs() * 100, np.nan)
    lgb = lgb[['product', 'ds', 'arr_lightgbm', 'mape_lightgbm']]
    lgb = lgb.drop_duplicates(['product', 'ds'], keep='first')
    s = s.merge(lgb, on=['product', 'ds'], how='left')
else:
    s['arr_lightgbm'] = np.nan
    s['mape_lightgbm'] = np.nan

# ── FIX #10: fill forward MAPE with closed-week mean (app badges) ──────
for key in ['prophet', 'mstl_v2', 'ets', 'dhr_arima', 'lightgbm']:
    mc = f'mape_{key}'
    if mc not in s.columns:
        continue
    for prod in s['product'].unique():
        closed = (s['product'] == prod) & s[mc].notna()
        if closed.sum() > 0:
            avg = round(float(s.loc[closed, mc].mean()), 2)
            fwd = (s['product'] == prod) & s[mc].isna() & (s['forecast_type'] != 'actuals')
            s.loc[fwd, mc] = avg

s = s.drop(columns=['actual'], errors='ignore')
print(f"2026+ scaffold: {len(s)} rows, types: {s['forecast_type'].value_counts().to_dict()}")

# ═══ A1b. OVERRIDE FORWARD FORECAST WITH BOQ-PACED VALUES FROM CBE ════
# The p50 from pure_predictions is the raw model output. The CBE table's
# 'forecast' column uses BOQ-paced values (more conservative and aligned
# with FP&A planning). Override Most_Likely for forward weeks only.
# Bands (p05-p95, Worst_Case, Best_Case) stay from calibrated pure_predictions.
_UCC_CBE = f"{CATALOG}.{SCHEMA}.ucc_forecast_cbe"
_ITSG_CBE = f"{CATALOG}.{SCHEMA}.itsg_forecast_cbe"
# 2026-10-06: UCC forward weeks now come from the frozen BOQ plan
# (ucc_forecast_boq.forecast, the same figure the UCC notebook prints as
# "BOQ Plan": UCC 2026Q4 = $15.098M) instead of the CBE ($13.661M).
# ITSG is unchanged (CBE). Add 'ITSG' here to switch it too.
_BOQ_PRODUCTS = {'UCC'}
_BOQ_FY_END = f"{date.today().year}-12-31"  # current FY only; raw BOQ forecast overstates 2027+ vs p50
_UCC_BOQ = f"{CATALOG}.{SCHEMA}.ucc_forecast_boq"
_ITSG_BOQ = f"{CATALOG}.{SCHEMA}.itsg_forecast_boq"

_fwd_mask = s['forecast_type'].isin(['forecast', 'boq_forward'])
_overrides_applied = 0
for _prod, _cbe in [('UCC', _UCC_CBE), ('ITSG', _ITSG_CBE)]:
    if _prod in _BOQ_PRODUCTS:
        _src = _UCC_BOQ if _prod == 'UCC' else _ITSG_BOQ
        _cbe_fwd = (load_boq_weekly(_prod, _BOQ_FY_END)
                    .selectExpr("CAST(_boq_ds AS TIMESTAMP) AS ds", "_boq_p50 AS boq_forecast").toPandas())
        print(f"  {_prod}: forward Most_Likely from BOQ plan ({_src}), {len(_cbe_fwd)} weeks")
    else:
      _cbe_fwd = spark.sql(f"""
        SELECT CAST(ds AS TIMESTAMP) AS ds, forecast AS boq_forecast
        FROM {_cbe}
        WHERE model = 'Adaptive_Ensemble' AND source = 'forecast'
          AND run_date_utc = (SELECT MAX(run_date_utc) FROM {_cbe})
      """).toPandas()
    _cbe_fwd['ds'] = pd.to_datetime(_cbe_fwd['ds'])
    for _, _row in _cbe_fwd.iterrows():
        _match = (s['product'] == _prod) & (s['ds'] == _row['ds']) & _fwd_mask
        if _match.sum() > 0:
            s.loc[_match, 'Most_Likely'] = _row['boq_forecast']
            s.loc[_match, 'blended_p50'] = _row['boq_forecast']
            s.loc[_match, 'p50'] = _row['boq_forecast']
            _overrides_applied += _match.sum()

# Recompute Total forward weeks as UCC + ITSG
_total_fwd = (s['product'] == 'Total') & _fwd_mask
if _total_fwd.sum() > 0:
    for _ds in s.loc[_total_fwd, 'ds'].unique():
        _ucc_val = s.loc[(s['product'] == 'UCC') & (s['ds'] == _ds) & _fwd_mask, 'Most_Likely']
        _itsg_val = s.loc[(s['product'] == 'ITSG') & (s['ds'] == _ds) & _fwd_mask, 'Most_Likely']
        if len(_ucc_val) > 0 and len(_itsg_val) > 0:
            _total = float(_ucc_val.iloc[0]) + float(_itsg_val.iloc[0])
            _t_mask = (s['product'] == 'Total') & (s['ds'] == _ds) & _fwd_mask
            s.loc[_t_mask, 'Most_Likely'] = _total
            s.loc[_t_mask, 'blended_p50'] = _total
            s.loc[_t_mask, 'p50'] = _total

print(f"Forward-week override: {_overrides_applied} rows updated (UCC from BOQ plan, others from CBE)")

# ═══ A2. FILL 2026 ACTUALS GAP (closed weeks not in pure predictions) ════
# Pure predictions only has forward forecast + BOQ rows. Closed weeks in 2026
# (e.g. Q3 Jul 6 – Sep 14) are missing. Pull them from the CBE tables so the
# app displays a complete quarterly view with actuals + forecast.
UCC_CBE = f"{CATALOG}.{SCHEMA}.ucc_forecast_cbe"
ITSG_CBE = f"{CATALOG}.{SCHEMA}.itsg_forecast_cbe"

# Overlay CBE actuals onto scaffold — OVERRIDE hindcast rows for closed weeks
# (pure_predictions may have hindcast model data for Jul-Sep, but closed weeks
# need actual revenue in Actuals/Most_Likely/forecast_type)
actuals_gap = []
for product, cbe_tbl in [('UCC', UCC_CBE), ('ITSG', ITSG_CBE)]:
    cbe_act = spark.sql(f"""
        SELECT CAST(ds AS TIMESTAMP) AS ds, p50
        FROM {cbe_tbl}
        WHERE model = 'Adaptive_Ensemble' AND source = 'actual'
          AND ds >= '{CUTOFF}'
          AND run_date_utc = (SELECT MAX(run_date_utc) FROM {cbe_tbl})
    """).toPandas()
    cbe_act['ds'] = pd.to_datetime(cbe_act['ds'])

    # For dates already in scaffold (e.g. hindcast rows), OVERRIDE with actuals
    _existing_dates = set(s[s['product'] == product]['ds'].unique())
    _to_override = cbe_act[cbe_act['ds'].isin(_existing_dates)]
    for _, _ov in _to_override.iterrows():
        _mask = (s['product'] == product) & (s['ds'] == _ov['ds'])
        s.loc[_mask, 'Actuals'] = _ov['p50']
        s.loc[_mask, 'Most_Likely'] = _ov['p50']
        s.loc[_mask, 'blended_p50'] = _ov['p50']
        s.loc[_mask, 'p50'] = _ov['p50']
        s.loc[_mask, 'Worst_Case'] = _ov['p50']
        s.loc[_mask, 'Best_Case'] = _ov['p50']
        s.loc[_mask, 'forecast_type'] = 'actuals'
        s.loc[_mask, 'source'] = 'cbe_actuals'
        # Set p05-p95 to actual for closed weeks (no uncertainty)
        for _q in ['p05','p25','p75','p95']:
            s.loc[_mask, _q] = _ov['p50']
    if len(_to_override) > 0:
        print(f"  {product}: overrode {len(_to_override)} hindcast rows with CBE actuals")

    # For dates NOT in scaffold, add new rows (original logic)
    cbe_act = cbe_act[~cbe_act['ds'].isin(_existing_dates)]
    if len(cbe_act) > 0:
        gap = pd.DataFrame({
            'ds': cbe_act['ds'], 'product': product, 'sales_market': 'Total',
            'forecast_type': 'actuals', 'forecast_view': 'adaptive',
            'run_date': RUN, 'Actuals': cbe_act['p50'].values,
            'Most_Likely': cbe_act['p50'].values,
            'Worst_Case': cbe_act['p50'].values,
            'Best_Case': cbe_act['p50'].values,
            'p50': cbe_act['p50'].values, 'blended_p50': cbe_act['p50'].values,
            'source': 'cbe_actuals', 'bands_method': None,
        })
        for q in ['p05','p25','p75','p95']:
            gap[q] = cbe_act['p50'].values
        # ── Fill model columns from VINTAGE predictions (backtest) ──────
        # Use the earliest vintage that predicted each week (closest to
        # a genuine forward forecast). Consistent with Jan-Jun backtest_cv.
        _VINT_TBL = f"{CATALOG}.{SCHEMA}.{'ucc' if product == 'UCC' else 'itsg'}_forecast_vintages"
        _VINT_MODELS = {
            'Prophet_trend': 'prophet', 'MSTL_v2': 'mstl_v2',
            'ETS': 'ets', 'DHR_ARIMA': 'dhr_arima',
        }
        _LGB_PATTERN = 'Global_LGB_Q50' if product == 'UCC' else 'Global_LGB_Q50'
        _gap_dates_str = ','.join([f"'{d.strftime('%Y-%m-%d')}'" for d in cbe_act['ds']])
        try:
            _vint_df = spark.sql(f"""
                WITH ranked AS (
                    SELECT ds, model, p50,
                           ROW_NUMBER() OVER (PARTITION BY ds, model ORDER BY run_date_utc ASC) AS rn
                    FROM {_VINT_TBL}
                    WHERE grain_level = 'total' AND ds IN ({_gap_dates_str})
                )
                SELECT CAST(ds AS TIMESTAMP) AS ds, model, p50
                FROM ranked WHERE rn = 1
            """).toPandas()
            _vint_df['ds'] = pd.to_datetime(_vint_df['ds'])
            # Pivot vintage predictions into gap rows
            for mname, key in _VINT_MODELS.items():
                acol, mcol = f'arr_{key}', f'mape_{key}'
                _mv = _vint_df[_vint_df['model'] == mname][['ds', 'p50']].copy()
                _mv = _mv.rename(columns={'p50': acol})
                gap = gap.merge(_mv, on='ds', how='left')
                # Compute MAPE against actual
                gap[mcol] = np.where(
                    gap['Actuals'].notna() & (gap['Actuals'].abs() > 0),
                    (gap[acol] - gap['Actuals']).abs() / gap['Actuals'].abs() * 100, np.nan)
            # LightGBM (product-specific model name)
            _lgb_v = _vint_df[_vint_df['model'].str.contains('LGB')][['ds', 'p50']].copy()
            _lgb_v = _lgb_v.drop_duplicates('ds', keep='first').rename(columns={'p50': 'arr_lightgbm'})
            gap = gap.merge(_lgb_v, on='ds', how='left')
            gap['mape_lightgbm'] = np.where(
                gap['Actuals'].notna() & (gap['Actuals'].abs() > 0),
                (gap['arr_lightgbm'] - gap['Actuals']).abs() / gap['Actuals'].abs() * 100, np.nan)
            print(f"  {product}: {len(cbe_act)} actuals + vintage backtest predictions from {_VINT_TBL}")
        except Exception as _ve:
            print(f"  {product}: {len(cbe_act)} actuals added (vintage lookup failed: {_ve})")
            for _mc in ['arr_prophet','arr_mstl_v2','arr_ets','arr_dhr_arima','arr_lightgbm']:
                if _mc not in gap.columns: gap[_mc] = np.nan
            for _mc in ['mape_prophet','mape_mstl_v2','mape_ets','mape_dhr_arima','mape_lightgbm']:
                if _mc not in gap.columns: gap[_mc] = np.nan
        actuals_gap.append(gap)

if actuals_gap:
    gap_df = pd.concat(actuals_gap, ignore_index=True)
    # 2026-09-30 CRITICAL FIX: this used to build a 'Total' rollup row here by
    # summing gap_df PER DS -- but gap_df only contains the NEW rows just
    # added for UCC/ITSG in this section (dates that weren't already in `s`
    # for THAT product). If UCC had no new gap row for a date (because its
    # row already existed and was overridden in place, further up) while
    # ITSG DID get a new gap row for that same date, the old code summed
    # gap_df for that ds and got ONLY ITSG's value -- an incomplete "Total"
    # -- then appended it as a brand-new row WITHOUT checking whether `s`
    # already had a (correct, complete) 'Total' row for that date from the
    # original ens-based scaffold. Result: two 'Total' rows for the same
    # date, one complete and one an ITSG-only undercount, both later summed
    # into the FY total -- this was the root cause of the Dynamic-vs-Static
    # cross-check failing by +28.5% (ITSG) / +6.0% (Total) on a real run.
    # Fix: don't build a Total rollup from the incomplete gap_df at all.
    # Just add the per-product gap rows here; the block below recomputes
    # Total from the FINAL, complete state of `s` (after both products'
    # overrides/gap-fills are done), updating an existing Total row OR
    # inserting a new one -- so there is exactly one Total row per date.
    s = pd.concat([s, gap_df], ignore_index=True)
    print(f"  Per-product gap rows added: {len(gap_df)}")
else:
    print("  No new actuals gap rows (existing rows overridden in place)")

# ── Recompute Total product = UCC + ITSG for every date where both have
# actuals, from the FINAL state of `s` -- update the existing Total row if
# one exists, or insert a new one if it doesn't (covers dates that only
# just became gap rows above for one or both products). Exactly one Total
# row per date, by construction.
_ucc_act = s[(s['product'] == 'UCC') & (s['forecast_type'] == 'actuals')]
_itsg_act = s[(s['product'] == 'ITSG') & (s['forecast_type'] == 'actuals')]
_act_dates = set(_ucc_act['ds']) & set(_itsg_act['ds'])
_total_ov = 0
_total_new_rows = []
for _d in _act_dates:
    _t_mask = (s['product'] == 'Total') & (s['ds'] == _d)
    _existing_is_actual = _t_mask.sum() > 0 and s.loc[_t_mask, 'forecast_type'].iloc[0] == 'actuals'
    if _existing_is_actual:
        continue  # already correct (e.g. propagated on an earlier run) -- don't touch
    _u_val = float(_ucc_act.loc[_ucc_act['ds'] == _d, 'Actuals'].iloc[0])
    _i_val = float(_itsg_act.loc[_itsg_act['ds'] == _d, 'Actuals'].iloc[0])
    _total_val = _u_val + _i_val
    if _t_mask.sum() > 0:
        # Update the existing Total row(s) for this date in place.
        s.loc[_t_mask, 'Actuals'] = _total_val
        s.loc[_t_mask, 'Most_Likely'] = _total_val
        s.loc[_t_mask, 'blended_p50'] = _total_val
        s.loc[_t_mask, 'p50'] = _total_val
        s.loc[_t_mask, 'Worst_Case'] = _total_val
        s.loc[_t_mask, 'Best_Case'] = _total_val
        s.loc[_t_mask, 'forecast_type'] = 'actuals'
        s.loc[_t_mask, 'source'] = 'cbe_actuals'
        for _q in ['p05','p25','p75','p95']:
            s.loc[_t_mask, _q] = _total_val
        _total_ov += 1
    else:
        # No Total row exists yet for this date (e.g. both UCC and ITSG only
        # just got new gap rows above) -- insert exactly one.
        _total_new_rows.append({
            'ds': _d, 'product': 'Total', 'sales_market': 'Total',
            'forecast_type': 'actuals', 'forecast_view': 'adaptive',
            'run_date': RUN, 'Actuals': _total_val, 'Most_Likely': _total_val,
            'Worst_Case': _total_val, 'Best_Case': _total_val,
            'p50': _total_val, 'blended_p50': _total_val,
            'p05': _total_val, 'p25': _total_val, 'p75': _total_val, 'p95': _total_val,
            'source': 'cbe_actuals', 'bands_method': None,
        })
if _total_new_rows:
    s = pd.concat([s, pd.DataFrame(_total_new_rows)], ignore_index=True)
if _total_ov > 0 or _total_new_rows:
    print(f"  Total: overrode {_total_ov} existing rows + inserted {len(_total_new_rows)} new rows "
          f"with UCC+ITSG actuals sum (exactly one Total row per date)")

# ═══ B. Pre-2026 actuals from v2compat ═════════════════════════════════
max_rd = spark.sql(f"SELECT MAX(run_date) FROM {V2C}").collect()[0][0]
pre = spark.sql(f"""
    SELECT CAST(ds AS TIMESTAMP) AS ds, product, sales_market,
           forecast_type, forecast_view,
           Actuals, Most_Likely, Worst_Case, Best_Case,
           p05, p25, p75, p95,
           arr_ets, arr_prophet, arr_lightgbm, arr_mstl_v2, arr_dhr_arima,
           mape_ets, mape_prophet, mape_lightgbm, mape_mstl_v2, mape_dhr_arima
    FROM {V2C}
    WHERE run_date = '{max_rd}'
      AND forecast_type = 'actuals'
      AND ds < TIMESTAMP '{CUTOFF}'
""").toPandas()
pre['ds'] = pd.to_datetime(pre['ds'])
pre['p50']          = pre['Most_Likely']
pre['blended_p50']  = pre['Most_Likely']
pre['source']       = 'historical'
pre['bands_method'] = None
pre['run_date']     = RUN
print(f"Pre-2026 actuals: {len(pre)} rows ({pre['product'].nunique()} products)")

# ═══ C. Union + finalize ══════════════════════════════════════════════
OUT = ['ds', 'product', 'sales_market', 'forecast_type', 'forecast_view',
       'run_date', 'Actuals', 'Most_Likely', 'Worst_Case', 'Best_Case',
       'p05', 'p25', 'p50', 'p75', 'p95',
       'blended_p50', 'source', 'bands_method',
       'arr_ets', 'arr_prophet', 'arr_lightgbm', 'arr_mstl_v2', 'arr_dhr_arima',
       'mape_ets', 'mape_prophet', 'mape_lightgbm', 'mape_mstl_v2', 'mape_dhr_arima']

for df in [s, pre]:
    for c in OUT:
        if c not in df.columns:
            df[c] = np.nan

snap = pd.concat([pre[OUT], s[OUT]], ignore_index=True)
snap = snap.sort_values(['product', 'ds']).reset_index(drop=True)

# ── Validate ───────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"APP SNAPSHOT V2: {len(snap)} rows, {snap['ds'].nunique()} weeks")
print(f"  Forecast types: {snap['forecast_type'].value_counts().to_dict()}")
print(f"  Products: {sorted(snap['product'].unique())}")
for p in ['UCC', 'ITSG', 'Total']:
    fy = snap[(snap['product'] == p) & (snap['ds'] >= '2026-01-05') & (snap['ds'] <= '2026-12-28')]
    ml_total = fy['Most_Likely'].sum()
    bands_ok = fy['p05'].notna().sum()
    print(f"  {p} FY2026: ${ml_total/1e6:.2f}M  (bands on {bands_ok}/{len(fy)} rows)")

# Cross-check vs static table.
# 2026-09-30: this used to assert ~0.000% diff, which assumed `snap` never
# legitimately diverges from `raw`. That assumption is stale now that
# Sections A1b/A2 exist: BOQ overrides deliberately replace the raw model
# P50 with a more-conservative BOQ-paced value for forward weeks, and the
# CBE-actuals override deliberately replaces a raw model ESTIMATE (backtest/
# hindcast bands_method, not a real actual) with the true closed-week actual
# for weeks that have since closed -- both are intentional corrections, not
# bugs, and will show up here as a positive diff. A real run also surfaced
# an actual duplication bug (a 'Total' rollup row being double-counted -- see
# the fix note on the Total-recompute block above), which produced a much
# larger, genuinely-wrong diff (+28.5% ITSG, +6.0% Total) than the override
# corrections alone would explain; that bug is now fixed.
# So: flag anything large enough to suggest a bug (>15%, a threshold picked
# well above what a few weeks' worth of model-estimate-to-actual correction
# should plausibly move a full-year total) as needing a look, but no longer
# treat any nonzero diff as an error -- a modest positive diff here is
# expected behavior, not a regression.
CROSS_CHECK_INVESTIGATE_THRESHOLD_PCT = 15.0
for p in ['UCC', 'ITSG', 'Total']:
    stat_fy = raw[(raw['model'] == 'Adaptive_Ensemble') &
                  (raw['product'] == p) &
                  (raw['ds'] >= '2026-01-05') & (raw['ds'] <= '2026-12-28')]['blended_p50'].sum()
    dyn_fy = snap[(snap['product'] == p) & (snap['ds'] >= '2026-01-05') &
                  (snap['ds'] <= '2026-12-28')]['Most_Likely'].sum()
    diff = (dyn_fy - stat_fy) / stat_fy * 100 if stat_fy else 0
    if abs(diff) < 0.01:
        status = '✓ exact match'
    elif abs(diff) < CROSS_CHECK_INVESTIGATE_THRESHOLD_PCT:
        status = '~ expected (BOQ/actuals overrides)'
    else:
        status = f'✗ INVESTIGATE (exceeds {CROSS_CHECK_INVESTIGATE_THRESHOLD_PCT:.0f}% threshold)'
    print(f"  {status}  {p} Dynamic vs Static FY2026: ${dyn_fy/1e6:.2f}M vs ${stat_fy/1e6:.2f}M ({diff:+.3f}%)")
print(f"{'='*70}")

# ═══ C2. PERIOD-END UPLIFT (live-computed from previous years' actuals) ═══
# 2026-09-29: replaced frozen QE_UPLIFT=1.5 / ME_UPLIFT=1.3 constants (a
# one-time historical measurement that would silently go stale) with a
# factor recomputed fresh each run from actual/model-forecast ratios at
# PRIOR YEARS' period-end weeks, adapted here for this notebook's pandas
# pipeline and computed per product (UCC, ITSG, Total each get their own
# factor, since their scale/dynamics differ).
#
# THREE tiers, not two -- 2026-09-29 update: fiscal year-end (Q4, since GoTo
# FY = calendar year) is split out from ordinary quarter-end. Budget-flush
# behavior at FY close (unspent budget doesn't roll over) is a materially
# bigger, separately-documented effect than a routine Q1/Q2/Q3 close, so
# collapsing both into one "quarter-end" factor would understate FY-end and/
# or overstate a normal quarter-end. Precedence: year-end > quarter-end >
# month-end (a week is exactly one of the three).
#
# Method (per product, per tier):
#   For every week that is NOW closed (has a real Actuals value) and falls
#   on that tier's boundary, in a year strictly BEFORE the current forecast
#   year (CURRENT_FY, from Cell 2): ratio = actual / raw model P50. "Raw"
#   means `ens`'s own p50 from `raw` (forecast_pure_predictions), captured
#   in Section A BEFORE the CBE-actual override further up in this cell
#   overwrites `s`/`snap`'s p50 with the actual for closed weeks -- using
#   the already-overwritten value would make the ratio circular (~1.0x
#   every time).
#   factor = median(ratio) across all such prior-year weeks for that tier.
#   Floored at 1.0 (never shrinks the forecast), capped at 3.0, minimum 3
#   prior-year weeks required per (product, tier) or it falls back to the
#   documented default of 1.0 (no uplift) rather than guessing from too
#   little history -- year-end is the tier most likely to fall back to
#   default early on, since GoTo has at most one FY-end week per prior year.
UPLIFT_MIN_SAMPLE = 3
UPLIFT_FLOOR = 1.0
UPLIFT_CAP = 3.0
UPLIFT_DEFAULT = 1.0

_ds = pd.to_datetime(snap['ds'])
_next_week = _ds + pd.Timedelta(days=7)
_is_forecast = snap['forecast_type'].isin(['forecast', 'boq_forward'])
_is_month_end = _ds.dt.month != _next_week.dt.month
_is_quarter_end = _ds.dt.quarter != _next_week.dt.quarter
_is_year_end = _ds.dt.year != _next_week.dt.year   # implies quarter-end (Q4 -> Q1)

_ye_mask = _is_forecast & _is_year_end
_qe_mask = _is_forecast & _is_quarter_end & ~_is_year_end
_me_mask = _is_forecast & _is_month_end & ~_is_quarter_end

_q_cols = [c for c in ['p05','p25','p50','p75','p95',
                        'blended_p50','Most_Likely','Worst_Case','Best_Case',
                        'arr_ets','arr_prophet','arr_lightgbm','arr_mstl_v2','arr_dhr_arima']
           if c in snap.columns]

# Raw (pre-override) Adaptive_Ensemble model p50, keyed by (product, ds).
_raw_p50 = ens[['product', 'ds', 'p50']].rename(columns={'p50': 'raw_p50'}).drop_duplicates(['product', 'ds'])

_hist = snap[['ds', 'product', 'Actuals']].copy()
_hist['ds'] = pd.to_datetime(_hist['ds'])
_hist = _hist.merge(_raw_p50, on=['product', 'ds'], how='inner')
_hist_next_week = _hist['ds'] + pd.Timedelta(days=7)
_hist['is_year_end'] = _hist['ds'].dt.year != _hist_next_week.dt.year
_hist['is_qe'] = (_hist['ds'].dt.quarter != _hist_next_week.dt.quarter) & ~_hist['is_year_end']
_hist['is_me'] = (_hist['ds'].dt.month != _hist_next_week.dt.month) & ~_hist['is_qe'] & ~_hist['is_year_end']
_hist = _hist[
    _hist['Actuals'].notna() & _hist['raw_p50'].notna() & (_hist['raw_p50'] != 0)
    & (_hist['ds'].dt.year < CURRENT_FY)
]
_hist['ratio'] = _hist['Actuals'] / _hist['raw_p50']

def _live_uplift_factor(product, period_flag_col):
    sub = _hist[(_hist['product'] == product) & _hist[period_flag_col]]
    n = len(sub)
    if n < UPLIFT_MIN_SAMPLE:
        return UPLIFT_DEFAULT, n
    factor = float(np.clip(sub['ratio'].median(), UPLIFT_FLOOR, UPLIFT_CAP))
    return factor, n

print("Computing live year-end/quarter-end/month-end uplift factors from previous years' actuals:")
_uplift_factors = {}
for _prod in snap['product'].unique():
    _ye_f, _ye_n = _live_uplift_factor(_prod, 'is_year_end')
    _qe_f, _qe_n = _live_uplift_factor(_prod, 'is_qe')
    _me_f, _me_n = _live_uplift_factor(_prod, 'is_me')
    _uplift_factors[_prod] = (_ye_f, _qe_f, _me_f)
    print(f"  {_prod}: YE factor={_ye_f:.3f}x (n={_ye_n} prior FY-end wks), "
          f"QE factor={_qe_f:.3f}x (n={_qe_n} prior-year qtr-end wks), "
          f"ME factor={_me_f:.3f}x (n={_me_n} prior-year month-end wks)")

for _prod, (_ye_f, _qe_f, _me_f) in _uplift_factors.items():
    _prod_ye = _ye_mask & (snap['product'] == _prod)
    _prod_qe = _qe_mask & (snap['product'] == _prod)
    _prod_me = _me_mask & (snap['product'] == _prod)
    for _c in _q_cols:
        snap.loc[_prod_ye, _c] = (snap.loc[_prod_ye, _c] * _ye_f).round(2)
        snap.loc[_prod_qe, _c] = (snap.loc[_prod_qe, _c] * _qe_f).round(2)
        snap.loc[_prod_me, _c] = (snap.loc[_prod_me, _c] * _me_f).round(2)

print(f"\nPeriod-end uplift applied (live, per product, from previous years): "
      f"YE rows={_ye_mask.sum()}, QE rows={_qe_mask.sum()}, ME rows={_me_mask.sum()}")

# ═══ D. Write ════════════════════════════════════════════════════════════
sdf = spark.createDataFrame(snap)
sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SNAP)
print(f"\n✓ Wrote {SNAP} ({len(snap)} rows, run_date={RUN})")

spark.sql(f"GRANT SELECT ON TABLE {SNAP} TO `{APP_SP}`")
print(f"✓ Granted SELECT on {SNAP}")

# ── Also write to federated.sales for cross-catalog app access ──────────
FED_CAT   = "federated"
FED_SCHEMA = "federated.sales"
FED_SNAP   = "federated.sales.forecast_app_snapshot"
# The WRITE and the GRANTs are reported separately: a failed grant (e.g. no MANAGE on the
# catalog) must not be reported as a failed write.
try:
    sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FED_SNAP)
    print(f"✓ Wrote {FED_SNAP} ({len(snap)} rows, run_date={RUN})")
except Exception as e:
    print(f"✗ federated.sales WRITE FAILED -- app will keep reading the previous snapshot: {e}")
    raise
# Full grant chain: catalog → schema → table (non-fatal; usually already granted)
for _stmt in (f"GRANT USE CATALOG ON CATALOG {FED_CAT} TO `{APP_SP}`",
              f"GRANT USE SCHEMA ON SCHEMA {FED_SCHEMA} TO `{APP_SP}`",
              f"GRANT SELECT ON TABLE {FED_SNAP} TO `{APP_SP}`"):
    try:
        spark.sql(_stmt)
        print(f"✓ {_stmt}")
    except Exception as e:
        print(f"⚠ grant skipped (write already succeeded): {_stmt} -> {str(e)[:120]}")

In [ ]:
import pandas as pd
import numpy as np

# ══════════════════════════════════════════════════════════════════════════════
# APP SNAPSHOT "INCL. MIRADORE" -- pure-python builder (no Spark) + self-test
# ══════════════════════════════════════════════════════════════════════════════
# Miradore is NOT in the Salesforce ITSG target. Notebook 02 (Step 13M) publishes
#   itsg_miradore_outlook         (monthly: run_date, month[date], plan, actual, attainment, outlook, method, validated, notes)
#   itsg_forecast_incl_miradore   (weekly : run_date, ds[date], salesforce_itsg_p50, miradore_outlook, miradore_available, total, view_label)
# This cell turns them into a SECOND app snapshot with Miradore already added to ITSG and Total.
RUN_INCL_MIRADORE_SELF_TEST = True
MIRA_MONTHLY_COLS = ['run_date', 'month', 'actual', 'outlook']
MIRA_WEEKLY_COLS = ['run_date', 'ds', 'miradore_outlook', 'miradore_available']
MIRA_SHIFT_COLS = ['Actuals', 'Most_Likely', 'Worst_Case', 'Best_Case', 'blended_p50',
                   'p05', 'p25', 'p50', 'p75', 'p95']   # constant shift; per-model arr_*/mape_* are NOT touched


def _mira_latest(df, need, name):
    """Latest run_date slice; returns (df|None, reason)."""
    if df is None or len(df) == 0:
        return None, f'{name} is missing or empty'
    miss = [c for c in need if c not in df.columns]
    if miss:
        return None, f'{name} lacks columns {miss}. Actual columns: {list(df.columns)}'
    d = df.copy()
    d['run_date'] = pd.to_datetime(d['run_date'], errors='coerce')
    if d['run_date'].notna().sum() == 0:
        return None, f'{name} has no parseable run_date. Actual columns: {list(df.columns)}'
    d = d[d['run_date'] == d['run_date'].max()].copy()
    return d, ''


def _mondays_in_month(period):
    days = pd.date_range(period.to_timestamp(), (period + 1).to_timestamp() - pd.Timedelta(days=1), freq='D')
    return int((days.dayofweek == 0).sum())


def build_incl_miradore(snap_df, monthly_df, weekly_df):
    """Return (alt_df, info) or (None, reason). Never raises. alt_df = snap copy + miradore_amount + miradore_basis."""
    try:
        mo, why = _mira_latest(monthly_df, MIRA_MONTHLY_COLS, 'itsg_miradore_outlook')
        if mo is None:
            return None, why
        wk, why = _mira_latest(weekly_df, MIRA_WEEKLY_COLS, 'itsg_forecast_incl_miradore')
        if wk is None:
            return None, why
        mo['p'] = pd.to_datetime(mo['month'], errors='coerce').dt.to_period('M')
        mo['ext'] = (mo['extended'].astype(str).str.lower().isin(['true', '1', '1.0'])
                     if 'extended' in mo.columns else False)
        mo = mo.dropna(subset=['p']).drop_duplicates('p', keep='last').set_index('p')
        wk['dsn'] = pd.to_datetime(wk['ds'], errors='coerce').dt.normalize()
        wk['v'] = pd.to_numeric(wk['miradore_outlook'], errors='coerce')
        wk['ok'] = wk['miradore_available'].astype(str).str.lower().isin(['true', '1', '1.0']) & wk['v'].notna()
        wmap = {r.dsn: r.v for r in wk.itertuples() if r.ok}
        if len(mo) == 0:
            return None, f'itsg_miradore_outlook has no parseable month. Actual columns: {list(monthly_df.columns)}'
        out = snap_df.copy()
        ds = pd.to_datetime(out['ds']).dt.normalize()
        u = out.loc[out['product'] == 'ITSG', ['ds', 'forecast_type']].copy()
        u['dsn'] = pd.to_datetime(u['ds']).dt.normalize()
        u = u.drop_duplicates('dsn')
        amt, bas = {}, {}
        for r in u.itertuples():
            per = r.dsn.to_period('M')
            if r.forecast_type in ('forecast', 'boq_forward'):
                ext = bool(mo.at[per, 'ext']) if per in mo.index else False
                lab = 'extended' if ext else 'outlook'
                if r.dsn in wmap:
                    amt[r.dsn], bas[r.dsn] = float(wmap[r.dsn]), lab
                elif per in mo.index:
                    # weekly table only covers ~27 weeks; later forecast weeks come from the monthly outlook / Mondays
                    o_m = pd.to_numeric(mo.at[per, 'outlook'], errors='coerce')
                    a_m = pd.to_numeric(mo.at[per, 'actual'], errors='coerce')
                    if pd.notna(o_m) and pd.isna(a_m):
                        amt[r.dsn], bas[r.dsn] = float(o_m) / _mondays_in_month(per), lab
            elif r.forecast_type == 'actuals' and per in mo.index:
                n = _mondays_in_month(per)
                a, o = pd.to_numeric(mo.at[per, 'actual'], errors='coerce'), pd.to_numeric(mo.at[per, 'outlook'], errors='coerce')
                if pd.notna(a):
                    amt[r.dsn], bas[r.dsn] = float(a) / n, 'actual'
                elif pd.notna(o):
                    amt[r.dsn], bas[r.dsn] = float(o) / n, 'estimate'
        is_i = out['product'].isin(['ITSG', 'Total'])
        out['miradore_amount'] = np.where(is_i, ds.map(amt).astype(float), np.nan)
        out['miradore_amount'] = out['miradore_amount'].fillna(0.0).astype(float)
        _b = ds.map(bas).astype(object)
        out['miradore_basis'] = _b.where(is_i & _b.notna(), None).astype(object)
        for c in [c for c in MIRA_SHIFT_COLS if c in out.columns]:
            out[c] = pd.to_numeric(out[c], errors='coerce') + out['miradore_amount']
        # fixed original dtype for untouched cols; shifted cols stay float (NaN stays NaN)
        return out, {'n_weeks_with_amount': len(amt), 'basis_counts': pd.Series(list(bas.values())).value_counts().to_dict()}
    except Exception as exc:
        return None, f'unexpected {type(exc).__name__}: {str(exc)[:300]}'


def validate_incl_miradore(base, alt):
    """Asserts consistency; returns a quarter summary DataFrame."""
    assert len(base) == len(alt), 'row count changed'
    assert list(alt.columns[:len(base.columns)]) == list(base.columns), 'column order changed'
    assert list(alt.columns[len(base.columns):]) == ['miradore_amount', 'miradore_basis'], 'extra columns wrong'
    b, a = base.reset_index(drop=True), alt.reset_index(drop=True)
    fc = b['forecast_type'].isin(['forecast', 'boq_forward'])
    for prod in ('ITSG', 'Total'):
        m = fc & (b['product'] == prod) & b['Most_Likely'].notna()
        d = float((a.loc[m, 'Most_Likely'] - b.loc[m, 'Most_Likely']).sum())
        s = float(a.loc[m, 'miradore_amount'].sum())
        assert abs(d - s) < 1e-6 * max(1.0, abs(s)), f'{prod} forecast-week Most_Likely delta {d} != sum miradore {s}'
    ucc = b['product'] == 'UCC'
    assert (a.loc[ucc, 'miradore_amount'] == 0).all(), 'UCC got Miradore'
    for c in [c for c in b.columns if c not in MIRA_SHIFT_COLS]:
        assert b[c].equals(a[c]), f'unshifted column {c} changed'
    for c in [c for c in MIRA_SHIFT_COLS if c in b.columns]:
        assert b.loc[ucc, c].equals(a.loc[ucc, c]), f'UCC column {c} changed'
        assert (b[c].isna() == a[c].isna()).all(), f'NaN pattern changed in {c}'
    ti = (b['product'] == 'Total') & b['Most_Likely'].notna()
    key = pd.to_datetime(b['ds'])
    ia = a.loc[b['product'] == 'ITSG'].set_index(pd.to_datetime(b.loc[b['product'] == 'ITSG', 'ds']))['miradore_amount']
    ta = a.loc[b['product'] == 'Total'].set_index(pd.to_datetime(b.loc[b['product'] == 'Total', 'ds']))['miradore_amount']
    assert ia.groupby(level=0).sum().reindex(ta.index).fillna(0).round(6).equals(ta.groupby(level=0).sum().round(6)), 'Total miradore != ITSG miradore per week'
    rows = []
    for prod in ('ITSG', 'Total'):
        for lab, df in (('base', b), ('incl', a)):
            x = df[df['product'] == prod]
            v = x['Actuals'].fillna(x['Most_Likely'])
            q = pd.to_datetime(x['ds']).dt.to_period('Q').astype(str)
            rows.append(v.groupby(q.values).sum().rename(f'{prod}_{lab}'))
    summ = pd.concat(rows, axis=1)
    for prod in ('ITSG', 'Total'):
        summ[f'{prod}_delta'] = summ[f'{prod}_incl'] - summ[f'{prod}_base']
    return summ


def _selftest_incl_miradore():
    cols = ['product', 'ds', 'forecast_type', 'Actuals', 'Most_Likely', 'Worst_Case', 'Best_Case', 'blended_p50',
            'p05', 'p25', 'p50', 'p75', 'p95', 'arr_ets', 'mape_ets']
    spec = [('2026-02-02', 'actuals', 1000.0), ('2026-02-09', 'actuals', 1000.0),   # Feb-26: 4 Mondays, known actual 400
            ('2026-08-03', 'actuals', 2000.0),                                       # Aug-26: 5 Mondays, NO actual, outlook 300
            ('2026-10-05', 'forecast', np.nan), ('2026-10-12', 'forecast', np.nan),  # weekly outlook 42 / not available
            ('2027-03-01', 'forecast', np.nan), ('2027-03-08', 'forecast', np.nan)]  # Mar-27 extension: monthly 150 / 5 Mondays = 30
    rows = []
    for prod, k in (('UCC', 5000.0), ('ITSG', 100.0), ('Total', 5100.0)):
        for d, ft, act in spec:
            rows.append([prod, pd.Timestamp(d), ft, act if ft == 'actuals' else np.nan] + [k] * 9 + [k * 0.9, 7.5])
    snap0 = pd.DataFrame(rows, columns=cols)
    rd = pd.Timestamp('2026-10-01')
    mon = pd.DataFrame({'run_date': [rd, rd, pd.Timestamp('2026-09-01'), rd],
                        'month': [pd.Timestamp('2026-02-01').date(), pd.Timestamp('2026-08-01').date(), pd.Timestamp('2026-02-01').date(), pd.Timestamp('2027-03-01').date()],
                        'actual': [400.0, np.nan, 999.0, np.nan], 'outlook': [350.0, 300.0, 1.0, 150.0],
                        'extended': [False, False, False, True]})
    wkl = pd.DataFrame({'run_date': [rd, rd, rd], 'ds': [pd.Timestamp('2026-10-05').date(), pd.Timestamp('2026-10-12').date(), pd.Timestamp('2027-03-08').date()],
                        'miradore_outlook': [42.0, np.nan, 30.0], 'miradore_available': [True, False, True]})
    alt, info = build_incl_miradore(snap0, mon, wkl)
    assert alt is not None, info
    g = lambda p, d, c: float(alt[(alt['product'] == p) & (alt['ds'] == pd.Timestamp(d))][c].iloc[0])
    for p in ('ITSG', 'Total'):
        assert g(p, '2026-02-02', 'miradore_amount') == 100.0 and g(p, '2026-02-09', 'miradore_amount') == 100.0
        assert g(p, '2026-08-03', 'miradore_amount') == 60.0
        assert g(p, '2026-10-05', 'miradore_amount') == 42.0 and g(p, '2026-10-12', 'miradore_amount') == 0.0
    b0 = {'ITSG': 100.0, 'Total': 5100.0}
    for p in ('ITSG', 'Total'):
        for c in ['Most_Likely', 'Worst_Case', 'Best_Case', 'blended_p50', 'p05', 'p25', 'p50', 'p75', 'p95']:
            assert g(p, '2026-10-05', c) == b0[p] + 42.0, (p, c)
            assert g(p, '2026-10-12', c) == b0[p], (p, c)
        assert g(p, '2026-02-02', 'Actuals') == 1100.0 and g(p, '2026-08-03', 'Actuals') == 2060.0
        assert g(p, '2026-10-05', 'arr_ets') == b0[p] * 0.9 and g(p, '2026-10-05', 'mape_ets') == 7.5   # model cols untouched
        assert np.isnan(g(p, '2026-10-05', 'Actuals'))                                                   # NaN stays NaN
    for p in ('ITSG', 'Total'):   # extension month beyond the weekly table: monthly 150 / 5 Mondays = 30 (week 1 from monthly, week 2 from weekly)
        assert g(p, '2027-03-01', 'miradore_amount') == 30.0 and g(p, '2027-03-08', 'miradore_amount') == 30.0
        assert g(p, '2027-03-01', 'Most_Likely') == b0[p] + 30.0 and g(p, '2027-03-08', 'p95') == b0[p] + 30.0
    assert (alt.loc[alt['product'] == 'UCC', 'miradore_amount'] == 0).all()
    assert alt.loc[alt['product'] == 'UCC', cols].reset_index(drop=True).equals(snap0.loc[snap0['product'] == 'UCC'].reset_index(drop=True))
    bas = alt[alt['product'] == 'ITSG'].set_index('ds')['miradore_basis']
    assert bas[pd.Timestamp('2026-02-02')] == 'actual' and bas[pd.Timestamp('2026-08-03')] == 'estimate'
    assert bas[pd.Timestamp('2026-10-05')] == 'outlook' and bas[pd.Timestamp('2026-10-12')] is None
    assert bas[pd.Timestamp('2027-03-01')] == 'extended' and bas[pd.Timestamp('2027-03-08')] == 'extended'
    validate_incl_miradore(snap0, alt)
    for bad in ((None, wkl), (mon, None), (mon.iloc[0:0], wkl), (mon.drop(columns=['outlook']), wkl), (mon, wkl.drop(columns=['ds']))):
        r, why = build_incl_miradore(snap0, *bad)
        assert r is None and why, 'missing/empty/mismatched inputs must skip'
    print('incl-Miradore self-test: PASS (4 Mondays->100/wk actual; 5 Mondays->60 estimate; +42 forecast; extension month 150/5=30 labelled extended; Total mirrors; UCC + model cols unchanged; bad inputs skip)')


if RUN_INCL_MIRADORE_SELF_TEST:
    try:
        _selftest_incl_miradore()
    except Exception as _exc_st:
        print(f'incl-Miradore self-test: FAIL -> {type(_exc_st).__name__}: {_exc_st}')
        INCL_MIRADORE_SELFTEST_OK = False
    else:
        INCL_MIRADORE_SELFTEST_OK = True
else:
    INCL_MIRADORE_SELFTEST_OK = True

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PUBLISH forecast_app_snapshot_incl_miradore (ITSG + Total with a Miradore estimate added)
# ══════════════════════════════════════════════════════════════════════════════
# Built from the FINAL in-memory `snap` (after period-end uplift). FAIL-SOFT: if the self-test fails or either
# Miradore table is missing/empty/mismatched, a WARNING is printed and nothing is written; the base snapshot
# above is never touched. Write failure raises ONLY for the federated write (same as the base snapshot).
SNAP_MIRA     = f"{CATALOG}.{SCHEMA}.forecast_app_snapshot_incl_miradore"
FED_SNAP_MIRA = "federated.sales.forecast_app_snapshot_incl_miradore"
MIRA_MONTHLY_TBL = f"{CATALOG}.{SCHEMA}.itsg_miradore_outlook"
MIRA_WEEKLY_TBL  = f"{CATALOG}.{SCHEMA}.itsg_forecast_incl_miradore"
INCL_MIRADORE_ENABLED = True
try:
    if not INCL_MIRADORE_ENABLED:
        print("incl-Miradore snapshot: disabled (INCL_MIRADORE_ENABLED=False).")
    elif not INCL_MIRADORE_SELFTEST_OK:
        print("WARNING: incl-Miradore snapshot SKIPPED -- self-test failed (see cell above).")
    else:
        _mo_pdf = _wk_pdf = None
        for _t in (MIRA_MONTHLY_TBL, MIRA_WEEKLY_TBL):
            try:
                _pdf = spark.table(_t).toPandas()
            except Exception as _e_rd:
                _pdf = None
                print(f"WARNING: cannot read {_t}: {str(_e_rd)[:200]}")
            if _t == MIRA_MONTHLY_TBL:
                _mo_pdf = _pdf
            else:
                _wk_pdf = _pdf
        for _n, _d in (("itsg_miradore_outlook", _mo_pdf), ("itsg_forecast_incl_miradore", _wk_pdf)):
            if _d is not None:
                print(f"  {_n}: {len(_d)} rows, columns={list(_d.columns)}")
        _alt, _info = build_incl_miradore(snap, _mo_pdf, _wk_pdf)
        if _alt is None:
            print(f"WARNING: incl-Miradore snapshot SKIPPED (no table written): {_info}")
        else:
            _summ = validate_incl_miradore(snap, _alt)
            print(f"incl-Miradore built: {len(_alt)} rows; weeks with Miradore={_info['n_weeks_with_amount']}; basis={_info['basis_counts']}")
            print("Per-quarter value (Actuals else Most_Likely), base vs incl. Miradore ($):")
            print(_summ.round(0).to_string())
            for _q in [q for q in _summ.index if q.endswith('Q4')]:
                _r = _summ.loc[_q]
                print(f"APP EXPECTATION {_q}: ITSG base {_r['ITSG_base']:,.0f} -> incl. Miradore {_r['ITSG_incl']:,.0f}; "
                      f"Total base {_r['Total_base']:,.0f} -> incl. Miradore {_r['Total_incl']:,.0f}")
            if _alt['miradore_basis'].notna().any():
                _sdf2 = spark.createDataFrame(_alt)
            else:
                from pyspark.sql import functions as _F
                _sdf2 = spark.createDataFrame(_alt.drop(columns=['miradore_basis'])).withColumn('miradore_basis', _F.lit(None).cast('string'))
            _sdf2.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SNAP_MIRA)
            print(f"✓ Wrote {SNAP_MIRA} ({len(_alt)} rows)")
            try:
                spark.sql(f"GRANT SELECT ON TABLE {SNAP_MIRA} TO `{APP_SP}`")
                print(f"✓ Granted SELECT on {SNAP_MIRA}")
            except Exception as _e_g:
                print(f"⚠ grant skipped (write already succeeded): {SNAP_MIRA} -> {str(_e_g)[:120]}")
            try:
                _sdf2.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FED_SNAP_MIRA)
                print(f"✓ Wrote {FED_SNAP_MIRA} ({len(_alt)} rows)")
                _fed_ok = True
            except Exception as _e_f:
                print(f"✗ federated.sales incl-Miradore WRITE FAILED -- app toggle will keep the previous table: {_e_f}")
                _fed_ok = False
            if _fed_ok:
                for _stmt in (f"GRANT USE CATALOG ON CATALOG {FED_CAT} TO `{APP_SP}`",
                              f"GRANT USE SCHEMA ON SCHEMA {FED_SCHEMA} TO `{APP_SP}`",
                              f"GRANT SELECT ON TABLE {FED_SNAP_MIRA} TO `{APP_SP}`"):
                    try:
                        spark.sql(_stmt)
                        print(f"✓ {_stmt}")
                    except Exception as _e_g2:
                        print(f"⚠ grant skipped (write already succeeded): {_stmt} -> {str(_e_g2)[:120]}")
            else:
                raise RuntimeError(f"federated write failed for {FED_SNAP_MIRA}")
except RuntimeError:
    raise
except Exception as _exc_im:
    print(f"WARNING: incl-Miradore snapshot SKIPPED (fail-soft): {type(_exc_im).__name__}: {str(_exc_im)[:300]}")

In [ ]:
%sql --name monthly_summary
WITH latest AS (
  SELECT *
  FROM datagroup_mdl.mdl_sales_analytics.forecast_results
  WHERE run_date = (SELECT MAX(run_date) FROM datagroup_mdl.mdl_sales_analytics.forecast_results)
    AND model IN ('Adaptive_Ensemble', 'ACTUAL')
    AND UPPER(TRIM(sales_market)) IN ('TOTAL', 'ALL')
    AND grain_level = 'total'
    AND (forecast_view = 'adaptive' OR forecast_view IS NULL)
    AND ds >= '2026-01-01' AND ds <= '2026-12-28'
),
monthly AS (
  SELECT
    product,
    DATE_FORMAT(ds, 'yyyy-MM') AS month,
    MONTH(ds) AS m_num,
    CONCAT('Q', QUARTER(ds)) AS quarter,
    SUM(CASE WHEN model = 'ACTUAL' THEN actual ELSE 0 END) / 1e6 AS actuals_M,
    SUM(CASE WHEN model = 'Adaptive_Ensemble' AND forecast_type IN ('rolling','roy','forward_1yr')
             THEN p50 ELSE 0 END) / 1e6 AS fcst_p50_M,
    SUM(CASE WHEN model = 'Adaptive_Ensemble' AND forecast_type IN ('rolling','roy','forward_1yr')
             THEN p05 ELSE 0 END) / 1e6 AS fcst_p05_M,
    SUM(CASE WHEN model = 'Adaptive_Ensemble' AND forecast_type IN ('rolling','roy','forward_1yr')
             THEN p95 ELSE 0 END) / 1e6 AS fcst_p95_M,
    COUNT(DISTINCT CASE WHEN model = 'ACTUAL' THEN ds END) AS actual_wks,
    COUNT(DISTINCT CASE WHEN model = 'Adaptive_Ensemble' AND forecast_type IN ('rolling','roy','forward_1yr') THEN ds END) AS fcst_wks
  FROM latest
  GROUP BY product, DATE_FORMAT(ds, 'yyyy-MM'), MONTH(ds), QUARTER(ds)
)
SELECT
  product,
  month,
  quarter,
  ROUND(actuals_M, 2)                  AS `Actuals ($M)`,
  ROUND(fcst_p50_M, 2)                 AS `Forecast P50 ($M)`,
  ROUND(actuals_M + fcst_p50_M, 2)     AS `Landing ($M)`,
  ROUND(actuals_M + fcst_p05_M, 2)     AS `Low P05 ($M)`,
  ROUND(actuals_M + fcst_p95_M, 2)     AS `High P95 ($M)`,
  actual_wks  AS `Act Wks`,
  fcst_wks    AS `Fwd Wks`,
  CASE
    WHEN fcst_wks = 0 THEN 'Closed'
    WHEN actual_wks > 0 THEN 'Partial'
    ELSE 'Forecast'
  END AS status
FROM monthly
ORDER BY product, m_num

In [ ]:
%sql --name comparison
-- Side-by-side: forecast_results (source of truth) vs arr_forecast_wide (Power BI)
WITH fr AS (
  -- From forecast_results (what monthly summary showed)
  SELECT
    product,
    DATE_FORMAT(ds, 'yyyy-MM') AS month,
    MONTH(ds) AS m_num,
    SUM(CASE WHEN model = 'ACTUAL' THEN actual ELSE 0 END) / 1e6 AS fr_actuals_M,
    SUM(CASE WHEN model = 'Adaptive_Ensemble' AND forecast_type IN ('rolling','roy','forward_1yr')
             THEN p50 ELSE 0 END) / 1e6 AS fr_p50_M
  FROM datagroup_mdl.mdl_sales_analytics.forecast_results
  WHERE run_date = (SELECT MAX(run_date) FROM datagroup_mdl.mdl_sales_analytics.forecast_results)
    AND model IN ('Adaptive_Ensemble', 'ACTUAL')
    AND UPPER(TRIM(sales_market)) IN ('TOTAL', 'ALL')
    AND grain_level = 'total'
    AND (forecast_view = 'adaptive' OR forecast_view IS NULL)
    AND ds >= '2026-01-01' AND ds <= '2026-12-28'
  GROUP BY product, DATE_FORMAT(ds, 'yyyy-MM'), MONTH(ds)
),
pbi_deduped AS (
  -- Deduplicate: one row per (ds, product) — take actuals from 'actuals' rows,
  -- ensemble forecast from 'forecast'/'rolling' rows to avoid double-counting
  SELECT ds, product,
    MAX(CASE WHEN forecast_type = 'actuals' THEN actuals END) AS actual_val,
    MAX(CASE WHEN forecast_type != 'actuals' THEN Adaptive_Ensemble END) AS ensemble_val
  FROM datagroup_mdl.mdl_sales_analytics.arr_forecast_wide
  WHERE UPPER(TRIM(sales_market)) = 'TOTAL'
    AND grain_level = 'total'
    AND ds >= '2026-01-01' AND ds <= '2026-12-28'
  GROUP BY ds, product
),
pbi AS (
  SELECT
    product,
    DATE_FORMAT(ds, 'yyyy-MM') AS month,
    MONTH(ds) AS m_num,
    SUM(actual_val) / 1e6 AS pbi_actuals_M,
    SUM(ensemble_val) / 1e6 AS pbi_ensemble_M
  FROM pbi_deduped
  GROUP BY product, DATE_FORMAT(ds, 'yyyy-MM'), MONTH(ds)
)
SELECT
  COALESCE(fr.product, pbi.product) AS product,
  COALESCE(fr.month, pbi.month) AS month,
  ROUND(fr.fr_actuals_M, 2) AS `FR Actuals ($M)`,
  ROUND(pbi.pbi_actuals_M, 2) AS `PBI Actuals ($M)`,
  ROUND(COALESCE(pbi.pbi_actuals_M,0) - COALESCE(fr.fr_actuals_M,0), 3) AS `Actuals Δ ($M)`,
  ROUND(fr.fr_p50_M, 2) AS `FR Fcst P50 ($M)`,
  ROUND(pbi.pbi_ensemble_M, 2) AS `PBI Ensemble ($M)`,
  ROUND(COALESCE(pbi.pbi_ensemble_M,0) - COALESCE(fr.fr_p50_M,0), 3) AS `Fcst Δ ($M)`,
  ROUND(fr.fr_actuals_M + fr.fr_p50_M, 2) AS `FR Landing ($M)`,
  ROUND(COALESCE(pbi.pbi_actuals_M,0) + COALESCE(pbi.pbi_ensemble_M,0), 2) AS `PBI Landing ($M)`,
  ROUND((COALESCE(pbi.pbi_actuals_M,0) + COALESCE(pbi.pbi_ensemble_M,0))
      - (COALESCE(fr.fr_actuals_M,0) + COALESCE(fr.fr_p50_M,0)), 3) AS `Landing Δ ($M)`
FROM fr
FULL OUTER JOIN pbi ON fr.product = pbi.product AND fr.month = pbi.month
ORDER BY COALESCE(fr.product, pbi.product), COALESCE(fr.m_num, pbi.m_num)

In [ ]:
%sql --name wide_monthly
-- arr_forecast_wide monthly summary — what Power BI actually sees
-- After Sep 17 fix: one row per (ds, product, market) — no dedup needed
SELECT
  product,
  DATE_FORMAT(ds, 'yyyy-MM') AS month,
  CONCAT('Q', QUARTER(ds)) AS quarter,
  ROUND(SUM(actuals) / 1e6, 2)              AS `Actuals ($M)`,
  ROUND(SUM(Adaptive_Ensemble) / 1e6, 2)    AS `Ensemble ($M)`,
  ROUND(SUM(Prophet) / 1e6, 2)              AS `Prophet ($M)`,
  ROUND(SUM(MSTL_v2) / 1e6, 2)             AS `MSTL ($M)`,
  ROUND(SUM(DHR_ARIMA) / 1e6, 2)           AS `DHR ($M)`,
  ROUND(SUM(COALESCE(Global_LGB_Q50_UCC, Global_LGB_Q50_ITSG)) / 1e6, 2) AS `LGB ($M)`,
  ROUND(SUM(ETS) / 1e6, 2)                 AS `ETS ($M)`,
  ROUND(COALESCE(SUM(actuals), 0) / 1e6 + COALESCE(SUM(Adaptive_Ensemble), 0) / 1e6, 2) AS `Landing ($M)`,
  COUNT(DISTINCT CASE WHEN actuals IS NOT NULL THEN ds END) AS `Act Wks`,
  COUNT(DISTINCT CASE WHEN Adaptive_Ensemble IS NOT NULL THEN ds END) AS `Fwd Wks`
FROM datagroup_mdl.mdl_sales_analytics.arr_forecast_wide
WHERE grain_level = 'total'
  AND UPPER(TRIM(sales_market)) = 'TOTAL'
  AND ds >= '2026-01-01' AND ds <= '2026-12-28'
GROUP BY product, DATE_FORMAT(ds, 'yyyy-MM'), QUARTER(ds)
ORDER BY product, month

In [ ]:
%sql --name actuals_source_compare
-- Compare ITSG actuals: forecast_prophet (feeds arr_forecast_wide) vs CBE (feeds forecast_results)
-- Both should be identical for closed weeks — any delta = source mismatch
WITH prophet_actuals AS (
  -- What arr_forecast_wide uses (via forecast_all_models → forecast_prophet)
  SELECT
    CAST(ds AS DATE) AS ds,
    product,
    sales_market,
    Actuals AS prophet_actual
  FROM datagroup_mdl.mdl_sales_analytics.forecast_prophet
  WHERE product = 'ITSG'
    AND UPPER(TRIM(sales_market)) = 'TOTAL'
    AND Actuals IS NOT NULL AND Actuals > 0
    AND ds >= '2026-01-01' AND ds <= '2026-09-30'
),
cbe_actuals AS (
  -- What forecast_results uses (CBE source in unify pipeline)
  SELECT
    ds,
    product,
    sales_market,
    actual AS cbe_actual
  FROM datagroup_mdl.mdl_sales_analytics.forecast_results
  WHERE run_date = (SELECT MAX(run_date) FROM datagroup_mdl.mdl_sales_analytics.forecast_results)
    AND model = 'ACTUAL'
    AND product = 'ITSG'
    AND UPPER(TRIM(sales_market)) IN ('TOTAL', 'ALL')
    AND grain_level = 'total'
    AND ds >= '2026-01-01' AND ds <= '2026-09-30'
)
SELECT
  COALESCE(p.ds, c.ds) AS ds,
  DATE_FORMAT(COALESCE(p.ds, c.ds), 'yyyy-MM') AS month,
  ROUND(p.prophet_actual / 1e6, 4) AS `Prophet Actual ($M)`,
  ROUND(c.cbe_actual / 1e6, 4) AS `CBE Actual ($M)`,
  ROUND((COALESCE(p.prophet_actual, 0) - COALESCE(c.cbe_actual, 0)) / 1e6, 4) AS `Delta ($M)`,
  CASE
    WHEN p.ds IS NULL THEN 'CBE only'
    WHEN c.ds IS NULL THEN 'Prophet only'
    WHEN ABS(p.prophet_actual - c.cbe_actual) < 1 THEN 'Match'
    ELSE 'MISMATCH'
  END AS status
FROM prophet_actuals p
FULL OUTER JOIN cbe_actuals c ON p.ds = c.ds
ORDER BY COALESCE(p.ds, c.ds)

In [ ]:
%sql --name itsg_market_rollup
-- ITSG: forecast_prophet has no 'Total' row — Cell 14 builds Total by summing markets.
-- Compare that sum-of-markets vs CBE total actuals
WITH prophet_by_mkt AS (
  SELECT
    CAST(ds AS DATE) AS ds,
    sales_market,
    Actuals AS actual_val
  FROM datagroup_mdl.mdl_sales_analytics.forecast_prophet
  WHERE product = 'ITSG'
    AND Actuals IS NOT NULL AND Actuals > 0
    AND ds >= '2026-01-01' AND ds <= '2026-09-30'
),
prophet_total AS (
  SELECT ds,
    SUM(actual_val) AS prophet_sum_markets,
    COUNT(DISTINCT sales_market) AS n_markets,
    CONCAT_WS(', ', COLLECT_SET(sales_market)) AS markets
  FROM prophet_by_mkt
  GROUP BY ds
),
cbe AS (
  SELECT ds, actual AS cbe_actual
  FROM datagroup_mdl.mdl_sales_analytics.forecast_results
  WHERE run_date = (SELECT MAX(run_date) FROM datagroup_mdl.mdl_sales_analytics.forecast_results)
    AND model = 'ACTUAL' AND product = 'ITSG'
    AND UPPER(TRIM(sales_market)) IN ('TOTAL', 'ALL')
    AND grain_level = 'total'
    AND ds >= '2026-01-01' AND ds <= '2026-09-30'
)
SELECT
  COALESCE(p.ds, c.ds) AS ds,
  DATE_FORMAT(COALESCE(p.ds, c.ds), 'yyyy-MM') AS month,
  p.n_markets,
  ROUND(p.prophet_sum_markets / 1e6, 4) AS `Prophet Sum-of-Markets ($M)`,
  ROUND(c.cbe_actual / 1e6, 4) AS `CBE Total ($M)`,
  ROUND((COALESCE(p.prophet_sum_markets,0) - COALESCE(c.cbe_actual,0)) / 1e6, 4) AS `Delta ($M)`,
  CASE
    WHEN p.ds IS NULL THEN 'CBE only'
    WHEN c.ds IS NULL THEN 'Prophet only'
    WHEN ABS(p.prophet_sum_markets - c.cbe_actual) < 100 THEN 'Match'
    ELSE 'MISMATCH'
  END AS status,
  p.markets
FROM prophet_total p
FULL OUTER JOIN cbe c ON p.ds = c.ds
ORDER BY COALESCE(p.ds, c.ds)

In [ ]:
%sql --name wide_itsg_detail
-- Check raw arr_forecast_wide rows for ITSG Total in Feb 2026
-- Are there duplicate rows per ds causing actuals double-count?
SELECT
  ds,
  sales_market,
  grain_level,
  forecast_type,
  ROUND(actuals / 1e6, 4) AS actuals_M,
  ROUND(Adaptive_Ensemble / 1e6, 4) AS ensemble_M,
  ROUND(Prophet / 1e6, 4) AS prophet_M
FROM datagroup_mdl.mdl_sales_analytics.arr_forecast_wide
WHERE product = 'ITSG'
  AND UPPER(TRIM(sales_market)) = 'TOTAL'
  AND grain_level = 'total'
  AND ds >= '2026-02-01' AND ds < '2026-03-01'
ORDER BY ds, forecast_type

In [ ]:
%sql --name quarterly_summary
-- Q1–Q4 FY2026: Actuals (closed) + Forecast (open/future) by product
WITH latest AS (
  SELECT *
  FROM datagroup_mdl.mdl_sales_analytics.forecast_results
  WHERE run_date = (SELECT MAX(run_date) FROM datagroup_mdl.mdl_sales_analytics.forecast_results)
    AND model IN ('Adaptive_Ensemble', 'ACTUAL')
    AND UPPER(TRIM(sales_market)) IN ('TOTAL', 'ALL')
    AND grain_level = 'total'
    AND (forecast_view = 'adaptive' OR forecast_view IS NULL)
    AND ds >= '2026-01-01' AND ds <= '2026-12-28'
),
qtr AS (
  SELECT
    product,
    CONCAT('Q', QUARTER(ds)) AS quarter,
    QUARTER(ds) AS q_num,
    -- Actuals from ACTUAL rows
    SUM(CASE WHEN model = 'ACTUAL' THEN actual ELSE 0 END) / 1e6 AS actuals_M,
    -- Forecast from Ensemble (rolling + roy + forward_1yr)
    SUM(CASE WHEN model = 'Adaptive_Ensemble' AND forecast_type IN ('rolling','roy','forward_1yr')
             THEN p50 ELSE 0 END) / 1e6 AS forecast_p50_M,
    SUM(CASE WHEN model = 'Adaptive_Ensemble' AND forecast_type IN ('rolling','roy','forward_1yr')
             THEN p05 ELSE 0 END) / 1e6 AS forecast_p05_M,
    SUM(CASE WHEN model = 'Adaptive_Ensemble' AND forecast_type IN ('rolling','roy','forward_1yr')
             THEN p95 ELSE 0 END) / 1e6 AS forecast_p95_M,
    COUNT(DISTINCT CASE WHEN model = 'ACTUAL' THEN ds END) AS actual_weeks,
    COUNT(DISTINCT CASE WHEN model = 'Adaptive_Ensemble' AND forecast_type IN ('rolling','roy','forward_1yr') THEN ds END) AS forecast_weeks
  FROM latest
  GROUP BY product, QUARTER(ds)
)
SELECT
  product,
  quarter,
  ROUND(actuals_M, 2) AS `Actuals ($M)`,
  ROUND(forecast_p50_M, 2) AS `Forecast P50 ($M)`,
  ROUND(actuals_M + forecast_p50_M, 2) AS `Landing P50 ($M)`,
  ROUND(actuals_M + forecast_p05_M, 2) AS `Landing P05 ($M)`,
  ROUND(actuals_M + forecast_p95_M, 2) AS `Landing P95 ($M)`,
  actual_weeks AS `Closed Wks`,
  forecast_weeks AS `Open Wks`,
  CASE
    WHEN forecast_weeks = 0 THEN 'Closed'
    WHEN actual_weeks > 0 THEN 'In Progress'
    ELSE 'Forecast'
  END AS status
FROM qtr
ORDER BY product, q_num